# Paper figures
Run the setup cell once, then any individual figure cell. The next 31 cells each contain data reading and actual plot code for one complete figure. Inputs are prepared in results/ by the scientific replay.


In [ ]:

# Run this setup once, then run any individual figure cell.
from pathlib import Path
import sys
import json
import shutil
from types import SimpleNamespace
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
if not (ROOT / "src/thermal_heterogeneity").is_dir():
    raise FileNotFoundError("Open this notebook from code or code/plot.")
RESULTS, PLOT = ROOT / "results", ROOT / "plot"
INTERMEDIATE = ROOT / "intermediate_files/notebook"
for folder in (PLOT / "main", PLOT / "supplementary", INTERMEDIATE):
    folder.mkdir(parents=True, exist_ok=True)
for folder in (ROOT / "src", ROOT / "src/analysis/scripts",
               ROOT / "src/analysis/energy_service_paper/scripts"):
    sys.path.insert(0, str(folder))
from thermal_heterogeneity.figures import apply_figure_style  # noqa: E402
from utilities.prepare_plot_data import refresh_plot_inputs  # noqa: E402
apply_figure_style()

# Export helper only. Each cell below contains data selection and actual plotting.
def save_plot(fig, label, dpi=600, size_mm=None, round_pixels=True, exact_pixels=False):
    from matplotlib.text import Text
    if size_mm is not None:
        fig.set_size_inches(size_mm[0]/25.4, size_mm[1]/25.4, forward=True)
    directory = PLOT / ("supplementary" if label.startswith("Figure_S") else "main")
    stem = directory / label
    for artist in fig.findobj(Text):
        artist.set_color("black")
    for extension in ("pdf", "svg"):
        metadata = {"CreationDate":None, "ModDate":None} if extension == "pdf" else {"Date":None}
        fig.savefig(stem.with_suffix("."+extension), metadata=metadata)
    for suffix, resolution in (("", dpi), ("_word", 300)):
        path = Path(str(stem)+suffix+".png")
        original_size = tuple(fig.get_size_inches())
        if exact_pixels:
            expected = tuple(round(value * resolution) for value in original_size)
            fig.set_size_inches(expected[0]/resolution, expected[1]/resolution, forward=True)
        fig.savefig(path, dpi=resolution)
        with Image.open(path) as image:
            rgb = image.convert("RGB")
            size = tuple(round(value * resolution) for value in fig.get_size_inches())
            if round_pixels and rgb.size != size:
                rgb = rgb.resize(size, Image.Resampling.LANCZOS)
            rgb.save(path)
        fig.set_size_inches(original_size, forward=True)
    caption = RESULTS / "captions" / (label+".txt")
    if caption.is_file():
        shutil.copyfile(caption, Path(str(stem)+"_caption.txt"))
    plt.close(fig)
    refresh_plot_inputs(RESULTS)
    print(label+": "+str(stem.with_suffix(".png")))
    if "get_ipython" in globals():
        from IPython.display import display, Image as NotebookImage
        display(NotebookImage(filename=str(stem.with_suffix(".png")), width=850))

In [ ]:
# Figure_01: edit this cell and rerun to update this figure.
def draw_figure_01():
    LABEL = "Figure_01"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import json

    from pathlib import Path

    from typing import Callable

    import matplotlib.pyplot as plt

    import numpy as np

    import pandas as pd

    from matplotlib.axes import Axes

    from matplotlib.figure import Figure

    from matplotlib.patches import FancyBboxPatch, Patch, Rectangle

    from thermal_heterogeneity.figures import (
        CONTEXT_COLORS as SHARED_CONTEXT_COLORS,
        DEMAND_CMAP,
        VARIANCE_COLORS,
        PALETTE,
        create_panel_figure,
        data_safe_legend,
        draw_world_land,
        style_panel,
    )

    CONTEXT_ORDER = ["Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019"]

    CONTEXT_LABELS = {
        "Miami": "Miami",
        "NewDelhi": "New Delhi",
        "PortAngeles_small2019": "Port Angeles",
        "Seattle_small2019": "Seattle",
    }

    CONTEXT_LEGEND_LABELS = {
        "Miami": "Miami",
        "NewDelhi": "New Delhi",
        "PortAngeles_small2019": "Port Angeles",
        "Seattle_small2019": "Seattle",
    }

    CONTEXT_COLORS = SHARED_CONTEXT_COLORS

    POPULATION_LABELS = {
        "us_archive_stratum": "United States",
        "multinational_source_balanced": "Multinational",
        "india_archive_stratum": "India",
    }

    OUTCOME_ORDER = [
        "same_direction_decrease",
        "same_direction_increase",
        "all_tied",
        "direction_not_stable",
        "match_lost",
        "all_unmatched",
    ]

    OUTCOME_LABELS = {
        "same_direction_decrease": "Decrease",
        "same_direction_increase": "Increase",
        "all_tied": "Tie",
        "direction_not_stable": "Unstable",
        "match_lost": "Unmatched",
        "all_unmatched": "Unmatched",
    }

    OUTCOME_COLORS = {
        "same_direction_decrease": PALETTE["blue"],
        "same_direction_increase": PALETTE["vermillion"],
        "all_tied": PALETTE["grey"],
        "direction_not_stable": PALETTE["purple"],
        "match_lost": PALETTE["light_grey"],
        "all_unmatched": PALETTE["light_grey"],
    }

    def _read_csv(root: Path, panel_id: str, required: set[str]) -> pd.DataFrame:
        path = root / f"{panel_id}.csv"
        frame = pd.read_csv(path)
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"{panel_id} missing columns: {sorted(missing)}")
        if frame.empty or set(frame["panel_id"]) != {panel_id}:
            raise ValueError(f"{panel_id} is empty or carries the wrong panel_id")
        return frame

    def _finish_axis(axis: Axes, letter: str, title: str, grid_axis: str | None = "y") -> None:
        style_panel(axis, letter, title, grid_axis=grid_axis)

    def build_figure_1(panel_root: Path | str) -> Figure:
        """Build Figure 1: observed geography, policy coordinates and variance scale."""

        root = Path(panel_root)
        a = _read_csv(root, "Fig1a", {"population", "records", "analysis_units"})
        b = _read_csv(
            root,
            "Fig1b",
            {"city", "latitude", "longitude", "supported", "analysis_units", "demand_center_c"},
        )
        c = _read_csv(
            root,
            "Fig1c",
            {"city", "latitude", "longitude", "supported", "analysis_units", "demand_center_c"},
        )
        d = _read_csv(
            root,
            "Fig1d",
            {"aggregation_level", "season", "name", "p10_c", "p50_c", "p90_c"},
        )
        e = _read_csv(root, "Fig1e", {"component", "share"})
        f = _read_csv(
            root,
            "Fig1f",
            {"policy", "position", "proportion", "ci_low", "ci_high", "records"},
        )
        g = _read_csv(
            root,
            "Fig1g",
            {"policy", "q10_delta_c", "q25_delta_c", "median_delta_c", "q75_delta_c", "q90_delta_c", "records"},
        )
        h = _read_csv(
            root,
            "Fig1h",
            {"reference", "weighting", "component", "loo_min_share", "loo_median_share", "loo_max_share", "n_leave_one_source_runs"},
        )

        figure, axes = create_panel_figure()
        figure.subplots_adjust(wspace=0.24, hspace=0.34)
        ax = axes.ravel()

        # a: separate record and analysis-unit flows prevent a people/record conflation.
        complete = a.set_index("population").loc["complete"]
        executable = a.set_index("population").loc["executable"]
        ax[0].set_xlim(-0.12, 1.04)
        ax[0].set_ylim(-0.4, 1.7)
        ax[0].set_xticks([])
        ax[0].set_yticks([])
        ax[0].axis("off")
        for y, key, label, loss in [
            (1.05, "records", "Records", 19),
            (0.25, "analysis_units", "Analysis units", 17),
        ]:
            values = [int(complete[key]), int(executable[key])]
            for x, value, stage in zip([0.02, 0.86], values, ["Complete", "Executable"]):
                width = 0.34 if stage == "Complete" else 0.32
                box = FancyBboxPatch(
                    (x - width / 2, y - 0.19),
                    width,
                    0.38,
                    boxstyle="round,pad=0.02",
                    facecolor="#EAF2F8" if x == 0 else "#D7EFE7",
                    edgecolor="#666666",
                    linewidth=0.65,
                )
                ax[0].add_patch(box)
                ax[0].text(x, y + 0.04, f"{value:,}", ha="center", va="center")
                ax[0].text(x, y - 0.10, stage, ha="center", va="center")
            ax[0].annotate("", xy=(0.68, y), xytext=(0.21, y), arrowprops={"arrowstyle": "->", "lw": 0.9})
            ax[0].text(0.45, y + 0.15, f"-{loss}", ha="center", va="center")
            ax[0].text(0.02, y + 0.30, label, ha="left", va="center")
        ax[0].text(0.46, 1.52, "187 buildings · 40 cities · 15 countries", ha="center", va="center")
        _finish_axis(ax[0], "a", "Archive and population", None)

        supported_centres = pd.concat([b, c], ignore_index=True)["demand_center_c"].dropna()
        vmin, vmax = float(supported_centres.min()), float(supported_centres.max())
        anchor_names = {"honolulu", "midland", "bangalore", "sydney"}
        anchor_offsets = {
            "honolulu": (4, -10),
            "midland": (3, 3),
            "bangalore": (3, -7),
            "sydney": (-4, 7),
        }
        anchor_labels = {"honolulu": "Honolulu", "midland": "UK", "bangalore": "Bengaluru", "sydney": "Sydney"}
        for panel_axis, frame, letter, title in [
            (ax[1], b, "b", "Warm-season observations"),
            (ax[2], c, "c", "Cold-season observations"),
        ]:
            draw_world_land(panel_axis)
            is_supported = frame["supported"].astype(str).str.lower().eq("true")
            unsupported = frame.loc[~is_supported]
            panel_axis.scatter(
                unsupported["longitude"],
                unsupported["latitude"],
                s=22,
                facecolors="none",
                edgecolors=PALETTE["grey"],
                linewidths=0.8,
                zorder=2,
            )
            shown = frame.loc[is_supported].copy()
            sizes = 18 + 110 * np.sqrt(shown["analysis_units"].astype(float) / shown["analysis_units"].max())
            panel_axis.scatter(
                shown["longitude"],
                shown["latitude"],
                c=shown["demand_center_c"],
                cmap=DEMAND_CMAP,
                vmin=vmin,
                vmax=vmax,
                s=sizes,
                edgecolors="white",
                linewidths=0.45,
                zorder=3,
            )
            for row in shown.loc[shown["city"].isin(anchor_names)].itertuples():
                city = str(row.city)
                annotation = panel_axis.annotate(
                    anchor_labels[str(row.city)],
                    (row.longitude, row.latitude),
                    xytext=anchor_offsets[city],
                    textcoords="offset points",
                    ha="right" if city == "sydney" else "left",
                    va="bottom",
                    annotation_clip=True,
                )
                annotation.set_clip_on(True)
            panel_axis.text(
                0.99,
                0.02 if letter == "b" else 0.08,
                f"{len(shown)}/40 supported",
                ha="right",
                va="bottom",
                transform=panel_axis.transAxes,
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 0.6},
            )
            if letter == "b":
                colour_map = plt.get_cmap(DEMAND_CMAP)
                for colour_index in range(8):
                    panel_axis.add_patch(
                        Rectangle(
                            (-34 + 12 * colour_index, -57),
                            12,
                            5,
                            facecolor=colour_map(colour_index / 7),
                            edgecolor="none",
                            zorder=4,
                        )
                    )
                panel_axis.text(14, -48, f"{vmin:.1f}–{vmax:.1f} °C", ha="center", va="bottom")
            else:
                panel_axis.scatter([-170, -135], [-54, -54], s=[28, 82], color=PALETTE["grey"], edgecolors="white", linewidths=0.4, zorder=4)
                panel_axis.scatter([-70], [-54], s=35, facecolors="none", edgecolors=PALETTE["grey"], linewidths=0.8, zorder=4)
                panel_axis.text(-120, -54, "area=n", ha="left", va="center")
                panel_axis.text(-59, -54, "no support", ha="left", va="center")
            panel_axis.set_xticks([-120, 0, 120])
            panel_axis.set_yticks([-30, 30, 60])
            if letter == "b":
                panel_axis.set_ylabel("")
                panel_axis.tick_params(axis="y", direction="in", pad=-24)
                for tick_label in panel_axis.get_yticklabels():
                    tick_label.set_horizontalalignment("left")
            _finish_axis(panel_axis, letter, title, None)

        # d: four compact ranked bands, preserving empirical P10--P90 intervals.
        group_order = [("city", "warm"), ("city", "cold"), ("source", "warm"), ("source", "cold")]
        y_base = 0
        yticks, ylabels = [], []
        for group_index, (level, season) in enumerate(group_order):
            group = d.loc[(d["aggregation_level"] == level) & (d["season"] == season)].sort_values("p50_c")
            count = len(group)
            ys = np.linspace(y_base, y_base + 0.68, count) if count > 1 else np.array([y_base + 0.34])
            ax[3].hlines(ys, group["p10_c"], group["p90_c"], color=PALETTE["sky"], linewidth=0.7)
            ax[3].scatter(group["p50_c"], ys, s=5, color=PALETTE["blue"], zorder=3)
            widths = group["p90_c"] - group["p10_c"]
            yticks.append(y_base + 0.34)
            ylabels.append(f"{'C' if level == 'city' else 'S'}/{season[0]}: {count}, {widths.median():.1f}°")
            y_base += 1.0
        ax[3].set_yticks([])
        ax[3].set_xlim(14, 32)
        ax[3].set_xticks([16, 20, 24, 28, 32])
        for yy, label in zip(yticks, ylabels):
            ax[3].text(14.15, yy, label, ha="left", va="center")
        ax[3].set_xlabel("Demand temperature (°C)", labelpad=-4.5)
        _finish_axis(ax[3], "d", "Within-place demand ranges", "x")

        # e: exact mutually exclusive variance allocation.
        component_labels = {
            "country": "Country",
            "city": "City",
            "release_building_group_id": "Building",
            "release_unit_id": "Within building",
            "within_unit": "Repeat",
        }
        order = ["country", "city", "release_building_group_id", "release_unit_id", "within_unit"]
        colors = VARIANCE_COLORS
        lookup = e.set_index("component")["share"]
        left = 0.0
        for component, color in zip(order, colors):
            value = float(lookup.loc[component]) * 100
            ax[4].barh([0], [value], left=left, color=color, height=0.45)
            offset_y = {"city": 0.13, "release_building_group_id": -0.13}.get(component, 0)
            ax[4].text(left + value / 2, offset_y, f"{value:.1f}%", ha="center", va="center")
            left += value
        ax[4].set_xlim(0, 100)
        ax[4].set_ylim(-0.65, 0.85)
        ax[4].set_yticks([])
        ax[4].set_xlabel("Share of variance (%)", labelpad=-4.5)
        handles = [Patch(facecolor=color, label=component_labels[name]) for name, color in zip(order, colors)]
        ax[4].legend(handles=handles, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.0), handlelength=0.8, columnspacing=0.7, handletextpad=0.3)
        _finish_axis(ax[4], "e", "Where demand variance resides", "x")

        # f: each policy keeps its own denominator and within-range CI.
        positions = ["below", "within", "above"]
        position_colors = {"below": PALETTE["blue"], "within": PALETTE["light_grey"], "above": PALETTE["vermillion"]}
        policy_order = list(dict.fromkeys(f["policy"]))
        compact_policy = {
            "Australia model COP": "AUS COP",
            "Brazil NR-17": "BRA NR17",
            "Canada CCOHS": "CAN CCOHS",
            "India ECSBC 2024": "IND ECSBC",
            "Indonesia Permenkes 48": "IDN P48",
            "Malaysia ICOP IAQ": "MYS ICOP",
            "Singapore SS 553": "SGP SS553",
            "Sweden AFS 2023:12": "SWE AFS",
            "USA OSHA guidance": "USA OSHA",
        }
        for y, policy in enumerate(policy_order):
            group = f.loc[f["policy"] == policy].set_index("position")
            left = 0.0
            for position in positions:
                value = float(group.loc[position, "proportion"])
                ax[5].barh(y, value, left=left, color=position_colors[position], height=0.68)
                left += value
            within = group.loc["within"]
            within_left = float(group.loc["below", "proportion"])
            centre = within_left + float(within["proportion"]) / 2
            half = (float(within["ci_high"]) - float(within["ci_low"])) / 2
            ax[5].errorbar(centre, y, xerr=half, color="#222222", capsize=1.5, linewidth=0.8)
            ax[5].text(1.04, y, f"{float(within['proportion'])*100:.0f}%", ha="left", va="center")
            ax[5].text(1.50, y, f"{int(within['records']):,}", ha="right", va="center")
        ax[5].set_yticks([])
        for y, policy in enumerate(policy_order):
            ax[5].text(-0.02, y, compact_policy[policy], ha="right", va="center")
        ax[5].set_ylim(len(policy_order) - 0.5, -2.25)
        ax[5].set_xlim(-0.46, 1.55)
        ax[5].set_xticks([0, 0.5, 1.0], ["0", "50", "100"])
        ax[5].set_xlabel("Demand records by range (%)", labelpad=-4.5)
        handles = [Patch(facecolor=position_colors[p], label=p.title()) for p in positions]
        ax[5].legend(handles=handles, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.99), handlelength=0.8, columnspacing=0.7, handletextpad=0.3)
        _finish_axis(ax[5], "f", "Policy-range coverage", "x")

        # g: empirical demand-minus-target quantiles.
        y = np.arange(len(g))
        ax[6].axvline(0, color="#333333", linewidth=0.7)
        for index, row in enumerate(g.itertuples()):
            ax[6].hlines(index, row.q10_delta_c, row.q90_delta_c, color=PALETTE["sky"], linewidth=1.2)
            ax[6].hlines(index, row.q25_delta_c, row.q75_delta_c, color=PALETTE["blue"], linewidth=4)
            ax[6].scatter(row.median_delta_c, index, s=26, color=PALETTE["vermillion"], zorder=3)
            label_y = index + 0.14 if index < len(g) - 1 else index - 0.14
            value_label = ax[6].text(
                2.55,
                label_y,
                f"{row.median_delta_c:+.1f}° (n={int(row.records)})",
                ha="right",
                va="top" if index < len(g) - 1 else "bottom",
                clip_on=True,
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.78, "pad": 0.15},
            )
            value_label.set_gid("fig1g-value-label")
        ax[6].set_yticks(y, [])
        for index, value in enumerate(g["policy"]):
            label = value.replace("China summer rule", "China").replace("Japan Cool Biz", "Japan").replace("Thailand EGAT campaign", "Thailand")
            ax[6].text(-5.05, index, label, ha="left", va="center")
        ax[6].invert_yaxis()
        ax[6].set_xlim(-5.2, 4.8)
        ax[6].set_xticks([-4, -2, 0, 2, 4])
        ax[6].set_xlabel("Demand centre − campaign target (°C)")
        _finish_axis(ax[6], "g", "Guidance relative to demand", "x")

        # h: all 12 coefficient/weighting specifications remain visible.
        component_names = {
            "between_analysis_unit_within_building": "Within bldg.",
            "between_country": "Country",
            "between_city_within_country": "City",
            "between_building_within_city": "Building",
            "within_analysis_unit": "Repeat",
        }
        comp_order = list(component_names)
        for y, component in enumerate(comp_order):
            group = h.loc[h["component"] == component]
            for _, row in group.iterrows():
                ax[7].hlines(y, row["loo_min_share"] * 100, row["loo_max_share"] * 100, color="#C9C9C9", linewidth=0.6)
                ax[7].scatter(row["loo_median_share"] * 100, y, s=10, color=PALETTE["blue"], alpha=0.8)
            ax[7].hlines(y, group["loo_min_share"].min() * 100, group["loo_max_share"].max() * 100, color=PALETTE["vermillion"], linewidth=2.2)
            ax[7].scatter(group["loo_median_share"].median() * 100, y, s=28, color=PALETTE["vermillion"], zorder=3)
        ax[7].set_yticks([])
        for y, component in enumerate(comp_order):
            ax[7].text(-1.5, y, component_names[component], ha="right", va="center")
        ax[7].invert_yaxis()
        ax[7].set_xlim(-20, 75)
        ax[7].set_xticks([0, 25, 50, 75])
        ax[7].set_xlabel("Variance share (%)")
        _finish_axis(ax[7], "h", "Robustness: 12 specs × 27 removals", "x")

        return figure

    CAPTIONS = {
        1: (
            "**Figure 1 | Observed human demand is geographically structured but dominated by "
            "within-building heterogeneity.** **a,** The complete office archive contains 21,418 records "
            "from 12,130 analysis units; the executable set retains 21,399 records from 12,113 "
            "analysis units in 187 buildings, 40 cities and 15 countries. **b,c,** Observed-city warm- "
            "and cold-season demand centres. Filled circles denote season-supported cities, hollow "
            "circles observed but unsupported cities, circle area the number of analysis units and "
            "colour the demand centre on the same 19.5–30.6 °C scale; unsampled geography is left "
            "blank and no spatial interpolation is applied. **d,** Empirical P10–P90 intervals and "
            "medians for city- and source-level warm/cold aggregates under analysis-unit-equal "
            "weighting; labels give the number of aggregates and median interval width. **e,** "
            "Hierarchical variance allocation across country (22.3%), city within country (7.7%), "
            "building within city (5.8%), analysis unit within building (56.1%) and repeated "
            "measurements (8.2%). **f,** Weighted demand records below, within and above nine bilateral "
            "policy or design ranges at Griffiths coefficient 0.50; labels give the within-range "
            "share and eligible-record denominator, and whiskers are nested-bootstrap 95% confidence "
            "intervals for the within-range share. **g,** Empirical P10–P90 (thin), P25–P75 (thick) and "
            "median (point) of demand centre minus campaign target for three prespecified campaigns; "
            "positive values indicate a demand centre warmer than the target, and negative values "
            "indicate a demand centre cooler than the target. **h,** Leave-one-source sensitivity ranges "
            "and medians across 12 coefficient/weighting specifications, each based on 27 source "
            "removals. Policy documents in **f,g** are external temperature coordinates rather than "
            "individual comfort ground truth."
        ),
        2: (
            "**Figure 2 | Independent shared-room votes reveal demand disagreement hidden by "
            "averages.** **a,** Prespecified construction of the NUS 30-min validation cohort from "
            "4,378 raw rows to 63 temperature-homogeneous room-windows containing 225 valid "
            "person-window votes. **b,** Cooler, no-change and warmer vote shares in all 63 "
            "room-windows, ordered by net vote and disagreement; 49 windows contain more than one "
            "response category and six contain warmer and cooler requests within the same 30-min "
            "room-window. **c,** Vote counts in the six room-windows with within-window opposing "
            "requests; grey circles denote no-change votes and labels give vote count, mean room "
            "temperature and within-window temperature span. **d,** Full request burden, mean "
            "absolute vote, versus burden after averaging, absolute mean vote, for each window. "
            "Hollow orange points are exact cancellations; the dashed line is equality and the "
            "median compression gap is zero despite a positive mean gap. **e,** The four exact-"
            "cancellation windows, in which non-zero full burden is represented as zero after "
            "averaging. **f,** False acceptance and false rejection at burden thresholds 0, 0.25, "
            "0.50 and 0.75; all four thresholds retain zero false rejections and 2–5 false "
            "acceptances among 63 windows. **g,** Disagreement, within-window opposing requests and "
            "mean Jensen gap under 15-, 30- and 60-min windows; whiskers are crossed room/date "
            "cluster-bootstrap 95% confidence intervals and n is the eligible-window denominator. "
            "**h,** Independent ENTH replication (34 windows, 81 votes) beside NUS (63 windows, 225 "
            "votes). Windows, rather than votes, are the inferential unit throughout."
        ),
        3: (
            "**Figure 3 | Demand location redistributes annual energy and carbon under three "
            "controls.** **a,** Uniform-average, personal-centre and personal-range control "
            "families generate physical actions in the same building under identical operating "
            "permissions. These controls are distinct from the mean and full-distribution "
            "qualification representations used to score the same actions. **b,** Assigned P10–P90 "
            "demand intervals, centres and seed variation for three population compositions in four "
            "building/climate contexts. **c,** Prespecified orthogonal transformation "
            "T′=μ+Δ+λ(T−μ), with demand-location shift Δ from −2 to +2 °C and spread scale λ from 0 "
            "to 1.5; the open point is the Δ=0, λ=1 reference. **d,e,** Median and interquartile "
            "range of matched service-qualified changes in purchased space-HVAC site energy and "
            "operational CO₂ relative to the reference. The coloured denominator tables give "
            "matched n for Miami, New Delhi, Port Angeles and Seattle in left-to-right Δ order (−2, "
            "−1, +1 and +2 °C). **f,** Median signed target-minus-reference changes in heating "
            "electricity, cooling electricity, fan electricity and gas HHV for Δ=+2 °C, using the "
            "amended reference/target carrier fields directly; diamonds show median total-energy "
            "change and the two-line context labels give 22, 20, 8 and 9 matched cells. The artwork "
            "states the full accounting boundary of 59 matched and 85 unmatched design rows; "
            "unmatched rows contribute no fabricated delta. **g,** Energy direction for all 130 "
            "default uniform-versus-alternative paired definitions, separated by control and "
            "context: 54 increases, 3 decreases, 36 ties, 7 direction-unstable comparisons and 30 "
            "unmatched comparisons. **h,** Outcome shares across Griffiths energy, Griffiths carbon "
            "and personal-range-width sensitivity families with explicit denominators and the same outcome "
            "colour/hatch key as panel g. Population assignments are controlled archive scenarios "
            "rather than claims of local population representativeness; carbon factors are fixed "
            "regional scenarios."
        ),
    }

    ALT_TEXT = {
        1: (
            "Eight-panel portrait figure. An archive flow retains 21,399 of 21,418 records. Two "
            "observed-city maps show 31 warm-season and 26 cold-season supported cities without "
            "interpolation. Ranked empirical intervals show broad within-city and within-source "
            "demand ranges. A variance bar assigns the largest share, 56.1%, to analysis units "
            "within buildings. Policy-range bars show wide variation in observed-demand coverage. "
            "Panel g plots demand centre minus campaign target: positive values mean that the demand "
            "centre is warmer than the target, and negative values mean that it is cooler. "
            "Leave-one-source sensitivity preserves the importance of within-building heterogeneity, "
            "while showing that its magnitude depends on weighting and the Griffiths coefficient."
        ),
        2: (
            "Eight-panel portrait figure. A filtering flow yields 63 shared-room windows and 225 "
            "direct votes. A 63-row composition chart shows disagreement in 49 windows, including "
            "six with warmer and cooler requests within the same 30-min room-window. Six diverging "
            "bars detail those within-window opposing requests. A scatter shows that averaging can "
            "only preserve or reduce represented request burden and exactly cancels four non-neutral "
            "windows. Threshold bars show false acceptance but no false rejection. Results persist "
            "across 15-, 30- and 60-min windows and replicate in a second ENTH dataset, although "
            "confidence intervals are wider there."
        ),
        3: (
            "Eight-panel portrait figure. Panel a compares three control families—uniform-average, "
            "personal-centre and personal-range controls—under the same building and operating "
            "permissions. Mean and full-distribution are separate qualification representations "
            "used to score generated actions, not additional controls. Population scenarios differ "
            "in both demand centre and width. A centre-by-spread grid defines independent "
            "perturbations. Energy and carbon response curves change sign across contexts: warmer "
            "demand reduces median annual energy and carbon in Miami and New Delhi but increases "
            "them in Port Angeles and Seattle. Coloured tables above both curves report the matched "
            "denominator at each demand shift without covering the data. Carrier bars show "
            "concurrent shifts among heating electricity, cooling electricity, fans and gas and "
            "state that 59 rows match while 85 remain unmatched. Stacked outcome bars retain "
            "increases, decreases, ties, unstable directions and unmatched cases. Sensitivity bars "
            "reuse that outcome key and show that direction and match availability depend on demand "
            "assumptions, so personalization does not imply universal savings."
        ),
    }
    from render_revised_main_figures import restructure, readable_display_labels
    from thermal_heterogeneity.figures import apply_figure_style, apply_data_axis_style
    DPI = 600
    apply_figure_style()
    PANEL_ROOT = RESULTS / "panel_data"
    figure = build_figure_1(PANEL_ROOT)
    figure._stage7_panel_root = PANEL_ROOT
    figure._stage7_decision_value = RESULTS / "decision_value/MAIN_FIG6G_Q5_REGRET.csv"
    restructure(figure, 1)
    readable_display_labels(figure, 1)
    # Compact text keeps the data frames wide while preserving equal columns.
    result_axes = sorted(
        (axis for axis in figure.axes if axis.get_title(loc="left").startswith(tuple(f"({letter})" for letter in "abcdefgh"))),
        key=lambda axis: axis.get_title(loc="left")[1],
    )
    for index, axis in enumerate(result_axes):
        position = axis.get_position()
        axis.set_position((.16 if index % 2 == 0 else .64, position.y0, .34, position.height))
    variance_axis = result_axes[3]
    variance_axis.set_yticks(variance_axis.get_yticks(), [
        label.get_text().replace("Within building", "Within\nbuilding")
        for label in variance_axis.get_yticklabels()
    ])
    result_axes[7].set_yticks([0, 1], ["Singapore", "Longitudinal"])
    for text in list(figure.texts):
        if "supported" in text.get_text() or text.get_text() == "°C":
            text.remove()
    for axis, count in zip(result_axes[:2], [31, 26]):
        axis.text(.025, .055, f"{count} of 40 cities", transform=axis.transAxes,
                  ha="left", va="bottom", fontsize=11,
                  bbox={"facecolor": "white", "edgecolor": "none", "alpha": .9, "pad": 1})
    colorbar_axis = next(axis for axis in figure.axes if getattr(axis, "_colorbar", None) is not None)
    colorbar_axis.set_xlabel("Demand centre (°C)", fontsize=11, labelpad=3)
    # The six simulated demand compositions retain the actual 267 assigned profiles.
    # This replaces the repeated variance sensitivity; its full evidence remains in SI.
    composition_path = RESULTS / "Joint_main/DEMAND_COMPOSITION_PROFILES.csv"
    composition_profiles = pd.read_csv(composition_path)
    required = {"composition_id", "profile_id", "centre_c", "lower_c", "upper_c"}
    if (not required.issubset(composition_profiles.columns) or len(composition_profiles) != 1602
            or composition_profiles.composition_id.nunique() != 6
            or composition_profiles.duplicated(["composition_id", "profile_id"]).any()
            or not composition_profiles.groupby("composition_id").size().eq(267).all()):
        raise ValueError("Figure 1g requires six complete fixed demand compositions with 267 profiles each")
    composition_profiles["width_c"] = composition_profiles.upper_c - composition_profiles.lower_c
    if (composition_profiles.width_c <= 0).any():
        raise ValueError("Every displayed Personal range must have positive width")
    demand_panel = next(axis for axis in figure.axes if axis.get_title(loc="left").startswith("(g)"))
    demand_panel.clear()
    demand_panel.set_axis_off()
    demand_panel.set_xticks([])
    demand_panel.set_yticks([])
    demand_panel.set_title("(g) Simulated demand", loc="left", fontsize=12, weight="bold", pad=9)
    centre_axis = demand_panel.inset_axes([0, 0, .43, 1])
    width_axis = demand_panel.inset_axes([.61, 0, .39, 1])
    source_rows = []
    composition_ids = sorted(composition_profiles.composition_id.unique())
    for index, composition_id in enumerate(composition_ids):
        group = composition_profiles[composition_profiles.composition_id.eq(composition_id)]
        for child, metric, colour in [(centre_axis, "centre_c", "#6F9EBC"),
                                      (width_axis, "width_c", "#C78470")]:
            q10, q25, q50, q75, q90 = np.quantile(group[metric], [.1, .25, .5, .75, .9])
            child.hlines(index, q10, q90, color=colour, linewidth=1.1)
            child.hlines(index, q25, q75, color=colour, linewidth=3.3)
            child.scatter(q50, index, s=15, color=colour, edgecolors="white", linewidths=.35, zorder=4)
            source_rows.append({"composition_id": int(composition_id), "composition_label": f"Composition {index + 1}",
                                "profiles": len(group), "metric": metric, "p10": q10, "p25": q25,
                                "p50": q50, "p75": q75, "p90": q90})
    for child in [centre_axis, width_axis]:
        child.set_ylim(5.65, -.65)
        child.tick_params(axis="both", labelsize=11, pad=3, length=2.5)
        child.spines["right"].set_visible(False)
        child.spines["top"].set_visible(False)
        child.spines["left"].set_visible(False)
        child.tick_params(axis="y", length=0)
        child.set_yticks(range(6))
    centre_axis.set_yticklabels([f"Composition {index + 1}" for index in range(6)])
    width_axis.set_yticklabels([])
    centre_axis.set_xlim(15, 34)
    centre_axis.set_xticks([16, 24, 32])
    width_axis.set_xlim(0, 13)
    width_axis.set_xticks([0, 6, 12])
    centre_axis.set_xlabel("Centre (°C)", fontsize=11, labelpad=6)
    width_axis.set_xlabel("Width (°C)", fontsize=11, labelpad=6)
    composition_summary = INTERMEDIATE / LABEL
    composition_summary.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(source_rows).to_csv(composition_summary / "Figure_01g_composition_quantiles.csv", index=False)
    (RESULTS / "captions/Figure_01.txt").write_text('Figure 1 | Thermal demand across places and shared rooms. Panels a–f use ASHRAE office observations; g uses both harmonized archives. a,b, Reconstructed city demand centres in warm/cold seasons. Marker area gives analysis-unit count and colour gives temperature; hollow markers indicate unsupported summaries. Map labels count supported city summaries: 31/40 warm and 26/40 cold. c, City/source medians and 10th–90th percentiles. d, Unit-weighted variance across 187 building groups (150 original, 37 inferred identifiers). e, Record-equal shares below, within and above nine policy/design ranges, with nested-bootstrap 95% intervals for within-range coverage. f, Demand centre minus campaign target. g, Six simulation inputs of 267 profiles each: reference centres (blue) and model-generated range widths (terracotta), on separate scales. Thin/thick segments in f,g show 10th–90th/25th–75th percentiles; points show medians. h, Direct requests from the Singapore (63 windows) and longitudinal (34 windows) studies. Categories are identical (Same), different without opposing requests (Mixed), and simultaneous warmer/cooler (Opposed). Different requests equal Mixed plus Opposed: 43 + 6 = 49 and 20 + 2 = 22, respectively.', encoding="utf-8")
    apply_data_axis_style(figure, map_axes=figure.axes[:2] if 1 == 1 else [])
    save_plot(figure, LABEL, dpi=DPI, exact_pixels=True)


draw_figure_01()


In [ ]:
# Figure_02: edit this cell and rerun to update this figure.
def draw_figure_02():
    LABEL = "Figure_02"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import argparse

    import hashlib

    import json

    from pathlib import Path

    import matplotlib

    import matplotlib.pyplot as plt

    from matplotlib.font_manager import FontProperties, findfont

    from matplotlib.lines import Line2D

    from matplotlib.text import Text

    from PIL import Image

    import numpy as np

    import pandas as pd

    from matplotlib.colors import LinearSegmentedColormap

    from matplotlib.ticker import MaxNLocator

    FONT = "Times New Roman"

    WIDTH_MM, HEIGHT_MM = 180, 183

    CONTROL_ORDER = ("uniform_average", "personal_center", "ltpref_interval")

    CONTROL_LABELS = {
        "uniform_average": "Uniform average",
        "personal_center": "Personal centre",
        "ltpref_interval": "Personal range",
    }

    CONTROL_COLORS = {
        "uniform_average": "#7F7F7F",
        "personal_center": "#C8A36A",
        "ltpref_interval": "#7FA995",
    }

    CONTEXT_ORDER = ("Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019")

    CONTEXT_COLORS = dict(zip(CONTEXT_ORDER, ("#C78470", "#AF94AD", "#6F9EBC", "#7FA995")))

    CONTEXT_LABELS = dict(zip(CONTEXT_ORDER, ("Miami", "New Delhi", "Port Angeles", "Seattle")))

    CONTEXT_SHORT = ("Miami", "New\nDelhi", "Port\nAngeles", "Seattle")

    CONTROL_MARKERS = dict(zip(CONTROL_ORDER, ("o", "s", "^")))

    ALTERNATIVES = CONTROL_ORDER[1:]

    DATA_BOUNDS_MM = tuple(
        (x, y, 67.0, 35.0) for y in (125.0, 72.0, 19.0) for x in (19.0, 109.0)
    )

    def configure_style() -> dict[str, str]:
        fonts = {}
        for weight in ("normal", "bold"):
            prop = FontProperties(family=FONT, weight=weight)
            path = Path(findfont(prop, fallback_to_default=False))
            if FontProperties(fname=str(path)).get_name() != FONT:
                raise ValueError(f"Exact {FONT} {weight} is required")
            fonts[weight] = str(path)
        plt.rcParams.update({
            "font.family": FONT, "font.size": 11, "text.color": "black",
            "axes.labelsize": 11, "axes.labelcolor": "black", "axes.labelpad": 6,
            "axes.titlesize": 12, "axes.titleweight": "bold",
            "axes.spines.top": False, "axes.spines.right": False,
            "axes.edgecolor": "#777777", "axes.linewidth": 0.75,
            "xtick.labelsize": 11, "ytick.labelsize": 11,
            "xtick.major.pad": 4, "ytick.major.pad": 4,
            "xtick.color": "black", "ytick.color": "black",
            "legend.fontsize": 11, "legend.frameon": False,
            "figure.facecolor": "white", "savefig.facecolor": "white",
            "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
            "svg.hashsalt": "thermal-heterogeneity-stage8",
        })
        return fonts

    def six_panel_canvas(titles: list[str], *, control_key: bool = True, bounds=None, height=None):
        """Create equal data frames with separate title, axis and shared-key bands."""
        if len(titles) != 6:
            raise ValueError("The main result template has exactly six data panels")
        configure_style()
        page_height = height or HEIGHT_MM
        fig = plt.figure(figsize=(WIDTH_MM / 25.4, page_height / 25.4))
        axes = []
        for index, (title, panel_bounds) in enumerate(zip(titles, bounds or DATA_BOUNDS_MM)):
            x, y, width, height_mm = panel_bounds
            ax = fig.add_axes((x / WIDTH_MM, y / page_height, width / WIDTH_MM, height_mm / page_height))
            ax.set_title(f"({chr(97 + index)}) {title}", loc="left", pad=8)
            ax.grid(False)
            ax.set_axisbelow(True)
            axes.append(ax)
        if control_key:
            handles = [Line2D([], [], marker=marker, linestyle="none", color=CONTROL_COLORS[key],
                              markersize=5, label=CONTROL_LABELS[key])
                       for key, marker in zip(CONTROL_ORDER, ("o", "s", "^"))]
            fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.995),
                       ncol=3, handletextpad=0.3, columnspacing=0.9)
        return fig, axes

    def layout_audit(fig, axes) -> dict:
        """Record geometric candidates. Actual image inspection is a separate gate."""
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        outside, texts = [], []
        hidden_ticks = set()
        for ax in axes:
            for ticker, limits in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
                low, high = sorted(limits)
                for tick in ticker.get_major_ticks() + ticker.get_minor_ticks():
                    if tick.get_loc() < low - 1e-9 or tick.get_loc() > high + 1e-9:
                        hidden_ticks.update((id(tick.label1), id(tick.label2)))
        for text in fig.findobj(Text):
            if not text.get_visible() or not text.get_text().strip() or id(text) in hidden_ticks:
                continue
            box = text.get_window_extent(renderer)
            item = {"text": text.get_text(), "bounds_px": [round(v, 2) for v in box.bounds]}
            texts.append((text, box, item))
            if box.x0 < -0.5 or box.y0 < -0.5 or box.x1 > fig.bbox.width + 0.5 or box.y1 > fig.bbox.height + 0.5:
                outside.append(item)
        overlaps = []
        for index, (left, a, a_item) in enumerate(texts):
            for right, b, b_item in texts[index + 1:]:
                if left is right:
                    continue
                dx, dy = min(a.x1, b.x1) - max(a.x0, b.x0), min(a.y1, b.y1) - max(a.y0, b.y0)
                if dx > 1 and dy > 1:
                    overlaps.append({"left": a_item, "right": b_item})
        return {"status": "REQUIRES_ACTUAL_IMAGE_REVIEW",
                "data_frames": [{"panel": chr(97 + i), "bounds": list(ax.get_position().bounds),
                                 "xlim": list(ax.get_xlim()), "ylim": list(ax.get_ylim())}
                                for i, ax in enumerate(axes)],
                "text_outside_canvas": outside, "text_overlap_candidates": overlaps}

    def file_sha256(path: Path) -> str:
        return hashlib.sha256(path.read_bytes()).hexdigest()

    def export_result(fig, axes, directory: Path, stem: str, *, caption: str,
                      sources: dict[str, Path], documented_empty_axes=()) -> dict:
        """Export real results only, with exact input hashes and no inferred visual pass."""
        empty = [ax for ax in axes if not ax.has_data()]
        if set(empty) != set(documented_empty_axes):
            raise ValueError("Empty panels require explicit source-backed missing-status documentation")
        if any(not any(t.get_text() == "No qualified trajectory" for t in ax.texts) for ax in empty):
            raise ValueError("Each documented empty panel must visibly identify its qualification status")
        if not sources or not caption.strip() or not axes:
            raise ValueError("Every panel needs real data, a caption and traced source files before export")
        before = {name: file_sha256(path) for name, path in sources.items()}
        audit = layout_audit(fig, axes)
        typography = []
        for text in fig.findobj(Text):
            if not text.get_visible() or not text.get_text().strip():
                continue
            is_panel_title = any(text is ax.title or text is ax._left_title for ax in axes)
            expected_size = 12 if is_panel_title else 11
            expected_weight = "bold" if is_panel_title else "normal"
            prop = text.get_fontproperties()
            font_path = Path(findfont(prop, fallback_to_default=False))
            if (FontProperties(fname=str(font_path)).get_name() != FONT
                    or abs(text.get_fontsize() - expected_size) > 1e-9
                    or prop.get_weight() != expected_weight or text.get_color() != "black"):
                raise ValueError(f"Unexpected typography: {text.get_text()!r}")
            typography.append({"text": text.get_text(), "font_file": str(font_path),
                               "size_pt": expected_size, "weight": expected_weight, "colour": "black"})
        if audit["text_outside_canvas"]:
            raise ValueError("Figure has text outside its canvas; adjust layout before exporting")
        directory.mkdir(parents=True, exist_ok=True)
        pdf, svg = directory / f"{stem}.pdf", directory / f"{stem}.svg"
        fig.savefig(pdf, metadata={"CreationDate": None, "ModDate": None})
        fig.savefig(svg, metadata={"Date": None})
        outputs = [pdf, svg]
        for suffix, dpi in (("", 600), ("_word", 300)):
            size = tuple(round(value * dpi) for value in fig.get_size_inches())
            png = directory / f"{stem}{suffix}.png"
            fig.savefig(png, dpi=dpi)
            with Image.open(png) as image:
                rgb = image.convert("RGB")
                if rgb.size != size:
                    rgb = rgb.resize(size, Image.Resampling.LANCZOS)
                rgb.save(png)
            outputs.append(png)
        (directory / f"{stem}_caption.txt").write_text(caption.strip() + "\n", encoding="utf-8")
        if before != {name: file_sha256(path) for name, path in sources.items()}:
            raise ValueError("Scientific input changed during figure rendering")
        result = {"figure": stem, "sources": before, "outputs": {p.name: file_sha256(p) for p in outputs},
                  "layout_audit": audit, "typography_audit": {"status": "PASS", "text": typography},
                  "caption_word_count": len(caption.split()), "actual_visual_review": "PENDING"}
        (directory / f"{stem}_qa.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
        return result

    def point_summary(ax, frame, metric, ylabel, *, scale=1.0, controls=CONTROL_ORDER,
                      signed=False, percentage=False):
        """Show every finite scenario plus medians/IQRs, never a selected favourable subset."""
        records = []
        offsets = np.linspace(-0.23, 0.23, len(controls))
        for context_index, context in enumerate(CONTEXT_ORDER):
            cases = frame[frame.context.eq(context)]
            if all(column in cases for column in ("composition_id", "layout")):
                for _, case in cases.groupby(["composition_id", "layout"], sort=True):
                    x_line, y_line = [], []
                    for control, offset in zip(controls, offsets):
                        part = case[case.control.eq(control)]
                        if len(part) == 1 and pd.notna(part.iloc[0][metric]):
                            x_line.append(context_index + offset)
                            y_line.append(float(part.iloc[0][metric]) * scale)
                    if len(y_line) > 1:
                        ax.plot(x_line, y_line, color="#BDB9B2", linewidth=.6, alpha=.65, zorder=1)
            for control, offset in zip(controls, offsets):
                selected = frame[frame.context.eq(context) & frame.control.eq(control)].copy()
                values = pd.to_numeric(selected[metric], errors="raise").to_numpy(dtype=float) * scale
                finite = values[np.isfinite(values)]
                if finite.size:
                    x = context_index + offset
                    jitter = np.linspace(-0.045, 0.045, finite.size) if finite.size > 1 else np.zeros(1)
                    ax.scatter(x + jitter, finite, s=12, color=CONTROL_COLORS[control], alpha=0.78,
                               marker=CONTROL_MARKERS[control], linewidths=.2, edgecolors="white", zorder=2)
                    low, median, high = np.quantile(finite, [0.25, 0.5, 0.75])
                    ax.errorbar(x, median, yerr=[[median - low], [high - median]],
                                fmt=CONTROL_MARKERS[control], color=CONTROL_COLORS[control],
                                markersize=4.5, linewidth=1.5, capsize=2.3, zorder=3)
                else:
                    low = median = high = np.nan
                records.append({"context": context, "control": control, "metric": metric,
                                "declared_cases": len(selected), "finite_cases": len(finite),
                                "q25_display": low, "median_display": median, "q75_display": high})
        if signed:
            ax.axhline(0, color="#A5A5A5", linewidth=0.65, zorder=0)
        if percentage:
            finite_values = pd.to_numeric(frame[metric], errors="raise").to_numpy(float) * scale
            finite_values = finite_values[np.isfinite(finite_values)]
            if len(finite_values):
                span = max(np.ptp(finite_values), 8)
                ax.set_ylim(max(-2, float(finite_values.min()) - span * .12),
                            min(102, float(finite_values.max()) + span * .12))
        if ax.get_position().width * ax.figure.get_figwidth() * 25.4 < 55:
            ax.set_xticks(range(4), list(CONTEXT_LABELS.values()), rotation=75, ha="right")
        else:
            ax.set_xticks(range(4), CONTEXT_SHORT)
        ax.set_xlim(-0.55, 3.55)
        ax.set_ylabel(ylabel)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.ticklabel_format(axis="y", style="plain", useOffset=False)
        return records

    def read_annual_tables(root: Path) -> dict[str, pd.DataFrame]:
        names = ("ANNUAL_DIRECT", "ANNUAL_SERVICE_SELECTIONS", "CONTROL_COMPARISONS",
                 "SPATIAL_PAIRS", "LAYOUT_PAIRS", "STATIC_ZONE_INTERVAL_COMPATIBILITY")
        tables = {name: pd.read_csv(root / f"{name}.csv", keep_default_na=False,
                                    na_values=[""]) for name in names}
        direct = tables["ANNUAL_DIRECT"]
        if len(direct) != 324 or set(direct.control) != set(CONTROL_ORDER):
            raise ValueError("Expected the complete 324-row three-control direct design")
        if len(tables["ANNUAL_SERVICE_SELECTIONS"]) != 1296:
            raise ValueError("Expected every controller, space and common-service fraction")
        return tables

    def write_panel_summaries(directory, number, records):
        pd.DataFrame(records).to_csv(directory / f"Figure_{number:02d}_numerical_summary.csv", index=False)

    def empirical_steps(ax, frame, metric, xlabel, scale=1.0):
        """Empirical case distributions, with no fitted density or synthetic observations."""
        for control in CONTROL_ORDER:
            values = np.sort(pd.to_numeric(frame.loc[frame.control.eq(control), metric]).dropna().to_numpy() * scale)
            if len(values):
                cumulative = np.arange(1, len(values) + 1) / len(values) * 100
                ax.step(values, cumulative, where="post", color=CONTROL_COLORS[control], linewidth=1.6)
                ax.scatter(values, cumulative, color=CONTROL_COLORS[control],
                           marker=CONTROL_MARKERS[control], s=12, linewidths=0, zorder=3)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Cases at or below (%)")
        ax.set_ylim(-3, 103)
        ax.yaxis.set_major_locator(MaxNLocator(4))
        ax.xaxis.set_major_locator(MaxNLocator(4))


    DPI = 600
    output = INTERMEDIATE / LABEL
    output.mkdir(parents=True, exist_ok=True)
    annual = RESULTS / "Joint_main/annual"
    direct = pd.read_csv(annual / "ANNUAL_DIRECT.csv")
    direct = direct[direct.permission.eq("native_zone")].copy()
    if len(direct) != 144 or direct.composition_id.nunique() != 6:
        raise ValueError("Expected six compositions, two layouts and four weather contexts")
    direct["strict_profile_coverage_pct"] = 100 * direct.profiles_with_no_occupied_hour_interval_violation / direct.assigned_profile_count
    valid = direct[direct.selection_status.eq("selected")].copy()
    keys = ["context", "composition_id", "layout"]
    complete_keys = valid.groupby(keys).control.nunique()
    complete_keys = complete_keys[complete_keys.eq(3)].reset_index()[keys]
    complete = valid.merge(complete_keys, on=keys, validate="many_to_one")
    if len(complete) != 72:
        raise ValueError("Expected 24 complete three-control conditions")
    pairs = pd.read_csv(annual / "CONTROL_COMPARISONS.csv")
    pairs = pairs.query("permission == 'native_zone' and comparison_mode == 'direct' and status == 'matched'")
    hierarchy = pd.read_csv(RESULTS / "decision_value/INFORMATION_HIERARCHY_SUMMARY.csv")
    information = hierarchy.loc[
        hierarchy.evidence_block.eq("zone_mean_to_full_profile_decision_review")
    ].copy()
    required_metrics = {
        "design_cells",
        "full_profile_resolvable_cells",
        "zone_mean_safe_cells",
        "distribution_required_cells",
        "finite_library_unresolved_cells",
        "zone_mean_selected_plan_fails_full_review_cells",
    }
    if set(information.metric) < required_metrics:
        raise ValueError("Figure 2f information hierarchy is incomplete")
    counts = {
        metric: int(information.loc[information.metric.eq(metric), "count"].iloc[0])
        for metric in required_metrics
    }
    denominators = {
        metric: int(information.loc[information.metric.eq(metric), "denominator"].iloc[0])
        for metric in required_metrics
    }
    if counts["zone_mean_safe_cells"] + counts["distribution_required_cells"] != counts["full_profile_resolvable_cells"]:
        raise ValueError("Resolved Figure 2f decisions are not partitioned")
    if counts["full_profile_resolvable_cells"] + counts["finite_library_unresolved_cells"] != counts["design_cells"]:
        raise ValueError("Resolved and unresolved Figure 2f decisions are not partitioned")
    if denominators["zone_mean_safe_cells"] != counts["full_profile_resolvable_cells"]:
        raise ValueError("Figure 2f mean-sufficient denominator is not the resolved set")
    if denominators["distribution_required_cells"] != counts["full_profile_resolvable_cells"]:
        raise ValueError("Figure 2f distribution-required denominator is not the resolved set")
    if denominators["zone_mean_selected_plan_fails_full_review_cells"] != counts["full_profile_resolvable_cells"]:
        raise ValueError("Figure 2f failed-selection denominator is not the resolved set")
    if denominators["finite_library_unresolved_cells"] != counts["design_cells"]:
        raise ValueError("Figure 2f unresolved decisions lost the full-design denominator")
    fig, axes = six_panel_canvas([
        "Average temperature mismatch", "Least-served group mismatch",
        "Temperature-range coverage", "Least-served profiles",
        "Service changes together", "Demand information needed"],
        bounds=[(19,125,67,35),(109,125,67,35),(19,72,67,35),
                (109,72,67,35),(19,19,67,35),(109,19,67,35)])
    summary=[]
    for panel,metric,label,scale,pct in (
        (0,"mean_reference_mismatch_c","Mean deviation (°C)",1,False),
        (1,"fixed_tail_reference_mismatch_c","Tail deviation (°C)",1,False)):
        summary += [dict(panel=chr(97+panel),**record) for record in point_summary(
            axes[panel], direct, metric, label, scale=scale, percentage=pct)]
    empirical_steps(axes[2], complete, "modeled_interval_person_hour_coverage", "Covered person-hours (%)",100)
    empirical_steps(axes[3], complete, "profile_coverage_p10", "10th-percentile coverage (%)",100)
    for control in ALTERNATIVES:
        group=pairs[pairs.control.eq(control)]
        axes[4].scatter(group.delta_interval_coverage_percentage_points,
            group.delta_modeled_interval_mean_excess_c, s=22, marker=CONTROL_MARKERS[control],
            color=CONTROL_COLORS[control], edgecolors="white", linewidths=.35,alpha=.85)
    axes[4].axhline(0,color="#ADADAD",linewidth=.65)
    axes[4].axvline(0,color="#ADADAD",linewidth=.65)
    axes[4].set_xlabel("Coverage change\n(percentage points)",labelpad=3)
    axes[4].set_ylabel("Outside-range change (°C)")
    axes[4].xaxis.set_major_locator(MaxNLocator(4))
    axes[4].yaxis.set_major_locator(MaxNLocator(4))

    resolved = counts["full_profile_resolvable_cells"]
    sufficient = counts["zone_mean_safe_cells"]
    distribution = counts["distribution_required_cells"]
    failed = counts["zone_mean_selected_plan_fails_full_review_cells"]
    unresolved = counts["finite_library_unresolved_cells"]
    design = counts["design_cells"]
    ax = axes[5]
    # Three aligned count rows replace the former annotation-only layout.
    # The middle bar is nested within the distribution-required segment.
    ax.barh(1.20, sufficient, left=0, height=0.25, color="#A9C7D4", edgecolor="white")
    ax.barh(1.20, distribution, left=sufficient, height=0.25, color="#D7B2A3", edgecolor="white")
    ax.text(sufficient / 2, 1.20, f"{sufficient}", ha="center", va="center")
    ax.text(sufficient + distribution / 2, 1.20, f"{distribution}", ha="center", va="center")
    ax.text(sufficient / 2, 1.49, "Mean\nsufficient", ha="center", va="center")
    ax.text(sufficient + distribution / 2, 1.49, "Distribution\nrequired", ha="center", va="center")
    ax.barh(0.69, failed, left=sufficient, height=0.15, color="#C78470", edgecolor="white")
    ax.text(sufficient + failed / 2, 0.69, f"{failed}", ha="center", va="center")
    ax.barh(0.16, resolved, left=0, height=0.25, color="#B8C6C8", edgecolor="white")
    ax.barh(0.16, unresolved, left=resolved, height=0.25, color="#D7D7D7",
            edgecolor="white", hatch="////")
    ax.text(resolved / 2, 0.16, f"{resolved} resolved", ha="center", va="center")
    ax.text(resolved + unresolved / 2, 0.16, f"{unresolved} unresolved", ha="center", va="center")
    ax.set_xlim(0, design)
    ax.set_ylim(-0.08, 1.70)
    ax.set_xticks([0, 250, 500, 750, 1000])
    ax.set_yticks([1.20, 0.69, 0.16],
                  ["Resolvable", "Selected plan\nfailed review", "Tested library"])
    ax.set_xlabel("Decision settings (count)")
    ax.tick_params(axis="y", pad=3)
    ax.spines["left"].set_visible(True)
    ax.spines["bottom"].set_visible(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    direct.to_csv(output/"Figure_02_all_direct_source.csv",index=False)
    complete.to_csv(output/"Figure_02_complete_three_source.csv",index=False)
    pairs.to_csv(output/"Figure_02_service_change_source.csv",index=False)
    information.to_csv(output/"Figure_02_information_sufficiency_source.csv",index=False)
    summary += [
        {"panel":"f", "metric":metric, "count":value,
         "comparison":"zone_mean_to_full_profile_decision_review"}
        for metric,value in counts.items()
    ]
    pd.DataFrame(summary).to_csv(output/"Figure_02_numerical_summary.csv",index=False)
    for ax in axes[:5]:
        ax.yaxis.set_label_coords(-.17,.5)
    caption=("Figure 2 | Thermal service from three controls and the demand information needed for plan review. "
        "Six fixed demand compositions, two placements and four building–weather contexts give 48 conditions per control, "
        "with independent commands for existing thermal zones. Personal range uses reference demand intervals defined in "
        "Methods. Points are engineering-checked cases; larger symbols and whiskers in a,b show medians and interquartile "
        "ranges. Grey lines connect the same condition. a, Occupied-person-hour-weighted absolute deviation from individual "
        "reference temperatures. b, Deviation for the same 27 profiles worst served by common 25°C operation. c,d, Empirical "
        "cumulative distributions for the 24 conditions available under all three controls: occupied person-hours inside "
        "assigned ranges (c) and tenth-percentile individual coverage (d). e, Paired changes from Uniform average in coverage "
        "and mean degrees outside the ranges; each point is one matched comparison. f, Separate fixed-plan-library review of "
        "information sufficiency. Of 606 decisions with a full-profile-qualified action, 275 were mean sufficient and 331 "
        "required the within-zone demand distribution; the zone-mean-selected plan failed full-profile review in 250 of the "
        "latter. The 402 of 1,008 design cells without a full-profile-qualified action in the tested library are shown "
        "separately and are not included in the 606-decision denominator. Panels a–e use simulated temperatures and "
        "reconstructed demand references. Availability by condition is retained in Source Data.")
    (RESULTS/"captions"/(LABEL+".txt")).write_text(caption,encoding="utf-8")
    review=layout_audit(fig,axes)
    review.update(complete_three_control_conditions=len(complete)//3, direct_selected=len(valid),
                  matched_service_change_pairs=len(pairs), visual_review="PENDING")
    (output/"ALIGNMENT_REVIEW.json").write_text(json.dumps(review,indent=2),encoding="utf-8")
    save_plot(fig,LABEL,dpi=DPI)


draw_figure_02()


In [ ]:
# Figure_03: edit this cell and rerun to update this figure.
def draw_figure_03():
    LABEL = "Figure_03"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import argparse

    import hashlib

    import json

    from pathlib import Path

    import matplotlib

    import matplotlib.pyplot as plt

    from matplotlib.font_manager import FontProperties, findfont

    from matplotlib.lines import Line2D

    from matplotlib.text import Text

    from PIL import Image

    import numpy as np

    import pandas as pd

    from matplotlib.colors import LinearSegmentedColormap

    from matplotlib.ticker import MaxNLocator

    FONT = "Times New Roman"

    WIDTH_MM, HEIGHT_MM = 180, 183

    CONTROL_ORDER = ("uniform_average", "personal_center", "ltpref_interval")

    CONTROL_LABELS = {
        "uniform_average": "Uniform average",
        "personal_center": "Personal centre",
        "ltpref_interval": "Personal range",
    }

    CONTROL_COLORS = {
        "uniform_average": "#7F7F7F",
        "personal_center": "#C8A36A",
        "ltpref_interval": "#7FA995",
    }

    CONTEXT_ORDER = ("Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019")

    CONTEXT_COLORS = dict(zip(CONTEXT_ORDER, ("#C78470", "#AF94AD", "#6F9EBC", "#7FA995")))

    CONTEXT_LABELS = dict(zip(CONTEXT_ORDER, ("Miami", "New Delhi", "Port Angeles", "Seattle")))

    CONTEXT_SHORT = ("Miami", "New\nDelhi", "Port\nAngeles", "Seattle")

    CONTROL_MARKERS = dict(zip(CONTROL_ORDER, ("o", "s", "^")))

    ALTERNATIVES = CONTROL_ORDER[1:]

    DATA_BOUNDS_MM = tuple(
        (x, y, 67.0, 35.0) for y in (125.0, 72.0, 19.0) for x in (19.0, 109.0)
    )

    def configure_style() -> dict[str, str]:
        fonts = {}
        for weight in ("normal", "bold"):
            prop = FontProperties(family=FONT, weight=weight)
            path = Path(findfont(prop, fallback_to_default=False))
            if FontProperties(fname=str(path)).get_name() != FONT:
                raise ValueError(f"Exact {FONT} {weight} is required")
            fonts[weight] = str(path)
        plt.rcParams.update({
            "font.family": FONT, "font.size": 11, "text.color": "black",
            "axes.labelsize": 11, "axes.labelcolor": "black", "axes.labelpad": 6,
            "axes.titlesize": 12, "axes.titleweight": "bold",
            "axes.spines.top": False, "axes.spines.right": False,
            "axes.edgecolor": "#777777", "axes.linewidth": 0.75,
            "xtick.labelsize": 11, "ytick.labelsize": 11,
            "xtick.major.pad": 4, "ytick.major.pad": 4,
            "xtick.color": "black", "ytick.color": "black",
            "legend.fontsize": 11, "legend.frameon": False,
            "figure.facecolor": "white", "savefig.facecolor": "white",
            "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
            "svg.hashsalt": "thermal-heterogeneity-stage8",
        })
        return fonts

    def six_panel_canvas(titles: list[str], *, control_key: bool = True, bounds=None, height=None):
        """Create equal data frames with separate title, axis and shared-key bands."""
        if len(titles) != 6:
            raise ValueError("The main result template has exactly six data panels")
        configure_style()
        page_height = height or HEIGHT_MM
        fig = plt.figure(figsize=(WIDTH_MM / 25.4, page_height / 25.4))
        axes = []
        for index, (title, panel_bounds) in enumerate(zip(titles, bounds or DATA_BOUNDS_MM)):
            x, y, width, height_mm = panel_bounds
            ax = fig.add_axes((x / WIDTH_MM, y / page_height, width / WIDTH_MM, height_mm / page_height))
            ax.set_title(f"({chr(97 + index)}) {title}", loc="left", pad=8)
            ax.grid(False)
            ax.set_axisbelow(True)
            axes.append(ax)
        if control_key:
            handles = [Line2D([], [], marker=marker, linestyle="none", color=CONTROL_COLORS[key],
                              markersize=5, label=CONTROL_LABELS[key])
                       for key, marker in zip(CONTROL_ORDER, ("o", "s", "^"))]
            fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.995),
                       ncol=3, handletextpad=0.3, columnspacing=0.9)
        return fig, axes

    def layout_audit(fig, axes) -> dict:
        """Record geometric candidates. Actual image inspection is a separate gate."""
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        outside, texts = [], []
        hidden_ticks = set()
        for ax in axes:
            for ticker, limits in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
                low, high = sorted(limits)
                for tick in ticker.get_major_ticks() + ticker.get_minor_ticks():
                    if tick.get_loc() < low - 1e-9 or tick.get_loc() > high + 1e-9:
                        hidden_ticks.update((id(tick.label1), id(tick.label2)))
        for text in fig.findobj(Text):
            if not text.get_visible() or not text.get_text().strip() or id(text) in hidden_ticks:
                continue
            box = text.get_window_extent(renderer)
            item = {"text": text.get_text(), "bounds_px": [round(v, 2) for v in box.bounds]}
            texts.append((text, box, item))
            if box.x0 < -0.5 or box.y0 < -0.5 or box.x1 > fig.bbox.width + 0.5 or box.y1 > fig.bbox.height + 0.5:
                outside.append(item)
        overlaps = []
        for index, (left, a, a_item) in enumerate(texts):
            for right, b, b_item in texts[index + 1:]:
                if left is right:
                    continue
                dx, dy = min(a.x1, b.x1) - max(a.x0, b.x0), min(a.y1, b.y1) - max(a.y0, b.y0)
                if dx > 1 and dy > 1:
                    overlaps.append({"left": a_item, "right": b_item})
        return {"status": "REQUIRES_ACTUAL_IMAGE_REVIEW",
                "data_frames": [{"panel": chr(97 + i), "bounds": list(ax.get_position().bounds),
                                 "xlim": list(ax.get_xlim()), "ylim": list(ax.get_ylim())}
                                for i, ax in enumerate(axes)],
                "text_outside_canvas": outside, "text_overlap_candidates": overlaps}

    def file_sha256(path: Path) -> str:
        return hashlib.sha256(path.read_bytes()).hexdigest()

    def export_result(fig, axes, directory: Path, stem: str, *, caption: str,
                      sources: dict[str, Path], documented_empty_axes=()) -> dict:
        """Export real results only, with exact input hashes and no inferred visual pass."""
        empty = [ax for ax in axes if not ax.has_data()]
        if set(empty) != set(documented_empty_axes):
            raise ValueError("Empty panels require explicit source-backed missing-status documentation")
        if any(not any(t.get_text() == "No qualified trajectory" for t in ax.texts) for ax in empty):
            raise ValueError("Each documented empty panel must visibly identify its qualification status")
        if not sources or not caption.strip() or not axes:
            raise ValueError("Every panel needs real data, a caption and traced source files before export")
        before = {name: file_sha256(path) for name, path in sources.items()}
        audit = layout_audit(fig, axes)
        typography = []
        for text in fig.findobj(Text):
            if not text.get_visible() or not text.get_text().strip():
                continue
            is_panel_title = any(text is ax.title or text is ax._left_title for ax in axes)
            expected_size = 12 if is_panel_title else 11
            expected_weight = "bold" if is_panel_title else "normal"
            prop = text.get_fontproperties()
            font_path = Path(findfont(prop, fallback_to_default=False))
            if (FontProperties(fname=str(font_path)).get_name() != FONT
                    or abs(text.get_fontsize() - expected_size) > 1e-9
                    or prop.get_weight() != expected_weight or text.get_color() != "black"):
                raise ValueError(f"Unexpected typography: {text.get_text()!r}")
            typography.append({"text": text.get_text(), "font_file": str(font_path),
                               "size_pt": expected_size, "weight": expected_weight, "colour": "black"})
        if audit["text_outside_canvas"]:
            raise ValueError("Figure has text outside its canvas; adjust layout before exporting")
        directory.mkdir(parents=True, exist_ok=True)
        pdf, svg = directory / f"{stem}.pdf", directory / f"{stem}.svg"
        fig.savefig(pdf, metadata={"CreationDate": None, "ModDate": None})
        fig.savefig(svg, metadata={"Date": None})
        outputs = [pdf, svg]
        for suffix, dpi in (("", 600), ("_word", 300)):
            size = tuple(round(value * dpi) for value in fig.get_size_inches())
            png = directory / f"{stem}{suffix}.png"
            fig.savefig(png, dpi=dpi)
            with Image.open(png) as image:
                rgb = image.convert("RGB")
                if rgb.size != size:
                    rgb = rgb.resize(size, Image.Resampling.LANCZOS)
                rgb.save(png)
            outputs.append(png)
        (directory / f"{stem}_caption.txt").write_text(caption.strip() + "\n", encoding="utf-8")
        if before != {name: file_sha256(path) for name, path in sources.items()}:
            raise ValueError("Scientific input changed during figure rendering")
        result = {"figure": stem, "sources": before, "outputs": {p.name: file_sha256(p) for p in outputs},
                  "layout_audit": audit, "typography_audit": {"status": "PASS", "text": typography},
                  "caption_word_count": len(caption.split()), "actual_visual_review": "PENDING"}
        (directory / f"{stem}_qa.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
        return result

    def point_summary(ax, frame, metric, ylabel, *, scale=1.0, controls=CONTROL_ORDER,
                      signed=False, percentage=False):
        """Show every finite scenario plus medians/IQRs, never a selected favourable subset."""
        records = []
        offsets = np.linspace(-0.23, 0.23, len(controls))
        for context_index, context in enumerate(CONTEXT_ORDER):
            cases = frame[frame.context.eq(context)]
            if all(column in cases for column in ("composition_id", "layout")):
                for _, case in cases.groupby(["composition_id", "layout"], sort=True):
                    x_line, y_line = [], []
                    for control, offset in zip(controls, offsets):
                        part = case[case.control.eq(control)]
                        if len(part) == 1 and pd.notna(part.iloc[0][metric]):
                            x_line.append(context_index + offset)
                            y_line.append(float(part.iloc[0][metric]) * scale)
                    if len(y_line) > 1:
                        ax.plot(x_line, y_line, color="#BDB9B2", linewidth=.6, alpha=.65, zorder=1)
            for control, offset in zip(controls, offsets):
                selected = frame[frame.context.eq(context) & frame.control.eq(control)].copy()
                values = pd.to_numeric(selected[metric], errors="raise").to_numpy(dtype=float) * scale
                finite = values[np.isfinite(values)]
                if finite.size:
                    x = context_index + offset
                    jitter = np.linspace(-0.045, 0.045, finite.size) if finite.size > 1 else np.zeros(1)
                    ax.scatter(x + jitter, finite, s=12, color=CONTROL_COLORS[control], alpha=0.78,
                               marker=CONTROL_MARKERS[control], linewidths=.2, edgecolors="white", zorder=2)
                    low, median, high = np.quantile(finite, [0.25, 0.5, 0.75])
                    ax.errorbar(x, median, yerr=[[median - low], [high - median]],
                                fmt=CONTROL_MARKERS[control], color=CONTROL_COLORS[control],
                                markersize=4.5, linewidth=1.5, capsize=2.3, zorder=3)
                else:
                    low = median = high = np.nan
                records.append({"context": context, "control": control, "metric": metric,
                                "declared_cases": len(selected), "finite_cases": len(finite),
                                "q25_display": low, "median_display": median, "q75_display": high})
        if signed:
            ax.axhline(0, color="#A5A5A5", linewidth=0.65, zorder=0)
        if percentage:
            finite_values = pd.to_numeric(frame[metric], errors="raise").to_numpy(float) * scale
            finite_values = finite_values[np.isfinite(finite_values)]
            if len(finite_values):
                span = max(np.ptp(finite_values), 8)
                ax.set_ylim(max(-2, float(finite_values.min()) - span * .12),
                            min(102, float(finite_values.max()) + span * .12))
        if ax.get_position().width * ax.figure.get_figwidth() * 25.4 < 55:
            ax.set_xticks(range(4), list(CONTEXT_LABELS.values()), rotation=45, ha="right", rotation_mode="anchor")
        else:
            ax.set_xticks(range(4), CONTEXT_SHORT)
        ax.set_xlim(-0.55, 3.55)
        ax.set_ylabel(ylabel)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.ticklabel_format(axis="y", style="plain", useOffset=False)
        return records

    def read_annual_tables(root: Path) -> dict[str, pd.DataFrame]:
        names = ("ANNUAL_DIRECT", "ANNUAL_SERVICE_SELECTIONS", "CONTROL_COMPARISONS",
                 "SPATIAL_PAIRS", "LAYOUT_PAIRS", "STATIC_ZONE_INTERVAL_COMPATIBILITY")
        tables = {name: pd.read_csv(root / f"{name}.csv", keep_default_na=False,
                                    na_values=[""]) for name in names}
        direct = tables["ANNUAL_DIRECT"]
        if len(direct) != 324 or set(direct.control) != set(CONTROL_ORDER):
            raise ValueError("Expected the complete 324-row three-control direct design")
        if len(tables["ANNUAL_SERVICE_SELECTIONS"]) != 1296:
            raise ValueError("Expected every controller, space and common-service fraction")
        return tables

    def write_panel_summaries(directory, number, records):
        pd.DataFrame(records).to_csv(directory / f"Figure_{number:02d}_numerical_summary.csv", index=False)


    DPI = 600
    output=INTERMEDIATE/LABEL
    output.mkdir(parents=True,exist_ok=True)
    root=RESULTS/"Joint_main"
    annual=root/"annual"
    pairs=pd.read_csv(annual/"CONTROL_COMPARISONS.csv")
    native=pairs[pairs.permission.eq("native_zone")].copy()
    direct=native.query("comparison_mode == 'direct'").copy()
    common=native.query("comparison_mode == 'equal_service' and service_fraction == 1").copy()
    candidates=pd.read_csv(annual/"ANNUAL_ALL_CANDIDATES.csv")
    reference=candidates.drop_duplicates("run_id").set_index("run_id").annual_operational_co2_kg
    direct["delta_carbon_pct"]=100*direct.delta_annual_operational_co2_kg/direct.reference_run_id.map(reference)
    WIDTH_MM,HEIGHT_MM=180,224
    bounds=[(19,177,67,29),(109,177,67,29),(19,123,157,31),
            (19,70,43,27),(76,70,43,27),(133,70,43,27),
            (19,16,67,25),(109,16,67,25)]
    configure_style()
    fig=plt.figure(figsize=(WIDTH_MM/25.4,HEIGHT_MM/25.4))
    axes=[]
    titles=["Direct energy change","Direct carbon change","Where conditioning is added or avoided",
            "Common annual\nrequirements","Selected\noperating plans","Available\nservice options",
            "Annual and seasonal service","Seasonal energy offsets"]
    for i,(title,b) in enumerate(zip(titles,bounds)):
        ax=fig.add_axes(np.array(b)/[WIDTH_MM,HEIGHT_MM,WIDTH_MM,HEIGHT_MM])
        ax.set_title(f"({chr(97+i)}) {title}",loc="left",pad=8)
        axes.append(ax)
    fig.legend(handles=[Line2D([],[],color=CONTROL_COLORS[c],marker=CONTROL_MARKERS[c],linestyle="none",
        label=CONTROL_LABELS[c],markersize=5) for c in CONTROL_ORDER],ncol=3,loc="upper center",bbox_to_anchor=(.5,.998),
        handletextpad=.25,columnspacing=.8)
    for i,data,metric,label in ((0,direct,"delta_site_pct","Energy change (%)"),
                                (1,direct,"delta_carbon_pct","Carbon change (%)"),
                                (3,common,"delta_site_pct","Energy change (%)")):
        point_summary(axes[i],data,metric,label,controls=ALTERNATIVES,signed=True)
    ledger=pd.read_csv(root/"SIGNED_ENDUSE_REALLOCATION.csv")
    palette={"heating_electric":"#C78470","cooling_electric":"#6F9EBC",
             "fans_electric":"#AF94AD","gas_hhv":"#C8A36A"}
    labels={"heating_electric":"Heating electricity","cooling_electric":"Cooling electricity",
            "fans_electric":"Fans","gas_hhv":"Heating gas"}
    for index,context in enumerate(CONTEXT_ORDER):
        for offset,control in zip((-.19,.19),ALTERNATIVES):
            group=ledger[ledger.context.eq(context)&ledger.control.eq(control)]
            totals=[0.,0.]
            for component,color in palette.items():
                part=group[group.component.eq(component)]
                for side,col,sign in ((0,"added_kwh",1),(1,"avoided_kwh",-1)):
                    val=(part[col]/part.floor_area_m2).mean()
                    if np.isfinite(val):
                        axes[2].bar(index+offset,sign*val,width=.31,bottom=totals[side],
                            color=color,edgecolor="white",linewidth=.45)
                        totals[side]+=sign*val
            unique=group.drop_duplicates(["composition_id","layout"])
            values=unique.pair_net_site_kwh/unique.floor_area_m2
            axes[2].scatter(index+offset+np.linspace(-.05,.05,len(values)),values,s=14,
                color=CONTROL_COLORS[control],marker=CONTROL_MARKERS[control],edgecolors="white",linewidths=.35,zorder=4)
            if len(group):
                axes[2].scatter(index+offset,totals[0]+totals[1],s=37,facecolors="none",
                    edgecolors="black",marker=CONTROL_MARKERS[control],linewidths=.9,zorder=5)
    from matplotlib.patches import Patch
    axes[2].legend(handles=[Patch(facecolor=c,label=labels[k]) for k,c in palette.items()],
        ncol=2,loc="upper right",bbox_to_anchor=(1,1.32),handlelength=.9,handletextpad=.25,
        columnspacing=1.2,labelspacing=.15,borderpad=0)
    axes[2].set_xticks(range(4),CONTEXT_SHORT)
    axes[2].set_ylabel("Added / avoided\n(kWh/m²/year)")
    axes[2].axhline(0,color="#ADADAD",linewidth=.65)
    axes[2].yaxis.set_major_locator(MaxNLocator(4))
    audit=pd.read_csv(root/"service/DIRECT_COMMON_PAIR_AUDIT.csv")
    status=[]
    for c in ALTERNATIVES:
        g=audit[audit.control.eq(c)&audit.status.eq("matched")]
        counts=[int(g.same_physical_run.sum()),int((~g.same_physical_run&g.delta_site_pct.lt(0)).sum()),
                int((~g.same_physical_run&g.delta_site_pct.ge(0)).sum())]
        bottom=0
        for count,color in zip(counts,("#BCBEBB","#91B1C4","#CC9886")):
            axes[4].bar(ALTERNATIVES.index(c),count,bottom=bottom,width=.64,color=color,edgecolor="white",linewidth=.6)
            if count>=25:
                axes[4].text(ALTERNATIVES.index(c),bottom+count/2,str(count),ha="center",va="center")
            bottom+=count
        status.append(dict(control=c,same=counts[0],different_lower_energy=counts[1],different_higher_energy=counts[2]))
    axes[4].set_xticks([0,1],["Personal\ncentre","Personal\nrange"])
    axes[4].set_ylabel("Choices / 120")
    axes[4].set_ylim(0,125)
    axes[4].set_yticks([0,60,120])
    selections=pd.read_csv(annual/"ANNUAL_SERVICE_SELECTIONS.csv")
    selections=selections[selections.permission.eq("native_zone")]
    for c in CONTROL_ORDER:
        counts=selections[selections.control.eq(c)].groupby("service_fraction").selection_status.apply(lambda s:s.eq("selected").sum())
        axes[5].plot(counts.index,counts.values,marker=CONTROL_MARKERS[c],color=CONTROL_COLORS[c],markersize=4,linewidth=1.2)
    axes[5].set_ylim(-1,50)
    axes[5].set_yticks([0,24,48])
    axes[5].set_xticks([.5,.75,1],["0.5","0.75","1"])
    axes[5].set_xlabel("Allowed mismatch /\nreference")
    axes[5].set_ylabel("Choices / 48")
    diag=pd.read_csv(root/"seasonal/SEASONAL_SERVICE_BUDGET_DIAGNOSTIC.csv")
    diag=diag[diag.context.eq("Seattle_small2019")&diag.comparison_mode.eq("equal_service")&diag.data_status.eq("qualified")]
    periods=["Annual","Spring","Summer","Autumn","Winter"]
    seasonal=[]
    for c,off in zip(CONTROL_ORDER,[-.19,0,.19]):
        counts=[]
        for period in periods:
            g=diag[diag.control.eq(c)&diag.period.eq(period)]
            if len(g)!=12:
                raise ValueError("Each seasonal count needs all 12 fixed conditions")
            counts.append(int(g.both_budgets_pass.astype(str).str.lower().eq("true").sum()))
            seasonal.append(dict(control=c,period=period,cases=12,passing=counts[-1]))
        axes[6].scatter(np.arange(5)+off,counts,s=23,color=CONTROL_COLORS[c],marker=CONTROL_MARKERS[c],edgecolors="white",linewidths=.3)
    axes[6].set_xticks(range(5),periods,rotation=45,ha="right",rotation_mode="anchor")
    axes[6].set_ylabel("Choices passing / 12")
    axes[6].set_ylim(-.5,12.7)
    axes[6].set_yticks([0,6,12])
    compensation=pd.read_csv(root/"seasonal/TEMPORAL_COMPENSATION_PAIRS.csv")
    compensation=compensation[compensation.comparison_mode.eq("equal_service")
        &compensation.service_fraction.eq(1)&compensation.pair_status.eq("matched")].copy()
    period_totals=pd.read_csv(root/"seasonal/PERIOD_TOTALS.csv")
    annual_totals=period_totals[period_totals.period_type.eq("annual")].drop_duplicates(
        ["data_source","run_id"]).set_index(["data_source","run_id"]).purchased_site_kwh
    compensation["reference_annual_site_kwh"]=[annual_totals.loc[(source,run)]
        for source,run in zip(compensation.data_source,compensation.reference_run_id)]
    compensation["avoided_pct"]=100*compensation.seasonal_avoided_site_kwh/compensation.reference_annual_site_kwh
    compensation["added_pct"]=100*compensation.seasonal_added_site_kwh/compensation.reference_annual_site_kwh
    distinct=compensation[~compensation.same_physical_operation.astype(bool)].copy()
    if len(compensation)!=96 or compensation.same_physical_operation.astype(bool).sum()!=67 or len(distinct)!=29:
        raise ValueError("Seasonal offset panel requires 96 matched pairs, 67 identical and 29 distinct operations")
    for control in ALTERNATIVES:
        subset=distinct[distinct.alternative.eq(control)]
        axes[7].scatter(subset.avoided_pct,subset.added_pct,s=25,color=CONTROL_COLORS[control],
            marker=CONTROL_MARKERS[control],alpha=.82,edgecolors="white",linewidths=.35,zorder=3)
    limit=12
    axes[7].fill_between([0,limit],[0,limit],[limit,limit],color="#F3E6E1",alpha=.55,zorder=0)
    axes[7].fill_between([0,limit],[0,0],[0,limit],color="#E5EFF4",alpha=.55,zorder=0)
    axes[7].plot([0,limit],[0,limit],color="#8C8C8C",linestyle="--",linewidth=.8,zorder=1)
    axes[7].scatter([0],[0],s=55,facecolors="white",edgecolors=CONTROL_COLORS["uniform_average"],
        linewidths=1.2,zorder=4)
    axes[7].text(.4,10.7,"21 annual increases",ha="left",va="top")
    axes[7].text(6.9,.6,"8 annual reductions",ha="right",va="bottom")
    axes[7].set_xlim(-.35,7.1)
    axes[7].set_ylim(-.55,11.4)
    axes[7].set_xticks([0,3,6])
    axes[7].set_yticks([0,5,10])
    axes[7].set_xlabel("Seasonally avoided energy\n(% annual Uniform energy)")
    axes[7].set_ylabel("Seasonally added energy\n(% annual Uniform energy)")
    axes[3].set_xticks(range(4), CONTEXT_SHORT, rotation=45, ha="right", rotation_mode="anchor")
    for index in (0, 1, 6, 7):
        axes[index].yaxis.set_label_coords(-.13, .5)
    for index in (3, 4, 5):
        axes[index].yaxis.set_label_coords(-.14, .5)
    axes[2].yaxis.set_label_coords(-.045, .5)
    pd.concat([direct,common]).to_csv(output/"Figure_03_paired_source_data.csv",index=False)
    ledger.to_csv(output/"Figure_03_enduse_ledger_source.csv",index=False)
    pd.DataFrame(status).to_csv(output/"Figure_03_plan_choices_source.csv",index=False)
    pd.DataFrame(seasonal).to_csv(output/"Figure_03_seasonal_source.csv",index=False)
    compensation.to_csv(output/"Figure_03_temporal_offsets_source.csv",index=False)
    caption=("Figure 3 | Energy, carbon and the service supplied. Six demand compositions and two placements are combined. "
        "Personal range uses model-generated individual demand intervals. a,b, Paired direct changes in purchased heating, "
        "cooling and ventilation energy and operational carbon relative to Uniform average, under existing zone commands. "
        "Small points are matched conditions; larger symbols and whiskers show medians and interquartile ranges. "
        "c, Annual added and avoided hourly end-use energy per floor area. Stacks show component means, coloured symbols "
        "show individual net changes and open black symbols show the mean net change. Gas uses higher heating value. "
        "d, Minimum-energy choices under the same annual mean and fixed-tail temperature-mismatch limits. "
        "e, Across distinct permissions, choices identical to Uniform average (grey), or different with lower (blue) or higher "
        "(coral) energy, at reference limits. f, Available zone-level choices as both annual limits are tightened by the stated "
        "fraction of common-25°C values. g, Fixed annual choices in the Seattle representative office passing annual or "
        "corresponding seasonal limits. h, Seasonal energy added and avoided by the 29 personalized choices selecting a "
        "different physical operation under common annual limits, normalized by the corresponding Uniform-average annual "
        "energy. The diagonal marks zero annual change; the origin represents 67 choices selecting the same operation. "
        "Annual choices, demand compositions and the same 27 reference-tail profiles remain fixed in g,h. Carbon uses "
        "regional annual factors.")
    (RESULTS/"captions"/(LABEL+".txt")).write_text(caption,encoding="utf-8")
    (output/"ALIGNMENT_REVIEW.json").write_text(json.dumps(layout_audit(fig,axes),indent=2),encoding="utf-8")
    save_plot(fig,LABEL,dpi=DPI)


draw_figure_03()


In [ ]:
# Figure_04: edit this cell and rerun to update this figure.
def draw_figure_04():
    LABEL = "Figure_04"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import argparse

    import hashlib

    import json

    from pathlib import Path

    import matplotlib

    import matplotlib.pyplot as plt

    from matplotlib.font_manager import FontProperties, findfont

    from matplotlib.lines import Line2D

    from matplotlib.text import Text

    from PIL import Image

    import numpy as np

    import pandas as pd

    from matplotlib.colors import LinearSegmentedColormap

    from matplotlib.ticker import MaxNLocator

    FONT = "Times New Roman"

    WIDTH_MM, HEIGHT_MM = 180, 183

    CONTROL_ORDER = ("uniform_average", "personal_center", "ltpref_interval")

    CONTROL_LABELS = {
        "uniform_average": "Uniform average",
        "personal_center": "Personal centre",
        "ltpref_interval": "Personal range",
    }

    CONTROL_COLORS = {
        "uniform_average": "#7F7F7F",
        "personal_center": "#C8A36A",
        "ltpref_interval": "#7FA995",
    }

    CONTEXT_ORDER = ("Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019")

    CONTEXT_COLORS = dict(zip(CONTEXT_ORDER, ("#C78470", "#AF94AD", "#6F9EBC", "#7FA995")))

    CONTEXT_LABELS = dict(zip(CONTEXT_ORDER, ("Miami", "New Delhi", "Port Angeles", "Seattle")))

    CONTEXT_SHORT = ("Miami", "New\nDelhi", "Port\nAngeles", "Seattle")

    CONTROL_MARKERS = dict(zip(CONTROL_ORDER, ("o", "s", "^")))

    ALTERNATIVES = CONTROL_ORDER[1:]

    DATA_BOUNDS_MM = tuple(
        (x, y, 67.0, 35.0) for y in (125.0, 72.0, 19.0) for x in (19.0, 109.0)
    )

    def configure_style() -> dict[str, str]:
        fonts = {}
        for weight in ("normal", "bold"):
            prop = FontProperties(family=FONT, weight=weight)
            path = Path(findfont(prop, fallback_to_default=False))
            if FontProperties(fname=str(path)).get_name() != FONT:
                raise ValueError(f"Exact {FONT} {weight} is required")
            fonts[weight] = str(path)
        plt.rcParams.update({
            "font.family": FONT, "font.size": 11, "text.color": "black",
            "axes.labelsize": 11, "axes.labelcolor": "black", "axes.labelpad": 6,
            "axes.titlesize": 12, "axes.titleweight": "bold",
            "axes.spines.top": False, "axes.spines.right": False,
            "axes.edgecolor": "#777777", "axes.linewidth": 0.75,
            "xtick.labelsize": 11, "ytick.labelsize": 11,
            "xtick.major.pad": 4, "ytick.major.pad": 4,
            "xtick.color": "black", "ytick.color": "black",
            "legend.fontsize": 11, "legend.frameon": False,
            "figure.facecolor": "white", "savefig.facecolor": "white",
            "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
            "svg.hashsalt": "thermal-heterogeneity-stage8",
        })
        return fonts

    def six_panel_canvas(titles: list[str], *, control_key: bool = True, bounds=None, height=None):
        """Create equal data frames with separate title, axis and shared-key bands."""
        if len(titles) != 6:
            raise ValueError("The main result template has exactly six data panels")
        configure_style()
        page_height = height or HEIGHT_MM
        fig = plt.figure(figsize=(WIDTH_MM / 25.4, page_height / 25.4))
        axes = []
        for index, (title, panel_bounds) in enumerate(zip(titles, bounds or DATA_BOUNDS_MM)):
            x, y, width, height_mm = panel_bounds
            ax = fig.add_axes((x / WIDTH_MM, y / page_height, width / WIDTH_MM, height_mm / page_height))
            ax.set_title(f"({chr(97 + index)}) {title}", loc="left", pad=8)
            ax.grid(False)
            ax.set_axisbelow(True)
            axes.append(ax)
        if control_key:
            handles = [Line2D([], [], marker=marker, linestyle="none", color=CONTROL_COLORS[key],
                              markersize=5, label=CONTROL_LABELS[key])
                       for key, marker in zip(CONTROL_ORDER, ("o", "s", "^"))]
            fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.995),
                       ncol=3, handletextpad=0.3, columnspacing=0.9)
        return fig, axes

    def layout_audit(fig, axes) -> dict:
        """Record geometric candidates. Actual image inspection is a separate gate."""
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        outside, texts = [], []
        hidden_ticks = set()
        for ax in axes:
            for ticker, limits in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
                low, high = sorted(limits)
                for tick in ticker.get_major_ticks() + ticker.get_minor_ticks():
                    if tick.get_loc() < low - 1e-9 or tick.get_loc() > high + 1e-9:
                        hidden_ticks.update((id(tick.label1), id(tick.label2)))
        for text in fig.findobj(Text):
            if not text.get_visible() or not text.get_text().strip() or id(text) in hidden_ticks:
                continue
            box = text.get_window_extent(renderer)
            item = {"text": text.get_text(), "bounds_px": [round(v, 2) for v in box.bounds]}
            texts.append((text, box, item))
            if box.x0 < -0.5 or box.y0 < -0.5 or box.x1 > fig.bbox.width + 0.5 or box.y1 > fig.bbox.height + 0.5:
                outside.append(item)
        overlaps = []
        for index, (left, a, a_item) in enumerate(texts):
            for right, b, b_item in texts[index + 1:]:
                if left is right:
                    continue
                dx, dy = min(a.x1, b.x1) - max(a.x0, b.x0), min(a.y1, b.y1) - max(a.y0, b.y0)
                if dx > 1 and dy > 1:
                    overlaps.append({"left": a_item, "right": b_item})
        return {"status": "REQUIRES_ACTUAL_IMAGE_REVIEW",
                "data_frames": [{"panel": chr(97 + i), "bounds": list(ax.get_position().bounds),
                                 "xlim": list(ax.get_xlim()), "ylim": list(ax.get_ylim())}
                                for i, ax in enumerate(axes)],
                "text_outside_canvas": outside, "text_overlap_candidates": overlaps}

    def file_sha256(path: Path) -> str:
        return hashlib.sha256(path.read_bytes()).hexdigest()

    def export_result(fig, axes, directory: Path, stem: str, *, caption: str,
                      sources: dict[str, Path], documented_empty_axes=()) -> dict:
        """Export real results only, with exact input hashes and no inferred visual pass."""
        empty = [ax for ax in axes if not ax.has_data()]
        if set(empty) != set(documented_empty_axes):
            raise ValueError("Empty panels require explicit source-backed missing-status documentation")
        if any(not any(t.get_text() == "No qualified trajectory" for t in ax.texts) for ax in empty):
            raise ValueError("Each documented empty panel must visibly identify its qualification status")
        if not sources or not caption.strip() or not axes:
            raise ValueError("Every panel needs real data, a caption and traced source files before export")
        before = {name: file_sha256(path) for name, path in sources.items()}
        audit = layout_audit(fig, axes)
        typography = []
        for text in fig.findobj(Text):
            if not text.get_visible() or not text.get_text().strip():
                continue
            is_panel_title = any(text is ax.title or text is ax._left_title for ax in axes)
            expected_size = 12 if is_panel_title else 11
            expected_weight = "bold" if is_panel_title else "normal"
            prop = text.get_fontproperties()
            font_path = Path(findfont(prop, fallback_to_default=False))
            if (FontProperties(fname=str(font_path)).get_name() != FONT
                    or abs(text.get_fontsize() - expected_size) > 1e-9
                    or prop.get_weight() != expected_weight or text.get_color() != "black"):
                raise ValueError(f"Unexpected typography: {text.get_text()!r}")
            typography.append({"text": text.get_text(), "font_file": str(font_path),
                               "size_pt": expected_size, "weight": expected_weight, "colour": "black"})
        if audit["text_outside_canvas"]:
            raise ValueError("Figure has text outside its canvas; adjust layout before exporting")
        directory.mkdir(parents=True, exist_ok=True)
        pdf, svg = directory / f"{stem}.pdf", directory / f"{stem}.svg"
        fig.savefig(pdf, metadata={"CreationDate": None, "ModDate": None})
        fig.savefig(svg, metadata={"Date": None})
        outputs = [pdf, svg]
        for suffix, dpi in (("", 600), ("_word", 300)):
            size = tuple(round(value * dpi) for value in fig.get_size_inches())
            png = directory / f"{stem}{suffix}.png"
            fig.savefig(png, dpi=dpi)
            with Image.open(png) as image:
                rgb = image.convert("RGB")
                if rgb.size != size:
                    rgb = rgb.resize(size, Image.Resampling.LANCZOS)
                rgb.save(png)
            outputs.append(png)
        (directory / f"{stem}_caption.txt").write_text(caption.strip() + "\n", encoding="utf-8")
        if before != {name: file_sha256(path) for name, path in sources.items()}:
            raise ValueError("Scientific input changed during figure rendering")
        result = {"figure": stem, "sources": before, "outputs": {p.name: file_sha256(p) for p in outputs},
                  "layout_audit": audit, "typography_audit": {"status": "PASS", "text": typography},
                  "caption_word_count": len(caption.split()), "actual_visual_review": "PENDING"}
        (directory / f"{stem}_qa.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
        return result

    def point_summary(ax, frame, metric, ylabel, *, scale=1.0, controls=CONTROL_ORDER,
                      signed=False, percentage=False):
        """Show every finite scenario plus medians/IQRs, never a selected favourable subset."""
        records = []
        offsets = np.linspace(-0.23, 0.23, len(controls))
        for context_index, context in enumerate(CONTEXT_ORDER):
            cases = frame[frame.context.eq(context)]
            if all(column in cases for column in ("composition_id", "layout")):
                for _, case in cases.groupby(["composition_id", "layout"], sort=True):
                    x_line, y_line = [], []
                    for control, offset in zip(controls, offsets):
                        part = case[case.control.eq(control)]
                        if len(part) == 1 and pd.notna(part.iloc[0][metric]):
                            x_line.append(context_index + offset)
                            y_line.append(float(part.iloc[0][metric]) * scale)
                    if len(y_line) > 1:
                        ax.plot(x_line, y_line, color="#BDB9B2", linewidth=.6, alpha=.65, zorder=1)
            for control, offset in zip(controls, offsets):
                selected = frame[frame.context.eq(context) & frame.control.eq(control)].copy()
                values = pd.to_numeric(selected[metric], errors="raise").to_numpy(dtype=float) * scale
                finite = values[np.isfinite(values)]
                if finite.size:
                    x = context_index + offset
                    jitter = np.linspace(-0.045, 0.045, finite.size) if finite.size > 1 else np.zeros(1)
                    ax.scatter(x + jitter, finite, s=12, color=CONTROL_COLORS[control], alpha=0.78,
                               marker=CONTROL_MARKERS[control], linewidths=.2, edgecolors="white", zorder=2)
                    low, median, high = np.quantile(finite, [0.25, 0.5, 0.75])
                    ax.errorbar(x, median, yerr=[[median - low], [high - median]],
                                fmt=CONTROL_MARKERS[control], color=CONTROL_COLORS[control],
                                markersize=4.5, linewidth=1.5, capsize=2.3, zorder=3)
                else:
                    low = median = high = np.nan
                records.append({"context": context, "control": control, "metric": metric,
                                "declared_cases": len(selected), "finite_cases": len(finite),
                                "q25_display": low, "median_display": median, "q75_display": high})
        if signed:
            ax.axhline(0, color="#A5A5A5", linewidth=0.65, zorder=0)
        if percentage:
            finite_values = pd.to_numeric(frame[metric], errors="raise").to_numpy(float) * scale
            finite_values = finite_values[np.isfinite(finite_values)]
            if len(finite_values):
                span = max(np.ptp(finite_values), 8)
                ax.set_ylim(max(-2, float(finite_values.min()) - span * .12),
                            min(102, float(finite_values.max()) + span * .12))
        if ax.get_position().width * ax.figure.get_figwidth() * 25.4 < 55:
            ax.set_xticks(range(4), list(CONTEXT_LABELS.values()), rotation=75, ha="right")
        else:
            ax.set_xticks(range(4), CONTEXT_SHORT)
        ax.set_xlim(-0.55, 3.55)
        ax.set_ylabel(ylabel)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.ticklabel_format(axis="y", style="plain", useOffset=False)
        return records

    def read_annual_tables(root: Path) -> dict[str, pd.DataFrame]:
        names = ("SPATIAL_PAIRS", "COMMON_LAYOUT_PAIRS", "STATIC_COVERAGE_BOUND", "STATIC_AND_ANNUAL_COVERAGE")
        tables = {name: pd.read_csv(root / f"{name}.csv", keep_default_na=False,
                                    na_values=[""]) for name in names}
        coverage = tables["STATIC_AND_ANNUAL_COVERAGE"]
        if (len(coverage) != 144 or coverage.composition_id.nunique() != 6
                or set(coverage.layout) != {"random_mixing", "demand_ordered"}
                or coverage.duplicated(["context", "composition_id", "layout", "control"]).any()):
            raise ValueError("Expected all six demand compositions, two placements, four contexts and three controls")
        qa = json.loads((root / "JOINT_SPATIAL_QA.json").read_text())
        if (qa["status"] != "PASS" or not qa["overlap_independently_recomputed"]
                or not qa["all_annual_equal_profile_coverages_below_static_bounds"]):
            raise ValueError("The static/annual comparison needs independently verified equal-profile weights")
        return tables

    def write_panel_summaries(directory, number, records):
        pd.DataFrame(records).to_csv(directory / f"Figure_{number:02d}_numerical_summary.csv", index=False)

    def placement_forest(ax, frame, metric, xlabel, *, scale=1.0):
        """All paired placements, separated by context and placement, without a heat table."""
        for context_index, context in enumerate(CONTEXT_ORDER):
            for offset, control in zip((-.22, 0, .22), CONTROL_ORDER):
                selected = frame[frame.context.eq(context) & frame.control.eq(control)]
                values = pd.to_numeric(selected[metric]).dropna().to_numpy() * scale
                if len(values):
                    ax.scatter(values, context_index + offset + np.linspace(-.045, .045, len(values)),
                               s=16, marker=CONTROL_MARKERS[control], color=CONTROL_COLORS[control],
                               alpha=.88, linewidths=.25, edgecolors="white", zorder=3)
                    if len(values) > 1:
                        ax.plot([min(values), max(values)], [context_index + offset] * 2,
                                color=CONTROL_COLORS[control], linewidth=1.1, zorder=2)
        ax.axvline(0, color="#A5A5A5", linewidth=.75, zorder=0)
        labels = CONTEXT_SHORT if ax.get_position().x0 < .2 else list(CONTEXT_LABELS.values())
        ax.set_yticks(range(4), labels)
        ax.set_ylim(3.6, -.6)
        ax.set_xlabel(xlabel)
        ax.xaxis.set_major_locator(MaxNLocator(4))
        ax.set_ylabel("")

    def render_figure_four(tables, annual_root, output):
        spatial = tables["SPATIAL_PAIRS"].query("alternative == 'native_zone'")
        direct = spatial.query("comparison_mode == 'direct'")
        equal = spatial.query("comparison_mode == 'equal_service' and service_fraction == 1.0")
        fig, axes = six_panel_canvas(["Permissions and service", "Permissions and energy", "Common-service energy",
                                     "Placement and service", "Placement and energy", "Compatibility and delivery"],
                                    height=204, bounds=[(19, 148, 67, 36), (109, 148, 67, 36),
                                                        (19, 86, 67, 36), (109, 86, 67, 36),
                                                        (19, 24, 67, 36), (109, 24, 67, 36)])
        summaries = []
        for letter, ax, frame, metric, label in (
            ("a", axes[0], direct, "delta_interval_coverage_percentage_points", "Coverage change\n(percentage points)"),
            ("b", axes[1], direct, "delta_site_pct", "Site energy change (%)"),
            ("c", axes[2], equal, "delta_site_pct", "Site energy change (%)"),
        ):
            summaries += [dict(panel=letter, **r) for r in point_summary(ax, frame, metric, label, signed=True)]
        common_layout_path = annual_root / "COMMON_LAYOUT_PAIRS.csv"
        common_layout = tables["COMMON_LAYOUT_PAIRS"]
        layouts = common_layout.query("permission == 'native_zone' and service_fraction == 1 "
                                      "and weight_view == 'profile_equal'").copy()
        layouts.to_csv(output / "Figure_04_common_absolute_anchor_layout_source.csv", index=False)
        for letter, ax, metric, ylabel in (
            ("d", axes[3], "delta_modeled_interval_coverage", "Coverage change\n(percentage points)"),
            ("e", axes[4], "delta_site_pct", "Energy change (%)"),
        ):
            matrix, cell_counts = [], []
            for context in CONTEXT_ORDER:
                row, count_row = [], []
                for layout in ("demand_ordered",):
                    for control in CONTROL_ORDER:
                        group = layouts[layouts.context.eq(context) & layouts.layout.eq(layout)
                                        & layouts.control.eq(control)]
                        values = pd.to_numeric(group[metric]).dropna().to_numpy()
                        if letter == "d":
                            values = values * 100
                        value = float(np.median(values)) if len(values) else np.nan
                        row.append(value)
                        count_row.append(len(values))
                        summaries.append({"panel": letter, "context": context, "control": control,
                                          "layout": layout, "metric": metric, "finite_cases": len(values),
                                          "median_display": value})
                matrix.append(row)
                cell_counts.append(count_row)
            placement_forest(ax, layouts, metric, ylabel, scale=100 if letter == "d" else 1)
        realization = tables["STATIC_AND_ANNUAL_COVERAGE"]
        static = tables["STATIC_COVERAGE_BOUND"]
        realization.to_csv(output / "Figure_04_static_and_annual_coverage_source.csv", index=False)
        for context_index, context in enumerate(CONTEXT_ORDER):
            for layout_index, layout in enumerate(("random_mixing", "demand_ordered")):
                row_y = context_index + (layout_index - .5) * .40
                selected = realization[realization.context.eq(context) & realization.layout.eq(layout)]
                for control_index, control in enumerate(CONTROL_ORDER):
                    group = selected[selected.control.eq(control) & selected.selection_status.eq("selected")].sort_values("composition_id")
                    for case_index, row in enumerate(group.itertuples(index=False)):
                        y = row_y + (control_index - 1) * .08 + (case_index - (len(group) - 1) / 2) * .010
                        annual_value = 100 * row.modeled_interval_coverage
                        bound = 100 * row.static_equal_profile_coverage_bound
                        axes[5].plot([annual_value, bound], [y, y], color=CONTEXT_COLORS[context],
                                     linewidth=.45, alpha=.32, zorder=1)
                        axes[5].scatter(annual_value, y, s=13, marker=CONTROL_MARKERS[control],
                                        edgecolors=CONTROL_COLORS[control], linewidths=.75,
                                        facecolors="white" if layout_index == 0 else CONTROL_COLORS[control], zorder=4)
                    summaries.append({"panel": "f", "context": context, "control": control, "layout": layout,
                                      "metric": "annual_equal_profile_interval_coverage_pct", "finite_cases": len(group),
                                      "median_display": float(100 * group.modeled_interval_coverage.median())})
                bounds = static[static.context.eq(context) & static.layout.eq(layout)].sort_values("composition_id")
                for case_index, row in enumerate(bounds.itertuples(index=False)):
                    axes[5].plot([100 * row.static_equal_profile_coverage_bound] * 2,
                                 [row_y - .15 + case_index * .01, row_y + .10 + case_index * .01],
                                 color=CONTEXT_COLORS[context], linewidth=.75, alpha=.85, zorder=2)
        axes[5].set_yticks(range(4), list(CONTEXT_LABELS.values()))
        axes[5].set_ylim(3.5, -.6)
        axes[5].set_xlim(48, 102)
        axes[5].set_xlabel("Range coverage (%)")
        axes[5].set_xticks([50, 65, 80, 95])
        write_panel_summaries(output, 4, summaries)
        return fig, axes

    DPI = 600
    annual_root = RESULTS / "Joint_main/joint_spatial"
    output = INTERMEDIATE / LABEL
    output.mkdir(parents=True, exist_ok=True)
    tables = read_annual_tables(annual_root)
    fig, axes = render_figure_four(tables, annual_root, output)

    (RESULTS / "captions" / (LABEL + ".txt")).write_text('Figure 4 | Placement and control permissions across six demand compositions. a,b, Direct changes in range coverage and purchased energy when separate zone commands replace one building command. Small points are matched cases; larger symbols and whiskers show medians and interquartile ranges. c, The same permission contrast under common annual mean and fixed-tail mismatch limits. d,e, Coverage and energy changes from grouping similar demands, relative to random placement. Both use the same random-placement absolute limits and 27 tail identities, native-zone commands and equal-profile service weights. f, Annual equal-profile coverage (control symbols) joined to its composition’s static overlap limit (context-coloured ticks). Open/filled symbols denote random/ordered placement. The limit sums the maximum simultaneously covered profiles in each zone and divides by 267. It assumes freely chosen temperatures without equipment constraints. Matched support is 88/144 in a,b, 144/144 in c, 54/72 in d,e and 126/144 annual outcomes in f. Missing cases remain unavailable. Context names identify representative building–weather models. Personal range uses model-generated individual demand intervals.',encoding="utf-8")
    for index, ax in enumerate(axes):
        ax.yaxis.set_label_coords(-.14, .5)
        ax.xaxis.set_label_coords(.5, -.30)
    save_plot(fig, LABEL, dpi=DPI)
    (INTERMEDIATE / LABEL / "Figure_04_layout_audit.json").write_text(json.dumps(layout_audit(fig, axes), indent=2) + "\n")


draw_figure_04()


In [ ]:
# Figure_05: edit this cell and rerun to update this figure.
def draw_figure_05():
    import matplotlib.pyplot as plt
    from matplotlib.font_manager import FontProperties, findfont
    from matplotlib.lines import Line2D
    from matplotlib.ticker import MaxNLocator
    import numpy as np
    import pandas as pd

    label = 'Figure_05'
    controls = ('uniform_average', 'personal_center', 'ltpref_interval')
    control_labels = ('Uniform average', 'Personal centre', 'Personal range')
    colors = dict(zip(controls, ('#7F7F7F', '#C8A36A', '#7FA995')))
    markers = dict(zip(controls, ('o', 's', '^')))
    contexts = ('Miami', 'NewDelhi', 'PortAngeles_small2019', 'Seattle_small2019')
    context_labels = ('Miami', 'New\nDelhi', 'Port\nAngeles', 'Seattle')
    context_colors = ('#C78470', '#AF94AD', '#6F9EBC', '#7FA995')
    for weight in ('normal', 'bold'):
        findfont(FontProperties(family='Times New Roman', weight=weight), fallback_to_default=False)
    plt.rcdefaults()
    plt.rcParams.update({
        'font.family': 'Times New Roman', 'font.size': 11, 'text.color': 'black',
        'axes.labelsize': 11, 'axes.labelcolor': 'black', 'axes.labelpad': 6,
        'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlecolor': 'black',
        'axes.spines.top': False, 'axes.spines.right': False,
        'axes.edgecolor': '#777777', 'axes.linewidth': .75,
        'xtick.labelsize': 11, 'ytick.labelsize': 11,
        'xtick.major.pad': 4, 'ytick.major.pad': 4,
        'xtick.color': 'black', 'ytick.color': 'black',
        'legend.fontsize': 11, 'legend.frameon': False,
        'figure.facecolor': 'white', 'savefig.facecolor': 'white',
        'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    })
    evidence = RESULTS / 'Joint_main'
    event_root, portfolio_root = evidence / 'joint_events', evidence / 'joint_portfolio'
    cases = pd.read_csv(event_root / 'JOINT_EVENT_CASE_SERVICE_ENERGY.csv')
    stages = pd.read_csv(event_root / 'JOINT_EVENT_STAGE_COUNTS_BY_CONTROL.csv')
    responses = pd.read_csv(event_root / 'JOINT_EVENT_PORTFOLIO_RESPONSE.csv')
    hourly = pd.read_csv(portfolio_root / 'portfolio_hourly.csv')
    chosen = cases[cases.event_month.eq(7) & cases.service_fraction.eq(1)
                   & cases.selection_status.eq('selected')].copy()
    if len(chosen) != 64 or set(chosen.layout) != {'random_mixing', 'demand_ordered'}:
        raise ValueError('The joint six-composition July response scope changed')
    selected_counts = chosen.control.value_counts().to_dict()
    if selected_counts != {'uniform_average': 24, 'personal_center': 20, 'ltpref_interval': 20}:
        raise ValueError('Control-specific response qualification changed')
    full = responses[responses.event_id.eq('month_07')
                     & responses.dr_participation_fraction.eq(1)].copy()
    joint = full[full.result_scope.eq('joint_144_offices')]
    composition = full[full.result_scope.eq('one_composition_24_offices')]
    if not joint.all_baselines_service_qualified.eq(True).all():
        raise ValueError('July portfolio response needs complete baseline service qualification')
    trace = hourly[hourly.event_id.eq('month_07') & hourly.scope.eq('native_zone')
                   & hourly.dr_participation_fraction.eq(1)].copy()
    if set(joint.portfolio_buildings) != {144}:
        raise ValueError('The joint portfolio scope must remain 144 offices')

    width_mm, height_mm = 180, 207
    fig = plt.figure(figsize=(width_mm / 25.4, height_mm / 25.4))
    bounds = [(19, 155, 67, 28), (19, 105, 67, 27), (109, 105, 67, 27),
              (19, 56, 67, 27), (109, 56, 67, 27), (19, 14, 157, 22)]
    titles = ('Response and recovery', 'Electricity after recovery', 'Zone-level portfolio',
              'Building-wide portfolio', 'Synchronized rebound', 'Response options through service checks')
    axes = []
    for index, ((x, y, w, h), title) in enumerate(zip(bounds, titles)):
        ax = fig.add_axes([x / width_mm, y / height_mm, w / width_mm, h / height_mm])
        ax.set_title(f'({chr(97 + index)}) {title}', loc='left', pad=8)
        ax.grid(False)
        ax.set_axisbelow(True)
        axes.append(ax)
    handles = [Line2D([], [], color=colors[c], marker=markers[c], linestyle='none',
                      markersize=5, label=name) for c, name in zip(controls, control_labels)]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(.5, .995),
               ncol=3, handletextpad=.3, columnspacing=.9)
    fig.text(.5, 195 / height_mm, 'Port Angeles: solid / Seattle: dashed',
             ha='center', va='center', fontsize=11)
    late = fig.add_axes([109 / width_mm, 155 / height_mm, 67 / width_mm, 28 / height_mm])
    late.set_title('Later recovery', loc='left', pad=8, fontsize=11, fontweight='normal')
    for control in controls:
        for context, style in (('PortAngeles_small2019', '-'), ('Seattle_small2019', '--')):
            part = trace[trace.control.eq(control) & trace.climate_context.eq(context)].sort_values('hour_index')
            time = part.hour_index + .5
            for ax, mask in ((axes[0], time.le(6)), (late, time.gt(6))):
                ax.plot(time[mask], part.electric_change_kwh[mask], color=colors[control],
                        linestyle=style, linewidth=1.4)
    axes[0].axvspan(0, 2, color='#C8A36A', alpha=.2, zorder=0)
    axes[0].set_xlim(0, 6)
    late.set_xlim(6, 50)
    axes[0].set_xticks([0, 2, 4, 6])
    late.set_xticks([6, 24, 48])
    for ax in (axes[0], late):
        ax.set_ylabel('Power change (kW)')
        ax.set_xlabel('Hours after event start', labelpad=3)
        ax.axhline(0, color='#A5A5A5', linewidth=.65, zorder=0)
        ax.yaxis.set_major_locator(MaxNLocator(4))

    ax = axes[1]
    metrics = ('event_electric_saving_kwh', 'recovery_extra_electric_kwh',
               'event_plus48h_net_electric_saving_kwh')
    for offset, control in zip((-.16, 0, .16), controls):
        group = chosen[chosen.control.eq(control)]
        medians = []
        for index, metric in enumerate(metrics):
            values = group[metric].to_numpy(float)
            jitter = np.linspace(-.035, .035, len(values))
            ax.scatter(index + offset + jitter, values, s=9, alpha=.4,
                       marker=markers[control], color=colors[control], linewidths=0, zorder=2)
            low, middle, high = np.quantile(values, [.25, .5, .75])
            medians.append(middle)
            ax.errorbar(index + offset, middle, yerr=[[middle - low], [high - middle]],
                        color=colors[control], marker=markers[control], markersize=4,
                        linewidth=1.1, capsize=2, zorder=4)
        ax.plot(np.arange(3) + offset, medians, color=colors[control], linewidth=.8, alpha=.8)
    ax.set_xticks([0, 1, 2], ['Event\nsaving', 'Recovery\nuse', 'Net\nsaving'])
    ax.set_xlim(-.45, 2.45)
    ax.set_ylabel('Electricity (kWh)')
    ax.yaxis.set_major_locator(MaxNLocator(4))
    ax.axhline(0, color='#A5A5A5', linewidth=.65, zorder=0)

    sources = {}
    def portfolio_panel(ax, scope, metric, xlabel):
        data = joint[joint.scope.eq(scope)]
        parts = composition[composition.scope.eq(scope)]
        for ci, (context, background) in enumerate(zip(contexts, context_colors)):
            ax.axhspan(ci - .46, ci + .46, color=background, alpha=.12, linewidth=0, zorder=0)
            for offset, control in zip((-.23, 0, .23), controls):
                row = data[data.climate_context.eq(context) & data.control.eq(control)].iloc[0]
                individual = parts[parts.climate_context.eq(context) & parts.control.eq(control)].sort_values('composition_id')
                if len(individual) != 6:
                    raise ValueError('A portfolio display must retain all six compositions')
                y = ci + offset
                ax.scatter(individual[metric], y + np.linspace(-.055, .055, 6), s=10,
                           marker=markers[control], facecolors='none', edgecolors=colors[control],
                           linewidths=.65, alpha=.7, zorder=2)
                value = float(row[metric])
                ax.plot([0, value], [y, y], color=colors[control], linewidth=1.5, alpha=.85, zorder=2)
                ax.scatter(value, y, color=colors[control], marker=markers[control], s=25,
                           edgecolors='white', linewidths=.3, zorder=3)
        ax.axvline(0, color='#A5A5A5', linewidth=.65, zorder=1)
        ax.set_yticks(range(4), context_labels)
        for tick in ax.get_yticklabels():
            tick.set_linespacing(.75)
        ax.set_ylim(3.55, -.55)
        ax.set_xlabel(xlabel)
        ax.xaxis.set_major_locator(MaxNLocator(4))
        lower, upper = ax.get_xlim()
        ticks = ax.get_xticks()
        ax.set_xticks(ticks[(ticks >= lower) & (ticks <= upper)])
        return pd.concat([data, parts])
    sources['c'] = portfolio_panel(axes[2], 'native_zone', 'event_average_reduction_kw_per_1000m2',
                                   'Reduction (kW per 1,000 m²)')
    sources['d'] = portfolio_panel(axes[3], 'building', 'event_average_reduction_kw_per_1000m2',
                                   'Reduction (kW per 1,000 m²)')
    sources['e'] = portfolio_panel(axes[4], 'native_zone', 'recovery_synchronous_peak_extra_kw_per_1000m2',
                                   'Recovery peak (kW per 1,000 m²)')

    counts = stages[stages.event_month.eq(7) & stages.service_fraction.eq(1)].copy()
    stage_order = ('raw_positive', 'baseline_qualified', 'event_qualified', 'recovery_qualified')
    for offset, control in zip((-.09, 0, .09), controls):
        group = counts[counts.control.eq(control)].set_index('stage').loc[list(stage_order)]
        x = np.arange(4) + offset
        y = group.decisions_with_positive_selection.to_numpy(int)
        axes[5].plot(x, y, color=colors[control], linewidth=1.25,
                     marker=markers[control], markersize=4)
    for index, total in enumerate((360, 360, 64, 64)):
        axes[5].text(index, 132, f'{total} total', ha='center', va='center', fontsize=11)
    axes[5].set_xticks(range(4), ['Positive power', 'Own baseline', 'Event service', 'Recovery service'])
    axes[5].set_xlim(-.35, 3.35)
    axes[5].set_ylim(0, 148)
    axes[5].set_yticks([0, 60, 120])
    axes[5].set_ylabel('Choices retained')
    sources.update(a=trace, b=chosen, f=counts)
    source_root = INTERMEDIATE / label
    source_root.mkdir(parents=True, exist_ok=True)
    for panel, frame in sources.items():
        frame.to_csv(source_root / f'{label}_panel_{panel}_source.csv', index=False)
    caption = 'Figure 5 | July response and recovery. Each control starts from its own selected annual baseline. The largest positive response among +0.5/+1°C two-hour shifts must meet common mean and fixed-tail mismatch limits for baseline and perturbed operation during the event and 48-hour recovery. a, Hourly electricity change for 144-office zone-control portfolios. Adjacent axes use different scales; shading marks the event. Solid/dashed lines denote Port Angeles/Seattle; Miami and Delhi have zero qualified change. b, Event savings, recovery use and net savings for 64 positive selections across distinct permissions. Small points show cases; larger symbols and whiskers show medians and interquartile ranges. c,d, Qualified zone/building-command response. e, Zone-command synchronous recovery peak. In c–e, filled symbols show the joint 144-office portfolio; open symbols show each composition’s 24-office portfolio. Both use twelve copies per placement and their respective area denominators. f, Sequential qualification across 120 decisions per control; annotations sum all three controls. Single-floor permission aliases are counted once. The fixed tail tracks 27 profiles worst served by common 25°C operation. Personal range uses model-generated demand intervals.'
    (RESULTS / 'captions' / f'{label}.txt').write_text(caption, encoding='utf-8')
    save_plot(fig, label, dpi=600)


draw_figure_05()


In [ ]:
# Figure_06: edit this cell and rerun to update this figure.
def draw_figure_06():
    LABEL = "Figure_06"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import argparse

    import hashlib

    import json

    from pathlib import Path

    import matplotlib

    import matplotlib.pyplot as plt

    from matplotlib.font_manager import FontProperties, findfont

    from matplotlib.lines import Line2D

    from matplotlib.text import Text

    from PIL import Image

    import numpy as np

    import pandas as pd

    from matplotlib.colors import LinearSegmentedColormap

    from matplotlib.ticker import MaxNLocator

    FONT = "Times New Roman"

    WIDTH_MM, HEIGHT_MM = 180, 183

    CONTROL_ORDER = ("uniform_average", "personal_center", "ltpref_interval")

    CONTROL_LABELS = {
        "uniform_average": "Uniform average",
        "personal_center": "Personal centre",
        "ltpref_interval": "Personal range",
    }

    CONTROL_COLORS = {
        "uniform_average": "#7F7F7F",
        "personal_center": "#C8A36A",
        "ltpref_interval": "#7FA995",
    }

    CONTEXT_ORDER = ("Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019")

    CONTEXT_COLORS = dict(zip(CONTEXT_ORDER, ("#C78470", "#AF94AD", "#6F9EBC", "#7FA995")))

    CONTEXT_LABELS = dict(zip(CONTEXT_ORDER, ("Miami", "New Delhi", "Port Angeles", "Seattle")))

    CONTEXT_SHORT = ("Miami", "New\nDelhi", "Port\nAngeles", "Seattle")

    CONTROL_MARKERS = dict(zip(CONTROL_ORDER, ("o", "s", "^")))

    ALTERNATIVES = CONTROL_ORDER[1:]

    DATA_BOUNDS_MM = tuple(
        (x, y, 67.0, 35.0) for y in (125.0, 72.0, 19.0) for x in (19.0, 109.0)
    )

    def configure_style() -> dict[str, str]:
        fonts = {}
        for weight in ("normal", "bold"):
            prop = FontProperties(family=FONT, weight=weight)
            path = Path(findfont(prop, fallback_to_default=False))
            if FontProperties(fname=str(path)).get_name() != FONT:
                raise ValueError(f"Exact {FONT} {weight} is required")
            fonts[weight] = str(path)
        plt.rcParams.update({
            "font.family": FONT, "font.size": 11, "text.color": "black",
            "axes.labelsize": 11, "axes.labelcolor": "black", "axes.labelpad": 6,
            "axes.titlesize": 12, "axes.titleweight": "bold",
            "axes.spines.top": False, "axes.spines.right": False,
            "axes.edgecolor": "#777777", "axes.linewidth": 0.75,
            "xtick.labelsize": 11, "ytick.labelsize": 11,
            "xtick.major.pad": 4, "ytick.major.pad": 4,
            "xtick.color": "black", "ytick.color": "black",
            "legend.fontsize": 11, "legend.frameon": False,
            "figure.facecolor": "white", "savefig.facecolor": "white",
            "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
            "svg.hashsalt": "thermal-heterogeneity-stage8",
        })
        return fonts

    def six_panel_canvas(titles: list[str], *, control_key: bool = True, bounds=None, height=None):
        """Create equal data frames with separate title, axis and shared-key bands."""
        if len(titles) != 6:
            raise ValueError("The main result template has exactly six data panels")
        configure_style()
        page_height = height or HEIGHT_MM
        fig = plt.figure(figsize=(WIDTH_MM / 25.4, page_height / 25.4))
        axes = []
        for index, (title, panel_bounds) in enumerate(zip(titles, bounds or DATA_BOUNDS_MM)):
            x, y, width, height_mm = panel_bounds
            ax = fig.add_axes((x / WIDTH_MM, y / page_height, width / WIDTH_MM, height_mm / page_height))
            ax.set_title(f"({chr(97 + index)}) {title}", loc="left", pad=8)
            ax.grid(False)
            ax.set_axisbelow(True)
            axes.append(ax)
        if control_key:
            handles = [Line2D([], [], marker=marker, linestyle="none", color=CONTROL_COLORS[key],
                              markersize=5, label=CONTROL_LABELS[key])
                       for key, marker in zip(CONTROL_ORDER, ("o", "s", "^"))]
            fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.995),
                       ncol=3, handletextpad=0.3, columnspacing=0.9)
        return fig, axes

    def layout_audit(fig, axes) -> dict:
        """Record geometric candidates. Actual image inspection is a separate gate."""
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        outside, texts = [], []
        hidden_ticks = set()
        for ax in axes:
            for ticker, limits in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
                low, high = sorted(limits)
                for tick in ticker.get_major_ticks() + ticker.get_minor_ticks():
                    if tick.get_loc() < low - 1e-9 or tick.get_loc() > high + 1e-9:
                        hidden_ticks.update((id(tick.label1), id(tick.label2)))
        for text in fig.findobj(Text):
            if not text.get_visible() or not text.get_text().strip() or id(text) in hidden_ticks:
                continue
            box = text.get_window_extent(renderer)
            item = {"text": text.get_text(), "bounds_px": [round(v, 2) for v in box.bounds]}
            texts.append((text, box, item))
            if box.x0 < -0.5 or box.y0 < -0.5 or box.x1 > fig.bbox.width + 0.5 or box.y1 > fig.bbox.height + 0.5:
                outside.append(item)
        overlaps = []
        for index, (left, a, a_item) in enumerate(texts):
            for right, b, b_item in texts[index + 1:]:
                if left is right:
                    continue
                dx, dy = min(a.x1, b.x1) - max(a.x0, b.x0), min(a.y1, b.y1) - max(a.y0, b.y0)
                if dx > 1 and dy > 1:
                    overlaps.append({"left": a_item, "right": b_item})
        return {"status": "REQUIRES_ACTUAL_IMAGE_REVIEW",
                "data_frames": [{"panel": chr(97 + i), "bounds": list(ax.get_position().bounds),
                                 "xlim": list(ax.get_xlim()), "ylim": list(ax.get_ylim())}
                                for i, ax in enumerate(axes)],
                "text_outside_canvas": outside, "text_overlap_candidates": overlaps}

    def point_summary(ax, frame, metric, ylabel, *, scale=1.0, controls=CONTROL_ORDER,
                      signed=False, percentage=False):
        """Show every finite scenario plus medians/IQRs, never a selected favourable subset."""
        records = []
        offsets = np.linspace(-0.23, 0.23, len(controls))
        for context_index, context in enumerate(CONTEXT_ORDER):
            cases = frame[frame.context.eq(context)]
            if all(column in cases for column in ("composition_id", "layout")):
                for _, case in cases.groupby(["composition_id", "layout"], sort=True):
                    x_line, y_line = [], []
                    for control, offset in zip(controls, offsets):
                        part = case[case.control.eq(control)]
                        if len(part) == 1 and pd.notna(part.iloc[0][metric]):
                            x_line.append(context_index + offset)
                            y_line.append(float(part.iloc[0][metric]) * scale)
                    if len(y_line) > 1:
                        ax.plot(x_line, y_line, color="#BDB9B2", linewidth=.6, alpha=.65, zorder=1)
            for control, offset in zip(controls, offsets):
                selected = frame[frame.context.eq(context) & frame.control.eq(control)].copy()
                values = pd.to_numeric(selected[metric], errors="raise").to_numpy(dtype=float) * scale
                finite = values[np.isfinite(values)]
                if finite.size:
                    x = context_index + offset
                    jitter = np.linspace(-0.045, 0.045, finite.size) if finite.size > 1 else np.zeros(1)
                    ax.scatter(x + jitter, finite, s=12, color=CONTROL_COLORS[control], alpha=0.78,
                               marker=CONTROL_MARKERS[control], linewidths=.2, edgecolors="white", zorder=2)
                    low, median, high = np.quantile(finite, [0.25, 0.5, 0.75])
                    ax.errorbar(x, median, yerr=[[median - low], [high - median]],
                                fmt=CONTROL_MARKERS[control], color=CONTROL_COLORS[control],
                                markersize=4.5, linewidth=1.5, capsize=2.3, zorder=3)
                else:
                    low = median = high = np.nan
                records.append({"context": context, "control": control, "metric": metric,
                                "declared_cases": len(selected), "finite_cases": len(finite),
                                "q25_display": low, "median_display": median, "q75_display": high})
        if signed:
            ax.axhline(0, color="#A5A5A5", linewidth=0.65, zorder=0)
        if percentage:
            finite_values = pd.to_numeric(frame[metric], errors="raise").to_numpy(float) * scale
            finite_values = finite_values[np.isfinite(finite_values)]
            if len(finite_values):
                span = max(np.ptp(finite_values), 8)
                ax.set_ylim(max(-2, float(finite_values.min()) - span * .12),
                            min(102, float(finite_values.max()) + span * .12))
        if ax.get_position().width * ax.figure.get_figwidth() * 25.4 < 55:
            ax.set_xticks(range(4), list(CONTEXT_LABELS.values()), rotation=75, ha="right")
        else:
            ax.set_xticks(range(4), CONTEXT_SHORT)
        ax.set_xlim(-0.55, 3.55)
        ax.set_ylabel(ylabel)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.ticklabel_format(axis="y", style="plain", useOffset=False)
        return records

    def annotated_heatmap(ax, values, xlabels, ylabel, *, signed, decimals=1, counts=None):
        """An exact-value matrix uses no extra inset that would shrink the data frame."""
        values = np.asarray(values, dtype=float)
        maximum = max(float(np.nanmax(np.abs(values))), 1e-9) if np.isfinite(values).any() else 1.0
        if signed:
            cmap = LinearSegmentedColormap.from_list("stage8_signed", ["#6F9EBC", "#F7F6F3", "#C78470"])
            limits = (-maximum, maximum)
        else:
            cmap = LinearSegmentedColormap.from_list("stage8_positive", ["#F7F6F3", "#7FA995"])
            limits = (0.0, max(100.0, maximum))
        cmap.set_bad("#EEEEEE")
        ax.imshow(np.ma.masked_invalid(values), aspect="auto", interpolation="nearest",
                  cmap=cmap, vmin=limits[0], vmax=limits[1])
        for row in range(values.shape[0]):
            for col in range(values.shape[1]):
                value = values[row, col]
                rounded = round(value, decimals) if np.isfinite(value) else value
                label = f"{0.0 if rounded == 0 else rounded:.{decimals}f}" if np.isfinite(value) else "NA"
                if counts is not None:
                    label += f"\n({int(counts[row][col])}/3)"
                ax.text(col, row, label, ha="center", va="center", color="black", fontsize=11)
        ax.set_xticks(range(len(xlabels)), xlabels)
        ax.set_yticks(range(4), list(CONTEXT_LABELS.values()))
        ax.set_ylabel(ylabel)
        ax.tick_params(length=0)
        return {"values": values.tolist(), "colour_limits": list(limits)}

    def placement_forest(ax, frame, metric, xlabel, *, scale=1.0):
        """All paired placements, separated by context and placement, without a heat table."""
        for context_index, context in enumerate(CONTEXT_ORDER):
            for layout_index, layout in enumerate(("demand_ordered", "source_structured")):
                center = context_index * 2.5 + layout_index
                for offset, control in zip((-.23, 0, .23), CONTROL_ORDER):
                    selected = frame[frame.context.eq(context) & frame.layout.eq(layout) & frame.control.eq(control)]
                    values = pd.to_numeric(selected[metric]).dropna().to_numpy() * scale
                    if len(values):
                        ax.scatter(values, center + offset + np.linspace(-.045, .045, len(values)),
                                   s=15, marker=CONTROL_MARKERS[control], color=CONTROL_COLORS[control],
                                   alpha=.85, linewidths=.2, edgecolors="white", zorder=3)
                        if len(values) > 1:
                            ax.plot([min(values), max(values)], [center + offset] * 2,
                                    color=CONTROL_COLORS[control], linewidth=1.1, zorder=2)
        ax.axvline(0, color="#A5A5A5", linewidth=.75, zorder=0)
        ax.set_yticks([i * 2.5 + .5 for i in range(4)], list(CONTEXT_LABELS.values()))
        ax.set_ylim(9.1, -.7)
        ax.set_xlabel(xlabel)
        ax.xaxis.set_major_locator(MaxNLocator(4))
        ax.set_ylabel("")
    import argparse

    import json

    from pathlib import Path

    import matplotlib.pyplot as plt

    import numpy as np

    import pandas as pd

    from matplotlib.lines import Line2D

    from matplotlib.ticker import MaxNLocator

    def read_table(path):
        return pd.read_csv(path, keep_default_na=False, na_values=[""], low_memory=False)

    def dump_source(output, number, panel, frame):
        frame.to_csv(output / f"Figure_{number:02d}_panel_{panel}_source.csv", index=False)

    def zero_axes(ax, *, x=False, y=True):
        if x:
            ax.axvline(0, color="#B0B0B0", linewidth=0.65, zorder=0)
        if y:
            ax.axhline(0, color="#B0B0B0", linewidth=0.65, zorder=0)
        ax.xaxis.set_major_locator(MaxNLocator(4))
        ax.yaxis.set_major_locator(MaxNLocator(4))
        ax.ticklabel_format(axis="both", style="plain", useOffset=False)

    def nonalias(frame):
        return frame[~(frame.permission.eq("floor") & frame.context.str.contains("small2019"))].copy()


    DPI=600
    output=INTERMEDIATE/LABEL
    output.mkdir(parents=True,exist_ok=True)
    root=RESULTS/"Joint_main/costs"
    costs=read_table(root/"all_signed_paired_costs.csv")
    bills=read_table(root/"bill_implementation_boundaries.csv")
    groups=read_table(root/"command_scope_management_boundaries.csv")
    packages=read_table(root/"implementation_cost_scenarios.csv")
    base=nonalias(costs[costs.scenario_id.eq("information_reuse__median")])
    direct=base.query("comparison_mode == 'direct' and permission == 'native_zone'")
    common=base.query("comparison_mode == 'equal_service' and service_fraction == 1")
    if len(common)!=168:
        raise ValueError("Expected 168 joint US distinct-permission common-requirement pairs")
    fig,axes=six_panel_canvas(["Direct service and bill","Common annual requirements","Implementation resources",
        "Delhi electricity and gas","Management-time allowance","Value needed to break even"],control_key=False,height=193,
        bounds=[(19,136,67,34),(109,136,67,34),(19,78,67,34),(109,78,67,34),(19,20,67,31),(109,20,67,31)])
    fig.legend(handles=[Line2D([],[],marker=CONTROL_MARKERS[c],linestyle="none",color=CONTROL_COLORS[c],
        markersize=5,label=CONTROL_LABELS[c]) for c in ALTERNATIVES],ncol=2,loc="upper center",bbox_to_anchor=(.5,.996))
    for c in ALTERNATIVES:
        g=direct[direct.control.eq(c)]
        axes[0].scatter(g.delta_interval_coverage_percentage_points,g.delta_energy_charge_usd_year,
            color=CONTROL_COLORS[c],marker=CONTROL_MARKERS[c],s=22,edgecolors="white",linewidths=.35,alpha=.85)
    axes[0].set_xlabel("Coverage change\n(percentage points)")
    axes[0].set_ylabel("Bill change (US$/year)")
    zero_axes(axes[0],x=True)
    for index,context in enumerate(("Miami","PortAngeles_small2019","Seattle_small2019")):
        for offset,c in zip((-.16,.16),ALTERNATIVES):
            g=common[common.context.eq(context)&common.control.eq(c)]
            values=g.delta_energy_charge_usd_year.to_numpy(float)
            axes[1].scatter(index+offset+np.linspace(-.04,.04,len(values)),values,s=15,marker=CONTROL_MARKERS[c],
                color=CONTROL_COLORS[c],alpha=.75,edgecolors="white",linewidths=.25)
            q25,med,q75=np.quantile(values,[.25,.5,.75])
            axes[1].errorbar(index+offset,med,yerr=[[med-q25],[q75-med]],fmt=CONTROL_MARKERS[c],
                color=CONTROL_COLORS[c],markersize=4,capsize=2,linewidth=1.2)
    axes[1].set_xticks(range(3),["Miami","Port\nAngeles","Seattle"])
    axes[1].set_ylabel("Bill change (US$/year)")
    axes[1].axhline(0,color="#B0B0B0",linewidth=.65)
    axes[1].yaxis.set_major_locator(MaxNLocator(4))
    names=["existing_bms_light__median","existing_bms_feedback__median","supported_feedback__median","optional_interface__median"]
    labels=["Existing","Feedback","Supported","Interface"]
    selected=packages.set_index("scenario_id").loc[names].copy()
    wage=float(read_table(root/"command_scope_management_boundaries.csv").query("scenario_id == 'information_reuse__median'").manager_wage_usd_hour.dropna().iloc[0])
    # Input packages specify manager and respondent time, using the released
    # operator/resource difference for the respondent valuation.
    elements={"Setup":selected.setup_hours*selected.annuity_factor*wage,
        "Management":selected.annual_maintenance_hours*wage,
        "Feedback time":selected.annual_resource_cost_usd-selected.annual_operator_cost_usd,
        "Software":selected.annual_software_usd,
        "Interface":selected.added_interface_capital_usd*selected.annuity_factor}
    colors=["#6F9EBC","#7FA995","#C8A36A","#AF94AD","#C78470"]
    totals=np.zeros(4)
    for (label,values),color in zip(elements.items(),colors):
        axes[2].barh(np.arange(4),values.to_numpy(),left=totals,height=.60,color=color,edgecolor="white",linewidth=.5,label=label)
        totals+=values.to_numpy()
    if not np.allclose(totals,selected.annual_resource_cost_usd,atol=1e-7):
        raise ValueError("Implementation component ledger does not close")
    axes[2].set_yticks(range(4),labels)
    axes[2].invert_yaxis()
    axes[2].set_xlabel("Annualized resources (US$/year)")
    axes[2].xaxis.set_major_locator(MaxNLocator(3))
    axes[2].legend(ncol=2,loc="upper right",bbox_to_anchor=(1,1.0),columnspacing=.7,
        handletextpad=.2,handlelength=.8,labelspacing=.1,borderpad=0)
    delhi=bills.query("context == 'NewDelhi' and comparison_mode == 'direct' and permission == 'native_zone' and status == 'matched'")
    for c in ALTERNATIVES:
        g=delhi[delhi.control.eq(c)]
        axes[3].scatter(g.delta_bill_intercept_INR_year_pf095/1000,g.delta_bill_gas_slope_kwh_hhv,
            s=40,color=CONTROL_COLORS[c],marker=CONTROL_MARKERS[c],edgecolors="white",linewidths=.5)
    axes[3].set_xlabel("Electricity charge change\n(10³ INR/year)")
    axes[3].set_ylabel("Gas change (kWh/year)")
    zero_axes(axes[3],x=True)
    d=groups.query("scenario_id == 'information_reuse__median' and comparison_mode == 'equal_service' and service_fraction == 1 and extra_groups_over_uniform > 0 and climate_context != 'NewDelhi'").copy()
    d=d[~(d.scope.eq("floor")&d.climate_context.str.contains("small2019"))]
    d["minutes_per_group_year"]=60*d.signed_resource_manager_hour_allowance_per_extra_group_year
    for index,count in enumerate((2,4,14)):
        for off,c in zip((-.16,.16),ALTERNATIVES):
            values=d[d.control.eq(c)&d.extra_groups_over_uniform.eq(count)].minutes_per_group_year.to_numpy(float)
            axes[4].scatter(index+off+np.linspace(-.04,.04,len(values)),values,color=CONTROL_COLORS[c],marker=CONTROL_MARKERS[c],
                s=15,alpha=.8,edgecolors="white",linewidths=.25)
            q25,med,q75=np.quantile(values,[.25,.5,.75])
            axes[4].errorbar(index+off,med,yerr=[[med-q25],[q75-med]],fmt=CONTROL_MARKERS[c],color=CONTROL_COLORS[c],capsize=2,markersize=4)
    axes[4].set_xticks(range(3),["2","4","14"])
    axes[4].set_xlabel("Additional control groups")
    axes[4].set_ylabel("Allowance (min/group/year)")
    axes[4].axhline(0,color="#B0B0B0",linewidth=.65)
    axes[4].yaxis.set_major_locator(MaxNLocator(4))
    # All US direct comparisons are retained. Packages add declared implementation
    # costs to each same signed energy bill, without assuming response revenue.
    colors_package=["#6F9EBC","#AF94AD","#C78470"]
    fsource=[]
    for package,color,label in zip((names[0],names[2],names[3]),colors_package,("Existing","Supported","Interface")):
        g=nonalias(costs[costs.scenario_id.eq(package)]).query("comparison_mode == 'direct' and permission == 'native_zone'")
        values=np.sort(g.minimum_additional_annual_external_value_for_resource_breakeven_usd.to_numpy(float))
        axes[5].step(values,100*np.arange(1,len(values)+1)/len(values),where="post",color=color,linewidth=1.5,label=label)
        fsource.append(g)
    axes[5].set_xlabel("Required value (US$/year)")
    axes[5].set_ylabel("Cases at or below (%)")
    axes[5].set_ylim(-2,104)
    axes[5].set_yticks([0,50,100])
    axes[5].xaxis.set_major_locator(MaxNLocator(3))
    axes[5].legend(loc="lower right",handlelength=1,labelspacing=.15,handletextpad=.3,borderpad=0)
    for index in (0, 2, 4):
        axes[index].yaxis.set_label_coords(-.13, .5)
    for index in (1, 3, 5):
        axes[index].tick_params(axis="y", pad=3)
        axes[index].yaxis.set_label_coords(-.12, .5)
    axes[3].yaxis.set_label_coords(-.10, .5)
    for letter,data in zip("abcdef",(direct,common,selected.reset_index(),delhi,d,pd.concat(fsource))):
        dump_source(output,6,letter,data)
    caption=("Figure 6 | Service, bills and implementation resources. Six fixed compositions and two placements are combined. "
        "Personal range uses model-generated individual demand intervals. a, Direct US zone-control changes in range coverage "
        "and annual energy charges relative to Uniform average. b, Bill changes under common annual mismatch limits across "
        "distinct permissions, with 168 matched comparisons; symbols and whiskers show medians and interquartile ranges. "
        "c, Annualized resources in four declared implementation packages, valued with the median wage assumptions. "
        "Components separate initial setup, ongoing management, respondent time, software and optional interfaces. "
        "d, Matched Delhi direct zone cases: electricity-charge change and heating-gas change on a higher-heating-value basis. "
        "e, Energy-bill-funded manager time per added independent command group before fixed costs; negative values "
        "denote bill increases. Small points retain every case. f, Empirical distributions of additional annual value needed "
        "to cover energy charges and three implementation packages, using direct US zone cases. Costs use the declared "
        "historical tariffs, wage values and annualization rule. Response revenue is not assigned to isolated events.")
    (RESULTS/"captions"/(LABEL+".txt")).write_text(caption,encoding="utf-8")
    (output/"ALIGNMENT_REVIEW.json").write_text(json.dumps(layout_audit(fig,axes),indent=2),encoding="utf-8")
    save_plot(fig,LABEL,dpi=DPI)


draw_figure_06()


In [ ]:
def draw_s01():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S01"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    variance = panel("Fig1h.csv")

    components = list(variance.component.drop_duplicates())

    fig, a = canvas(
        [
            "Record-equal weighting",
            "Unit-equal weighting",
            "Building-equal weighting",
            "Source-equal weighting",
            "Identified-unit check",
            "Variance components",
        ]
    )

    labels = {
        "between_country": "Country",
        "between_city_within_country": "City",
        "between_building_within_city": "Building",
        "between_analysis_unit_within_building": "Within\nbuilding",
        "within_analysis_unit": "Within-unit\nvariation",
    }

    for ax, (weight, d) in zip(a, variance.groupby("weighting", sort=True)):
        # The title follows the actual group to avoid relying on sort order.
        ax.set_title(
            f"({chr(97+a.index(ax))}) "
            + weight.replace("_equal", " equal").replace("_", " ").capitalize(),
            loc="left",
            pad=18,
        )
        for j, g in enumerate(sorted(d.griffiths.unique())):
            q = d[d.griffiths == g].set_index("component").reindex(components)
            y = np.arange(len(q)) + (j - 1) * 0.22
            center = q.loo_median_share.to_numpy() * 100
            ax.errorbar(
                center,
                y,
                xerr=[
                    center - q.loo_min_share.to_numpy() * 100,
                    q.loo_max_share.to_numpy() * 100 - center,
                ],
                fmt="o",
                ms=3,
                capsize=2,
                color=GRIFFITHS_COLORS[g],
                label=f"g = {g:g}",
            )
        ax.set_yticks(
            range(len(components)),
            [labels.get(x, x.replace("_", " ")) for x in components],
            fontsize=11,
        )
        ax.set_xlabel("Archive variance share (%)")
        ax.set_xlim(-2, 85)

    a[0].legend(frameon=False, fontsize=11, loc="upper right")

    identity_variance = pd.read_csv(args.science_audit / "variance_recheck.csv")

    comps = list(identity_variance.component.drop_duplicates())

    groups = list(identity_variance.cohort.drop_duplicates())

    y = np.arange(len(groups))

    left = np.zeros(len(groups))

    for j, comp in enumerate(comps):
        q = identity_variance[identity_variance.component == comp].set_index("cohort").reindex(groups)
        a[4].barh(
            y, q.share * 100, left=left, color=VARIANCE_COLORS[j], label=comp.replace("_", " ")
        )
        left += q.share.to_numpy() * 100

    a[4].set_yticks(y, ["All records", "Identified\nsubjects"])

    a[4].set_xlabel("Archive variance share (%)")

    a[4].set_ylim(-0.5, 2.45)

    component_labels = ["Country", "City", "Building", "Within\nbuilding", "Within-unit\nvariation"]

    handles = [Patch(facecolor=VARIANCE_COLORS[j], edgecolor="none") for j in range(len(comps))]

    a[4].legend(
        handles, [label.replace("\n", " ") for label in component_labels], frameon=False, loc="upper left", ncol=2,
        handlelength=1, columnspacing=0.7, labelspacing=0.2,
    )

    arrays = [
        identity_variance[identity_variance.cohort == group].set_index("component").reindex(comps).share.to_numpy() * 100
        for group in groups
    ]

    for y, (v, w) in enumerate(zip(*arrays)):
        a[5].plot([v, w], [y, y], color="#aaaaaa", lw=1)

    for values, color, label in zip(
        arrays, [PALETTE["blue"], PALETTE["vermillion"]], ["All records", "Identified subjects"]
    ):
        a[5].scatter(values, range(5), s=35, color=color, label=label, zorder=3)

    a[5].set_yticks(range(5), component_labels)

    a[5].invert_yaxis()

    a[5].set_xlim(0, 65)

    a[5].set_xlabel("Variance share (%)")

    a[5].legend(frameon=False, loc="upper right")

    manifest.append(
        save(
            fig,
            out,
            2,
            "Figure S2 | Hierarchical variance and identity sensitivity. a–d, Leave-one-source variance shares under the four weighting schemes identified in the panel titles. Points are medians across twenty-seven source removals and whiskers give the minimum and maximum. Blue, gold and green denote Griffiths coefficients 0.33, 0.50 and 0.67 °C−1. The five components are country, city within country, building within city, analysis unit within building and repeated measurement within analysis unit. Building labels denote database groups; 150 identifiers are original and 37 inferred. e,f, Hierarchical variance shares for all canonical records and the subset with subject-within-building identifiers. Component colours in e match Figure 1d. Blue and terracotta in f identify all records and identified subjects, respectively. The identified-subject subset uses only original-source building identifiers.",
            ["Fig1h.csv", "science_audit/variance_recheck.csv"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S02"
    final_stem = "Figure_S01"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S01.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s01()

In [ ]:
def draw_s02():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S02"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    human = read("human/human_profiles_minimized.csv.gz")

    fig, a = canvas(["Uniform 24 °C", "Uniform 25 °C", "Uniform 26 °C", "Climate-group contrast"])

    summary = []

    for ax, target in zip(a[:3], [24, 25, 26]):
        for j, (g, col) in enumerate(
            zip(["reference_g033_c", "reference_g050_c", "reference_g067_c"], G_COLORS)
        ):
            values = human[g].to_numpy() - target
            ax.hist(
                values,
                bins=np.arange(-12, 12.5, 0.5),
                weights=human.unit_equal_weight,
                histtype="step",
                density=True,
                color=col,
                label=["g = 0.33", "g = 0.50", "g = 0.67"][j],
            )
            summary.append(
                {
                    "target_c": target,
                    "g_column": g,
                    "record_count": len(values),
                    "mean_absolute_gap_c": np.average(
                        np.abs(values), weights=human.unit_equal_weight
                    ),
                }
            )
        ax.axvline(0, color="#555555", lw=0.7)
        ax.set_xlim(-10, 10)
        ax.set_xlabel("Demand centre − target (°C)")
        ax.set_ylabel("Weighted density")

    a[0].legend(frameon=False, fontsize=11, loc="upper left", handlelength=.8, handletextpad=.5)

    groups = human.groupby("climate_group").size().sort_values(ascending=False).head(6).index

    vals = [human.loc[human.climate_group == g, "reference_g050_c"].values for g in groups]

    box(
        a[3],
        vals,
        list(range(1, len(groups) + 1)),
        "Demand centre (°C)",
        colors=[PALETTE[k] for k in ["blue", "vermillion", "green", "purple", "sky", "orange"]],
    )

    a[3].set_xlabel("Climate label (see caption)")

    a[3].tick_params(axis="x", labelsize=11)

    pd.DataFrame(summary).to_csv(out / "baseline_coordinate_summary.csv", index=False)

    manifest.append(
        save(
            fig,
            out,
            1,
            "Figure S1 | Reconstructed demand relative to uniform thermal coordinates. a–c, Analysis-unit-weighted demand-centre-minus-target distributions for 24, 25 and 26 °C coordinates at Griffiths coefficients 0.33, 0.50 and 0.67 °C−1, shown in blue, gold and green. d, Record-level demand-centre distributions for the six most represented climate labels, numbered in order: "
            + "; ".join(f"{i+1}, {g}" for i, g in enumerate(groups))
            + ". Boxes show medians and interquartile ranges, whiskers extend to 1.5 interquartile ranges, and the underlying data retain outliers.",
            ["human/human_profiles_minimized.csv.gz"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S01"
    final_stem = "Figure_S02"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S02.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s02()


In [ ]:
def draw_s03():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S03"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    from matplotlib.path import Path as MplPath

    from matplotlib.patches import Polygon

    land = json.loads(args.land.read_text(encoding="utf-8"))

    polygons = []

    for feature in land["features"]:
        geom = feature["geometry"]
        shapes = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
        polygons.extend(np.asarray(shape[0], dtype=float) for shape in shapes)

    lon, lat = np.meshgrid(np.arange(-179.5, 180, 1), np.arange(-54.5, 80, 1))

    xy = np.column_stack([lon.ravel(), lat.ravel()])

    land_mask = np.zeros(len(xy), dtype=bool)

    for poly in polygons:
        land_mask |= MplPath(poly).contains_points(xy)

    def distance(points, observed):
        lp = np.radians(points)
        lo = np.radians(observed)
        aa = (
            np.sin((lp[:, None, 1] - lo[None, :, 1]) / 2) ** 2
            + np.cos(lp[:, None, 1])
            * np.cos(lo[None, :, 1])
            * np.sin((lp[:, None, 0] - lo[None, :, 0]) / 2) ** 2
        )
        return 6371 * 2 * np.arcsin(np.sqrt(np.clip(aa, 0, 1)))

    def interpolate(dist, values):
        idx = np.argsort(dist, axis=1)[:, :4]
        near = np.take_along_axis(dist, idx, axis=1)
        weights = 1 / np.maximum(near, 1) ** 2
        return (weights * values[idx]).sum(axis=1) / weights.sum(axis=1)

    fig = plt.figure(figsize=(180 / 25.4, 220 / 25.4))

    gs = fig.add_gridspec(
        3,
        2,
        left=0.13,
        right=0.97,
        top=0.90,
        bottom=0.10,
        hspace=0.55,
        wspace=0.5,
        height_ratios=[1, 1, 1.1],
    )

    mapaxes = [fig.add_subplot(gs[0, :]), fig.add_subplot(gs[1, :])]

    cv = []

    grid_summaries = []

    for ax, season, label in zip(
        mapaxes,
        ["warm", "cold"],
        ["(a) Warm-season interpolation", "(b) Cold-season interpolation"],
    ):
        d = read("curves/city_season_demand_support.csv")
        d = d[(d.season_group == season) & d.supported]
        supported_count = len(d)
        d = d.dropna(subset=["map_longitude_0p5deg", "map_latitude_0p5deg", "g050_p50_c"]).copy()
        coords = d[["map_longitude_0p5deg", "map_latitude_0p5deg"]].to_numpy()
        values = d.g050_p50_c.to_numpy()
        dist = distance(xy, coords)
        nearest = dist.min(axis=1)
        z = interpolate(dist, values)
        keep = land_mask & (nearest <= 1000)
        shown = np.where(keep, z, np.nan).reshape(lon.shape)
        for poly in polygons:
            ax.add_patch(Polygon(poly, facecolor="#efefef", edgecolor="#bdbdbd", lw=0.35))
        im = ax.pcolormesh(
            lon,
            lat,
            shown,
            cmap=DEMAND_CMAP,
            vmin=19.5,
            vmax=30.6,
            shading="nearest",
            rasterized=True,
        )
        ax.scatter(
            coords[:, 0],
            coords[:, 1],
            c=values,
            cmap=DEMAND_CMAP,
            vmin=19.5,
            vmax=30.6,
            s=15,
            edgecolor="#7F7F7F",
            linewidth=0.4,
        )
        ax.set_xlim(-180, 180)
        ax.set_ylim(-55, 80)
        ax.set_aspect("equal")
        ax.set_title(label, loc="left", fontsize=12, weight="bold", pad=8)
        ax.set_xticks([-120, -60, 0, 60, 120])
        ax.set_yticks([-30, 0, 30, 60])
        ax.set_ylabel("Latitude (°)")
        ax.text(
            0.99,
            0.02,
            f"{len(d)} geolocated cities; 1,000-km mask",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=11,
        )
        cvdist = distance(coords, coords)
        np.fill_diagonal(cvdist, np.inf)
        pred = interpolate(cvdist, values)
        near = cvdist.min(axis=1)
        for city, obs, pr, km in zip(d.city, values, pred, near):
            cv.append(
                {
                    "city": city,
                    "season": season,
                    "observed_c": obs,
                    "heldout_prediction_c": pr,
                    "absolute_error_c": abs(pr - obs),
                    "nearest_training_city_km": km,
                }
            )
        for km in [500, 1000, 1500]:
            grid_summaries.append(
                {
                    "season": season,
                    "supported_city_summaries": supported_count,
                    "geolocated_city_summaries": len(d),
                    "excluded_without_scientific_coordinates": supported_count - len(d),
                    "mask_radius_km": km,
                    "supported_land_cells": int((land_mask & (nearest <= km)).sum()),
                    "land_cells": int(land_mask.sum()),
                }
            )

    cax = fig.add_axes([0.60, 0.945, 0.34, 0.012])

    bar = fig.colorbar(im, cax=cax, orientation="horizontal")

    bar.set_label("Reconstructed demand centre (°C)", labelpad=1)

    cax.xaxis.set_label_position("top")

    # All observed-city markers have the same area; they are location anchors,
    # not a second quantitative size scale.
    fig.legend(
        [Line2D([], [], marker="o", linestyle="none", markersize=4,
                markerfacecolor="white", markeredgecolor="#7F7F7F",
                markeredgewidth=.6)],
        ["Observed city (equal size)"],
        loc="upper left", bbox_to_anchor=(.13, .985), frameon=False,
        handlelength=.8, handletextpad=.4, borderaxespad=0,
    )

    ax = fig.add_subplot(gs[2, 0])

    ay = fig.add_subplot(gs[2, 1])

    cv = pd.DataFrame(cv)

    for j, (season, d) in enumerate(cv.groupby("season")):
        ax.scatter(
            d.observed_c,
            d.heldout_prediction_c,
            s=20,
            color=SEASON_COLORS[season],
            alpha=0.8,
            label=season.title(),
        )
        ay.scatter(
            d.nearest_training_city_km,
            d.absolute_error_c,
            s=20,
            color=SEASON_COLORS[season],
            alpha=0.8,
        )

    ax.plot([18, 32], [18, 32], ls="--", color="gray", lw=0.8)

    ax.set_xlabel("Observed city median (°C)")

    ax.set_ylabel("Held-out prediction (°C)")

    ax.legend(frameon=False, loc="upper left")

    ay.set_xlabel("Nearest training city (km)")

    ay.set_ylabel("Absolute held-out error (°C)")

    ay.axvline(1000, ls="--", color="gray", lw=0.8)

    ax.set_title("(c) Leave-one-city-out", loc="left", fontsize=12, weight="bold", pad=14)

    ay.set_title("(d) Transfer distance and error", loc="left", fontsize=12, weight="bold", pad=14)

    cv.to_csv(out / "spatial_interpolation_city_holdout.csv", index=False)

    pd.DataFrame(grid_summaries).to_csv(
        out / "spatial_interpolation_support_masks.csv", index=False
    )

    manifest.append(
        save(
            fig,
            out,
            14,
            "Figure S14 | Support-masked interpolation and out-of-city diagnostics. a,b, Four-neighbour inverse-distance-squared interpolation of observed warm- and cold-season city medians at Griffiths coefficient 0.50 °C−1. The common colour scale matches Figure 1a,b. Coloured land lies within 1,000 km of a geolocated observation. Grey land lies outside this mask. Points mark thirty warm-season and twenty-five cold-season geolocated city summaries. One supported Bangalore summary per season has a display-only coordinate and is excluded. c, Leave-one-city-out predictions without the held-out city in training. d, Absolute prediction error versus nearest training-city distance. Blue/terracotta denote cold/warm seasons. Data include errors and support masks at 500, 1,000 and 1,500 km.",
            ["curves/city_season_demand_support.csv", "natural_earth_110m_land"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S14"
    final_stem = "Figure_S03"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S03.txt").read_text(encoding="utf-8")
    caption = caption.replace(
        "Points mark thirty warm-season and twenty-five cold-season geolocated city summaries.",
        "Equal-sized circles locate thirty warm-season and twenty-five cold-season city summaries. "
        "Circle colour gives the record-derived city median demand centre on the common temperature scale. "
        "Circle size does not encode record counts or geographic area.",
    )
    caption = caption.replace("Circle colour gives the observed city median on the common temperature scale.",
                              "Circle colour gives the record-derived city median demand centre on the common temperature scale.")
    (RESULTS / "captions/Figure_S03.txt").write_text(caption, encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s03()


In [ ]:
def draw_s04():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S04"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    audit = args.spatial_audit

    metrics = pd.read_csv(audit / "metrics_summary.csv")

    oof = pd.read_csv(audit / "oof_predictions.csv")

    paired = pd.read_csv(audit / "paired_cluster_bootstrap_vs_mean.csv")

    fig, a = canvas(
        [
            "Out-of-city RMSE",
            "Out-of-city MAE",
            "GPR interval calibration",
            "Coefficient sensitivity",
        ]
    )

    primary = metrics[metrics.target.eq("g050_p50_c")].copy()

    methods = ["global_training_mean", "idw2_4nn", "gpr_spherical_xyz"]

    method_labels = ["Training mean", "4-neighbour IDW²", "Spherical GPR"]

    scopes = ["all", "cold", "warm"]

    scope_labels = ["Pooled", "Cold", "Warm"]

    x = np.arange(len(scopes))

    width = 0.23

    for ax, metric, ylabel in [(a[0], "rmse_c", "RMSE (°C)"), (a[1], "mae_c", "MAE (°C)")]:
        for j, (method, label) in enumerate(zip(methods, method_labels)):
            q = primary[primary.model.eq(method)].set_index("season_scope").reindex(scopes)
            ax.bar(
                x + (j - 1) * width, q[metric], width, color=(COLORS + ["#777777"])[j], label=label
            )
        ax.set_xticks(x, scope_labels)
        ax.set_ylabel(ylabel)
        ax.set_ylim(0, 2.8)

    a[1].legend(frameon=False, fontsize=11, loc="upper left")

    gp = oof[(oof.target == "g050_p50_c") & (oof.model == "gpr_spherical_xyz")].copy()

    covered = gp.covered_95.astype(str).str.lower().eq("true")

    for mask, color, label in [(covered, COLORS[0], "Covered"), (~covered, COLORS[1], "Missed")]:
        q = gp[mask]
        a[2].vlines(q.observed_c, q.lower_95_c, q.upper_95_c, color=color, alpha=0.55, lw=0.7)
        a[2].scatter(
            q.observed_c, q.prediction_c, s=19, color=color, alpha=0.8, label=label, zorder=3
        )

    lo = min(gp.observed_c.min(), gp.lower_95_c.min())

    hi = max(gp.observed_c.max(), gp.upper_95_c.max())

    a[2].plot([lo, hi], [lo, hi], ls="--", lw=0.8, color="gray")

    a[2].set_xlim(lo - 0.3, hi + 0.3)

    a[2].set_ylim(lo - 0.3, hi + 0.3)

    a[2].set_xlabel("Observed city median (°C)")

    a[2].set_ylabel("GPR prediction and 95% interval (°C)")

    coverage = covered.mean() * 100

    wilson_low = (
        primary[
            (primary.model == "gpr_spherical_xyz") & (primary.season_scope == "all")
        ].coverage_95_wilson_low.iloc[0]
        * 100
    )

    wilson_high = (
        primary[
            (primary.model == "gpr_spherical_xyz") & (primary.season_scope == "all")
        ].coverage_95_wilson_high.iloc[0]
        * 100
    )

    a[2].text(
        0.03,
        0.96,
        f"{covered.sum()}/{len(gp)} covered ({coverage:.1f}%)\nWilson 95%: {wilson_low:.1f}–{wilson_high:.1f}%",
        transform=a[2].transAxes,
        va="top",
    )

    a[2].legend(frameon=False, loc="lower right")

    pooled = paired[
        (paired.season_scope == "all") & paired.model.isin(["idw2_4nn", "gpr_spherical_xyz"])
    ].copy()

    target_order = ["g033_p50_c", "g050_p50_c", "g067_p50_c"]

    rows = []

    for target in target_order:
        for method in ["idw2_4nn", "gpr_spherical_xyz"]:
            rows.append(pooled[(pooled.target == target) & (pooled.model == method)].iloc[0])

    y = np.arange(len(rows))

    labels = [
        f'g = {r.target[1:4].replace("0", ".", 1)} ' + ("IDW²" if r.model == "idw2_4nn" else "GPR")
        for r in rows
    ]

    for offset, metric, marker, color, label in [
        (-0.10, "mae", "o", COLORS[0], "ΔMAE"),
        (0.10, "rmse", "^", COLORS[1], "ΔRMSE"),
    ]:
        center = np.array([r[f"delta_{metric}_c_model_minus_mean"] for r in rows])
        low = np.array([r[f"delta_{metric}_c_ci_low"] for r in rows])
        high = np.array([r[f"delta_{metric}_c_ci_high"] for r in rows])
        a[3].errorbar(
            center,
            y + offset,
            xerr=[center - low, high - center],
            fmt=marker,
            color=color,
            capsize=2,
            ms=4,
            label=label,
        )

    a[3].axvline(0, color="gray", lw=0.8, ls="--")

    a[3].set_yticks(y, labels)

    a[3].invert_yaxis()

    a[3].set_xlabel("Spatial model − training mean (°C)")

    a[3].set_xlim(-0.7, 1.25)

    a[3].plot(0.84, 2.15, marker="o", color=COLORS[0], ms=5)

    a[3].text(0.94, 2.15, "ΔMAE", va="center")

    a[3].plot(0.84, 2.55, marker="^", color=COLORS[1], ms=5)

    a[3].text(0.94, 2.55, "ΔRMSE", va="center")

    manifest.append(
        save(
            fig,
            out,
            15,
            "Figure S15 | Whole-city holdout audit of coordinate-based spatial prediction. a,b, Held-out root-mean-square error (RMSE) and mean absolute error (MAE) for the training mean, four-neighbour inverse-distance-squared interpolation (IDW²) and spherical Gaussian-process regression (GPR), at Griffiths coefficient 0.50 °C−1. c, GPR predictions and nominal 95% intervals for fifty-five held-out city–season summaries. Blue/terracotta mark covered/missed observations. Forty-five are covered (81.8%, Wilson interval 69.7–89.8%), with mean interval width 6.82 °C. d, Paired city-cluster bootstrap differences from the training-mean baseline in pooled MAE and RMSE under three Griffiths coefficients. Points and horizontal lines show estimates and intervals from fixed out-of-fold predictions. Negative values favour the spatial method.",
            [
                "spatial_method_audit/metrics_summary.csv",
                "spatial_method_audit/oof_predictions.csv",
                "spatial_method_audit/paired_cluster_bootstrap_vs_mean.csv",
            ],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S15"
    final_stem = "Figure_S04"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S04.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s04()

In [ ]:
def draw_s05():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S05"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    gender = pd.read_csv(args.science_audit / "demographic_city_season_gender_summary.csv")

    age = pd.read_csv(args.science_audit / "demographic_city_season_age_summary.csv")

    fig, a = canvas(
        [
            "Gender centre contrast",
            "Gender width contrast",
            "Age-band descriptions",
            "Demographic coverage",
        ]
    )

    for ax, metric in zip(a[:2], ["median", "p10_p90_width"]):
        for j, g in enumerate(["033", "050", "067"]):
            field = f"reference_center_g{g}_{metric}_c"
            q = gender.pivot(index=["country", "city", "season"], columns="gender", values=field)
            values = (q["female"] - q["male"]).dropna()
            ax.hist(
                values,
                bins=np.arange(-4, 4.5, 0.5),
                histtype="step",
                color=G_COLORS[j],
                label=["g = 0.33", "g = 0.50", "g = 0.67"][j],
            )
        ax.axvline(0, color="gray", lw=0.7)
        ax.set_ylabel("City–season settings")

    a[0].set_xlabel("Female − male median (°C)")

    a[1].set_xlabel("Female − male width (°C)")

    a[0].legend(frameon=False, fontsize=11)

    bands = [
        x
        for x in ["under_20", "20_29", "30_39", "40_49", "50_59", "60_plus"]
        if x in set(age.age_band)
    ]

    if not bands:
        bands = list(age.age_band.drop_duplicates())

    box(
        a[2],
        [age.loc[age.age_band == x, "reference_center_g050_median_c"] for x in bands],
        [x.replace("under_", "<").replace("_plus", "+").replace("_", "–") for x in bands],
        "City–season median (°C)",
        colors=[PALETTE[k] for k in ["blue", "vermillion", "green", "purple", "sky", "orange"]],
    )

    a[2].set_xlabel("Age band (years)")

    a[2].tick_params(axis="x", rotation=30)

    a[3].bar(
        ["Age", "Gender"], [10283 / 21418 * 100, 12941 / 21418 * 100], color=COLORS[:2], alpha=0.8
    )

    a[3].set_ylim(0, 100)

    a[3].set_ylabel("Office records with data (%)")

    a[3].text(0, 52, "10,283 / 21,418", ha="center", fontsize=11)

    a[3].text(1, 64, "12,941 / 21,418", ha="center", fontsize=11)

    manifest.append(
        save(
            fig,
            out,
            11,
            "Figure S11 | Demographic demand descriptions and coverage. a,b, Female-minus-male contrasts in reconstructed median demand and P10–P90 width within matched country–city–season settings, under Griffiths coefficients 0.33, 0.50 and 0.67 °C−1. c, City–season median demand-centre distributions by recorded age band. Boxes show medians and interquartile ranges, with whiskers at 1.5 interquartile ranges. d, Original age/gender field coverage among 21,418 office records. Displayed setting–group summaries contain at least twenty records and ten analysis units.",
            [
                "science_audit/demographic_city_season_gender_summary.csv",
                "science_audit/demographic_city_season_age_summary.csv",
                "science_audit/demographic_coverage.csv",
            ],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S11"
    final_stem = "Figure_S05"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S05.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s05()

In [ ]:
def draw_s06():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S06"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    votes = read("human/direct_vote_windows_sanitized.csv.gz")

    eligible = votes[votes.eligible.astype(str).str.lower() == "true"]

    fig, a = canvas(
        ["Singapore room windows", "Longitudinal room windows", "Requests within windows", "Compression gaps"]
    )

    for ax, code in zip(a[:2], ["NUS_HAAS", "ENTH"]):
        q = eligible[eligible.dataset_code == code]
        ax.scatter(
            q.full_burden,
            q.compressed_burden,
            s=22,
            color=PALETTE["blue"] if code == "NUS_HAAS" else PALETTE["orange"],
            alpha=0.85,
        )
        ax.plot([0, 1], [0, 1], ls="--", color="gray", lw=0.8)
        ax.set_xlim(-0.03, 1.03)
        ax.set_ylim(-0.03, 1.03)
        ax.set_xlabel("Full vote burden")
        ax.set_ylabel("Burden after averaging")
        ax.text(0.04, 0.94, f"n = {len(q)} windows", transform=ax.transAxes, va="top")

    d = panel("Fig2h.csv").drop_duplicates("dataset")

    xs = np.arange(len(d))

    w = 0.24

    for j, (col, label) in enumerate(
        [
            ("disagreement_count", "Differing"),
            ("opposing_count", "Opposing"),
            ("cancellation_count", "Cancelled"),
        ]
    ):
        a[2].bar(
            xs + (j - 1) * w,
            d[col],
            w,
            label=label,
            color=[PALETTE["blue"], PALETTE["vermillion"], PALETTE["dark_grey"]][j],
        )

    a[2].set_xticks(xs, d.dataset.replace({"NUS_HAAS": "Singapore", "ENTH": "Longitudinal"}))

    a[2].set_ylabel("Windows")

    a[2].legend(frameon=False, fontsize=11)

    gap_values = [
        eligible.loc[eligible.dataset_code == x, "jensen_gap"].values for x in ["NUS_HAAS", "ENTH"]
    ]

    box(
        a[3],
        gap_values,
        ["Singapore", "Longitudinal"],
        "Full minus averaged burden",
        colors=[PALETTE["blue"], PALETTE["orange"]],
    )

    for j, values in enumerate(gap_values):
        a[3].scatter(
            j + 1 + np.linspace(-0.15, 0.15, len(values)),
            values,
            s=10,
            color=[PALETTE["blue"], PALETTE["orange"]][j],
            alpha=0.8,
        )

    manifest.append(
        save(
            fig,
            out,
            4,
            "Figure S4 | Shared-room vote replication. a,b, Mean absolute vote (full burden) versus absolute mean vote (burden after averaging) for eligible NUS and ENTH thirty-minute room-windows. Dashed lines mark equality. c, Window counts containing differing categories, opposing warmer/cooler requests, and exact cancellation. d, Full-minus-averaged burden distributions. The National University of Singapore (NUS) and ENTH datasets contain 63/34 windows and 225/81 votes. Boxes show medians and interquartile ranges, whiskers extend to 1.5 interquartile ranges, and points show individual windows.",
            ["human/direct_vote_windows_sanitized.csv.gz", "Fig2h.csv"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S04"
    final_stem = "Figure_S06"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S06.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s06()

In [ ]:
# Figure_S07: edit this cell and rerun to update this figure.
def draw_figure_s07():
    LABEL = "Figure_S07"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import json

    from pathlib import Path

    from typing import Callable

    import matplotlib.pyplot as plt

    import numpy as np

    import pandas as pd

    from matplotlib.axes import Axes

    from matplotlib.figure import Figure

    from matplotlib.patches import FancyBboxPatch, Patch, Rectangle

    from thermal_heterogeneity.figures import (
        CONTEXT_COLORS as SHARED_CONTEXT_COLORS,
        DEMAND_CMAP,
        VARIANCE_COLORS,
        PALETTE,
        create_panel_figure,
        data_safe_legend,
        draw_world_land,
        style_panel,
    )

    CONTEXT_ORDER = ["Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019"]

    CONTEXT_LABELS = {
        "Miami": "Miami",
        "NewDelhi": "New Delhi",
        "PortAngeles_small2019": "Port Angeles",
        "Seattle_small2019": "Seattle",
    }

    CONTEXT_LEGEND_LABELS = {
        "Miami": "Miami",
        "NewDelhi": "New Delhi",
        "PortAngeles_small2019": "Port Angeles",
        "Seattle_small2019": "Seattle",
    }

    CONTEXT_COLORS = SHARED_CONTEXT_COLORS

    POPULATION_LABELS = {
        "us_archive_stratum": "United States",
        "multinational_source_balanced": "Multinational",
        "india_archive_stratum": "India",
    }

    OUTCOME_ORDER = [
        "same_direction_decrease",
        "same_direction_increase",
        "all_tied",
        "direction_not_stable",
        "match_lost",
        "all_unmatched",
    ]

    OUTCOME_LABELS = {
        "same_direction_decrease": "Decrease",
        "same_direction_increase": "Increase",
        "all_tied": "Tie",
        "direction_not_stable": "Unstable",
        "match_lost": "Unmatched",
        "all_unmatched": "Unmatched",
    }

    OUTCOME_COLORS = {
        "same_direction_decrease": PALETTE["blue"],
        "same_direction_increase": PALETTE["vermillion"],
        "all_tied": PALETTE["grey"],
        "direction_not_stable": PALETTE["purple"],
        "match_lost": PALETTE["light_grey"],
        "all_unmatched": PALETTE["light_grey"],
    }

    def _read_csv(root: Path, panel_id: str, required: set[str]) -> pd.DataFrame:
        path = root / f"{panel_id}.csv"
        frame = pd.read_csv(path)
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"{panel_id} missing columns: {sorted(missing)}")
        if frame.empty or set(frame["panel_id"]) != {panel_id}:
            raise ValueError(f"{panel_id} is empty or carries the wrong panel_id")
        return frame

    def _finish_axis(axis: Axes, letter: str, title: str, grid_axis: str | None = "y") -> None:
        style_panel(axis, letter, title, grid_axis=grid_axis)

    def build_figure_2(panel_root: Path | str) -> Figure:
        """Build Figure 2: independent direct-vote evidence for compression loss."""

        root = Path(panel_root)
        a = _read_csv(root, "Fig2a", {"stage", "rows_or_windows", "exclusion_rule"})
        b = _read_csv(root, "Fig2b", {"window_id", "n_votes", "n_cooler", "n_no_change", "n_warmer", "share_cooler", "share_no_change", "share_warmer"})
        c = _read_csv(root, "Fig2c", {"window_id", "n_votes", "n_cooler", "n_no_change", "n_warmer", "temperature_mean_c", "temperature_spread_c"})
        d = _read_csv(root, "Fig2d", {"window_id", "n_votes", "full_burden", "compressed_burden", "jensen_gap"})
        e = _read_csv(root, "Fig2e", {"window_id", "full_burden", "compressed_burden", "n_votes", "n_cooler", "n_warmer"})
        f = _read_csv(root, "Fig2f", {"threshold", "false_accept_count", "false_reject_count", "eligible_windows"})
        g = _read_csv(root, "Fig2g", {"window_minutes", "metric", "estimate", "ci_low", "ci_high", "eligible_windows", "votes"})
        h = _read_csv(root, "Fig2h", {"dataset", "eligible_windows", "votes", "disagreement_count", "opposing_count", "cancellation_count", "metric", "estimate", "ci_low", "ci_high"})

        figure, axes = create_panel_figure()
        figure.subplots_adjust(left=0.13, wspace=0.38, hspace=0.40)
        ax = axes.ravel()

        # a: fixed construction flow.
        stage_order = ["raw rows", "declared cohort", "valid votes", "candidate windows", "minimum-user windows", "temperature-homogeneous windows", "person-window votes"]
        frame = a.set_index("stage").loc[stage_order].reset_index()
        y = np.arange(len(frame))[::-1]
        ax[0].set_xlim(0, 1)
        ax[0].set_ylim(-0.55, len(frame) - 0.45)
        ax[0].set_xticks([])
        ax[0].set_yticks([])
        ax[0].axis("off")
        for index, (yy, row) in enumerate(zip(y, frame.itertuples())):
            ax[0].text(0.04, yy, row.stage.replace("temperature-homogeneous", "temperature-homog."), ha="left", va="center")
            ax[0].text(0.80, yy, f"{int(row.rows_or_windows):,}", ha="right", va="center")
            if index < len(frame) - 1:
                ax[0].annotate("", xy=(0.88, yy - 0.72), xytext=(0.88, yy - 0.18), arrowprops={"arrowstyle": "->", "lw": 0.75, "color": "#666666"})
        ax[0].text(0.98, y[4], "≥3 users", ha="right", va="center")
        ax[0].text(0.98, y[5], "≤0.5 °C", ha="right", va="center")
        _finish_axis(ax[0], "a", "Independent cohort flow", None)

        # b: all 63 windows as thin composition bars.
        ordered = b.assign(net=b["share_warmer"] - b["share_cooler"], disagreement=(b[["n_cooler", "n_no_change", "n_warmer"]] > 0).sum(axis=1) > 1).sort_values(["net", "disagreement"])
        yy = np.arange(len(ordered))
        left = np.zeros(len(ordered))
        for field, label, color in [
            ("share_cooler", "Cooler", PALETTE["blue"]),
            ("share_no_change", "No change", PALETTE["light_grey"]),
            ("share_warmer", "Warmer", PALETTE["vermillion"]),
        ]:
            values = ordered[field].to_numpy(float)
            bars = ax[1].barh(yy, values, left=left, height=0.88, color=color, label=label)
            for patch in bars.patches:
                patch.set_gid("fig2b-window-bar")
            left += values
        ax[1].set_xlim(0, 1)
        ax[1].set_ylim(-1, len(ordered) + 30)
        ax[1].set_yticks([])
        ax[1].set_xticks([0, 0.5, 1], ["0", "50", "100"])
        ax[1].set_xlabel("Votes within room-window (%)")
        summary_text = ax[1].text(
            0.02,
            0.98,
            "49/63 disagreement · 6/63 opposite",
            ha="left",
            va="top",
            transform=ax[1].transAxes,
        )
        summary_text.set_gid("fig2b-summary")
        legend = ax[1].legend(
            loc="upper center",
            ncol=3,
            frameon=False,
            bbox_to_anchor=(0.5, 0.89),
            handlelength=0.8,
            columnspacing=0.7,
            handletextpad=0.3,
        )
        legend.set_gid("fig2b-legend")
        _finish_axis(ax[1], "b", "Votes within shared rooms", "x")

        # c: six opposite-request windows.
        labels = [f"Window {i+1}" for i in range(len(c))]
        yy = np.arange(len(c))
        ax[2].barh(yy, -c["n_cooler"], color=PALETTE["blue"], height=0.58)
        ax[2].barh(yy, c["n_warmer"], color=PALETTE["vermillion"], height=0.58)
        ax[2].scatter(np.zeros(len(c)), yy, s=18 + c["n_no_change"] * 18, color=PALETTE["light_grey"], edgecolors="#555555", zorder=3)
        for index, row in enumerate(c.itertuples()):
            ax[2].text(
                1.18,
                index,
                f"n={row.n_votes} | {row.temperature_mean_c:.1f}°C | {row.temperature_spread_c:.1f}°",
                ha="left",
                va="center",
                clip_on=True,
            )
        ax[2].axvline(0, color="#444444", linewidth=0.7)
        ax[2].set_xlim(-3.5, 5.3)
        ax[2].set_xticks([-2, 0, 2, 4])
        ax[2].set_yticks(yy, labels)
        ax[2].set_xlabel("Cooler ← vote count → warmer", labelpad=-2)
        ax[2].xaxis.label.set_gid("fig2c-xlabel")
        _finish_axis(ax[2], "c", "Within-window opposing requests", "x")

        # d: Jensen compression relation and exact cancellation.
        exact = (d["compressed_burden"] == 0) & (d["full_burden"] > 0)
        ax[3].plot([0, 1], [0, 1], color="#777777", linewidth=0.8, linestyle="--")
        for row in d.itertuples():
            ax[3].plot([row.full_burden, row.full_burden], [row.compressed_burden, row.full_burden], color="#D0D0D0", linewidth=0.5)
        ax[3].scatter(d.loc[~exact, "full_burden"], d.loc[~exact, "compressed_burden"], s=22, color=PALETTE["blue"], alpha=0.8)
        ax[3].scatter(d.loc[exact, "full_burden"], d.loc[exact, "compressed_burden"], s=34, facecolors="white", edgecolors=PALETTE["vermillion"], linewidths=1.2)
        ax[3].set_xlim(-0.03, 1.03)
        ax[3].set_ylim(-0.03, 1.03)
        ax[3].set_xticks([0, 0.5, 1])
        ax[3].set_yticks([0, 0.5, 1])
        ax[3].set_xlabel("Full burden")
        ax[3].set_ylabel("Averaged burden")
        ax[3].text(0.98, 0.07, f"median gap={d['jensen_gap'].median():.3f}\nn=63 windows", ha="right", va="bottom")
        ax[3].xaxis.label.set_gid("fig2d-xlabel")
        _finish_axis(ax[3], "d", "Averaging compresses burden", "both")

        # e: four exact cancellations.
        yy = np.arange(len(e))
        for index, row in enumerate(e.itertuples()):
            ax[4].plot([row.compressed_burden, row.full_burden], [index, index], color="#888888", linewidth=1.3)
            ax[4].scatter(
                row.compressed_burden,
                index,
                marker="s",
                s=32,
                color=PALETTE["dark_grey"],
                label="Averaged" if index == 0 else None,
            )
            ax[4].scatter(
                row.full_burden,
                index,
                marker="o",
                s=36,
                color=PALETTE["blue"],
                label="Full" if index == 0 else None,
            )
            ax[4].text(row.full_burden + 0.04, index, f"{row.n_cooler}C/{row.n_warmer}W · n={row.n_votes}", va="center")
        ax[4].set_yticks(yy, [f"Case {value+1}" for value in yy])
        ax[4].set_xlim(-0.03, 1.02)
        ax[4].set_xticks([0, 0.5, 1])
        ax[4].set_ylim(-0.35, len(e) + 0.45)
        ax[4].set_xlabel("Request burden")
        ax[4].xaxis.label.set_gid("fig2e-xlabel")
        ax[4].legend(
            loc="upper center",
            ncol=2,
            frameon=False,
            bbox_to_anchor=(0.5, 0.99),
            handlelength=0.8,
            columnspacing=0.8,
            handletextpad=0.3,
        )
        _finish_axis(ax[4], "e", "Exact cancellation cases", "x")

        # f: false rejects remain visible at zero.
        x = np.arange(len(f))
        width = 0.34
        ax[5].bar(x - width / 2, f["false_accept_count"], width, color=PALETTE["vermillion"], label="False accept")
        ax[5].bar(x + width / 2, f["false_reject_count"], width, facecolor="white", edgecolor=PALETTE["blue"], hatch="//", label="False reject")
        for index, row in enumerate(f.itertuples()):
            ax[5].text(index - width / 2, row.false_accept_count + 0.18, str(row.false_accept_count), ha="center", va="bottom")
            ax[5].text(index + width / 2, 0.18, str(row.false_reject_count), ha="center", va="bottom")
        ax[5].set_xticks(x, [f"{value:g}" for value in f["threshold"]])
        ax[5].set_ylim(0, max(f["false_accept_count"]) + 2.2)
        ax[5].set_yticks([0, 2, 4, 6])
        ax[5].set_xlabel("Service threshold · 63 windows")
        ax[5].set_ylabel("Windows (count)")
        ax[5].legend(loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.0), handlelength=0.9, columnspacing=0.8)
        _finish_axis(ax[5], "f", "One-sided qualification error", "y")

        # g: all metrics on a percentage scale; Jensen gap is multiplied by 100.
        metric_styles = {
            "disagreement_rate": ("Disagreement", PALETTE["blue"], "o"),
            "opposing_request_rate": ("Opposite", PALETTE["vermillion"], "^"),
            "mean_jensen_gap": ("Jensen gap", PALETTE["dark_grey"], "s"),
        }
        label_offsets = {
            "disagreement_rate": (8, 0),
            "opposing_request_rate": (8, 10),
            "mean_jensen_gap": (8, -1),
        }
        for metric, (label, color, marker) in metric_styles.items():
            group = g.loc[g["metric"] == metric].sort_values("window_minutes")
            xvals = group["window_minutes"].to_numpy(float)
            vals = group["estimate"].to_numpy(float) * 100
            low = group["ci_low"].to_numpy(float) * 100
            high = group["ci_high"].to_numpy(float) * 100
            ax[6].errorbar(xvals, vals, yerr=[vals-low, high-vals], color=color, marker=marker, linewidth=1.2, capsize=2, label=label)
            end_label = ax[6].annotate(
                label,
                (xvals[-1], vals[-1]),
                xytext=label_offsets[metric],
                textcoords="offset points",
                color=color,
                ha="left",
                va="center",
                annotation_clip=True,
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 0.2},
            )
            end_label.set_gid("fig2g-end-label")
            end_label.set_clip_on(True)
        ax[6].set_xlim(12, 92)
        ax[6].set_xticks([15, 30, 60], ["15 min\nn=65", "30 min\nn=63", "60 min\nn=56"])
        ax[6].set_ylim(0, 105)
        ax[6].set_yticks([0, 25, 50, 75, 100])
        ax[6].set_xlabel("")
        ax[6].set_ylabel("Rate or gap (%, 95% CI)")
        _finish_axis(ax[6], "g", "Window-definition sensitivity", "y")

        # h: side-by-side external replication on a common percentage scale.
        metric_order = ["disagreement_rate", "opposing_request_rate", "mean_jensen_gap"]
        metric_labels = ["Disagree", "Opposite", "Jensen gap"]
        x = np.arange(3)
        for dataset, offset, color, marker in [("NUS_HAAS", -0.11, PALETTE["blue"], "o"), ("ENTH", 0.11, PALETTE["orange"], "s")]:
            group = h.loc[h["dataset"] == dataset].set_index("metric").loc[metric_order]
            values = group["estimate"].to_numpy(float) * 100
            low = group["ci_low"].to_numpy(float) * 100
            high = group["ci_high"].to_numpy(float) * 100
            ax[7].errorbar(x + offset, values, yerr=[values-low, high-values], color=color, marker=marker, linestyle="", capsize=2, label="NUS" if dataset == "NUS_HAAS" else "ENTH")
        ax[7].set_xticks(x, metric_labels)
        ax[7].set_ylim(0, 105)
        ax[7].set_yticks([0, 25, 50, 75, 100])
        ax[7].set_ylabel("Estimate (%, 95% CI)")
        ax[7].legend(loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.0), handlelength=0.8, columnspacing=0.8)
        nus = h.loc[h["dataset"] == "NUS_HAAS"].iloc[0]
        enth = h.loc[h["dataset"] == "ENTH"].iloc[0]
        ax[7].text(0.98, 0.56, f"NUS {int(nus.disagreement_count)}/{int(nus.eligible_windows)} disagree\n{int(nus.opposing_count)} opposite · {int(nus.cancellation_count)} cancel\nENTH {int(enth.disagreement_count)}/{int(enth.eligible_windows)} disagree\n{int(enth.opposing_count)} opposite · {int(enth.cancellation_count)} cancel", ha="right", va="center", transform=ax[7].transAxes)
        _finish_axis(ax[7], "h", "Independent ENTH replication", "y")

        return figure

    CAPTIONS = {
        1: (
            "**Figure 1 | Observed human demand is geographically structured but dominated by "
            "within-building heterogeneity.** **a,** The complete office archive contains 21,418 records "
            "from 12,130 analysis units; the executable set retains 21,399 records from 12,113 "
            "analysis units in 187 buildings, 40 cities and 15 countries. **b,c,** Observed-city warm- "
            "and cold-season demand centres. Filled circles denote season-supported cities, hollow "
            "circles observed but unsupported cities, circle area the number of analysis units and "
            "colour the demand centre on the same 19.5–30.6 °C scale; unsampled geography is left "
            "blank and no spatial interpolation is applied. **d,** Empirical P10–P90 intervals and "
            "medians for city- and source-level warm/cold aggregates under analysis-unit-equal "
            "weighting; labels give the number of aggregates and median interval width. **e,** "
            "Hierarchical variance allocation across country (22.3%), city within country (7.7%), "
            "building within city (5.8%), analysis unit within building (56.1%) and repeated "
            "measurements (8.2%). **f,** Weighted demand records below, within and above nine bilateral "
            "policy or design ranges at Griffiths coefficient 0.50; labels give the within-range "
            "share and eligible-record denominator, and whiskers are nested-bootstrap 95% confidence "
            "intervals for the within-range share. **g,** Empirical P10–P90 (thin), P25–P75 (thick) and "
            "median (point) of demand centre minus campaign target for three prespecified campaigns; "
            "positive values indicate a demand centre warmer than the target, and negative values "
            "indicate a demand centre cooler than the target. **h,** Leave-one-source sensitivity ranges "
            "and medians across 12 coefficient/weighting specifications, each based on 27 source "
            "removals. Policy documents in **f,g** are external temperature coordinates rather than "
            "individual comfort ground truth."
        ),
        2: (
            "**Figure 2 | Independent shared-room votes reveal demand disagreement hidden by "
            "averages.** **a,** Prespecified construction of the NUS 30-min validation cohort from "
            "4,378 raw rows to 63 temperature-homogeneous room-windows containing 225 valid "
            "person-window votes. **b,** Cooler, no-change and warmer vote shares in all 63 "
            "room-windows, ordered by net vote and disagreement; 49 windows contain more than one "
            "response category and six contain warmer and cooler requests within the same 30-min "
            "room-window. **c,** Vote counts in the six room-windows with within-window opposing "
            "requests; grey circles denote no-change votes and labels give vote count, mean room "
            "temperature and within-window temperature span. **d,** Full request burden, mean "
            "absolute vote, versus burden after averaging, absolute mean vote, for each window. "
            "Hollow orange points are exact cancellations; the dashed line is equality and the "
            "median compression gap is zero despite a positive mean gap. **e,** The four exact-"
            "cancellation windows, in which non-zero full burden is represented as zero after "
            "averaging. **f,** False acceptance and false rejection at burden thresholds 0, 0.25, "
            "0.50 and 0.75; all four thresholds retain zero false rejections and 2–5 false "
            "acceptances among 63 windows. **g,** Disagreement, within-window opposing requests and "
            "mean Jensen gap under 15-, 30- and 60-min windows; whiskers are crossed room/date "
            "cluster-bootstrap 95% confidence intervals and n is the eligible-window denominator. "
            "**h,** Independent ENTH replication (34 windows, 81 votes) beside NUS (63 windows, 225 "
            "votes). Windows, rather than votes, are the inferential unit throughout."
        ),
        3: (
            "**Figure 3 | Demand location redistributes annual energy and carbon under three "
            "controls.** **a,** Uniform-average, personal-centre and personal-range control "
            "families generate physical actions in the same building under identical operating "
            "permissions. These controls are distinct from the mean and full-distribution "
            "qualification representations used to score the same actions. **b,** Assigned P10–P90 "
            "demand intervals, centres and seed variation for three population compositions in four "
            "building/climate contexts. **c,** Prespecified orthogonal transformation "
            "T′=μ+Δ+λ(T−μ), with demand-location shift Δ from −2 to +2 °C and spread scale λ from 0 "
            "to 1.5; the open point is the Δ=0, λ=1 reference. **d,e,** Median and interquartile "
            "range of matched service-qualified changes in purchased space-HVAC site energy and "
            "operational CO₂ relative to the reference. The coloured denominator tables give "
            "matched n for Miami, New Delhi, Port Angeles and Seattle in left-to-right Δ order (−2, "
            "−1, +1 and +2 °C). **f,** Median signed target-minus-reference changes in heating "
            "electricity, cooling electricity, fan electricity and gas HHV for Δ=+2 °C, using the "
            "amended reference/target carrier fields directly; diamonds show median total-energy "
            "change and the two-line context labels give 22, 20, 8 and 9 matched cells. The artwork "
            "states the full accounting boundary of 59 matched and 85 unmatched design rows; "
            "unmatched rows contribute no fabricated delta. **g,** Energy direction for all 130 "
            "default uniform-versus-alternative paired definitions, separated by control and "
            "context: 54 increases, 3 decreases, 36 ties, 7 direction-unstable comparisons and 30 "
            "unmatched comparisons. **h,** Outcome shares across Griffiths energy, Griffiths carbon "
            "and personal-range-width sensitivity families with explicit denominators and the same outcome "
            "colour/hatch key as panel g. Population assignments are controlled archive scenarios "
            "rather than claims of local population representativeness; carbon factors are fixed "
            "regional scenarios."
        ),
    }

    ALT_TEXT = {
        1: (
            "Eight-panel portrait figure. An archive flow retains 21,399 of 21,418 records. Two "
            "observed-city maps show 31 warm-season and 26 cold-season supported cities without "
            "interpolation. Ranked empirical intervals show broad within-city and within-source "
            "demand ranges. A variance bar assigns the largest share, 56.1%, to analysis units "
            "within buildings. Policy-range bars show wide variation in observed-demand coverage. "
            "Panel g plots demand centre minus campaign target: positive values mean that the demand "
            "centre is warmer than the target, and negative values mean that it is cooler. "
            "Leave-one-source sensitivity preserves the importance of within-building heterogeneity, "
            "while showing that its magnitude depends on weighting and the Griffiths coefficient."
        ),
        2: (
            "Eight-panel portrait figure. A filtering flow yields 63 shared-room windows and 225 "
            "direct votes. A 63-row composition chart shows disagreement in 49 windows, including "
            "six with warmer and cooler requests within the same 30-min room-window. Six diverging "
            "bars detail those within-window opposing requests. A scatter shows that averaging can "
            "only preserve or reduce represented request burden and exactly cancels four non-neutral "
            "windows. Threshold bars show false acceptance but no false rejection. Results persist "
            "across 15-, 30- and 60-min windows and replicate in a second ENTH dataset, although "
            "confidence intervals are wider there."
        ),
        3: (
            "Eight-panel portrait figure. Panel a compares three control families—uniform-average, "
            "personal-centre and personal-range controls—under the same building and operating "
            "permissions. Mean and full-distribution are separate qualification representations "
            "used to score generated actions, not additional controls. Population scenarios differ "
            "in both demand centre and width. A centre-by-spread grid defines independent "
            "perturbations. Energy and carbon response curves change sign across contexts: warmer "
            "demand reduces median annual energy and carbon in Miami and New Delhi but increases "
            "them in Port Angeles and Seattle. Coloured tables above both curves report the matched "
            "denominator at each demand shift without covering the data. Carrier bars show "
            "concurrent shifts among heating electricity, cooling electricity, fans and gas and "
            "state that 59 rows match while 85 remain unmatched. Stacked outcome bars retain "
            "increases, decreases, ties, unstable directions and unmatched cases. Sensitivity bars "
            "reuse that outcome key and show that direction and match availability depend on demand "
            "assumptions, so personalization does not imply universal savings."
        ),
    }
    from render_revised_main_figures import restructure, readable_display_labels
    from thermal_heterogeneity.figures import apply_figure_style, apply_data_axis_style
    DPI = 600
    apply_figure_style()
    PANEL_ROOT = RESULTS / "panel_data"
    figure = build_figure_2(PANEL_ROOT)
    figure._stage7_panel_root = PANEL_ROOT
    figure._stage7_decision_value = RESULTS / "decision_value/MAIN_FIG6G_Q5_REGRET.csv"
    restructure(figure, 2)
    readable_display_labels(figure, 2)
    apply_data_axis_style(figure, map_axes=figure.axes[:2] if 2 == 1 else [])
    save_plot(figure, LABEL, dpi=DPI, exact_pixels=True)


draw_figure_s07()

In [ ]:
def draw_s08():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S08"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    energies = panel("Fig3d.csv")

    carbon = panel("Fig3e.csv")

    fig, a = canvas(CONTEXTS)

    for ax, city in zip(a, CONTEXTS):
        d = energies[
            (energies.context == city) & (energies.matched.astype(str).str.lower() == "true")
        ]
        population_labels = {
            "india_archive_stratum": "India",
            "us_archive_stratum": "United States",
            "multinational_source_balanced": "Pooled",
        }
        for j, (pop, q) in enumerate(d.groupby("population")):
            q = q.groupby("delta_c").delta_energy_mwh.agg(["median", "count"])
            ax.plot(
                q.index,
                q["median"],
                marker=["o", "s", "^"][j],
                lw=1.2,
                color=POPULATION_COLORS[pop],
                label=population_labels.get(pop, pop),
            )
        ax.axhline(0, color="gray", lw=0.7)
        ax.set_xlabel("Centre shift (°C)")
        ax.set_ylabel("Annual energy change (MWh)")

    a[0].legend(frameon=False, fontsize=11)

    manifest.append(
        save(
            fig,
            out,
            5,
            "Figure S5 | Demand-centre energy effects by archive composition. a–d, Median annual site-energy changes relative to the Δ=0 reference among bilaterally resolved selections at archive-derived spread in Miami, New Delhi, Port Angeles and Seattle, respectively. The horizontal coordinate is the applied demand-centre shift. Blue, mauve and green denote the United States, India and multinational archive compositions. Curves have no value where a comparator is unavailable. These compositions are assigned archive scenarios.",
            ["Fig3d.csv"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S05"
    final_stem = "Figure_S08"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S08.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s08()

In [ ]:
# Figure_S09: edit this cell and rerun to update this figure.
def draw_figure_s09():
    LABEL = "Figure_S09"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import json

    from pathlib import Path

    from typing import Callable

    import matplotlib.pyplot as plt

    import numpy as np

    import pandas as pd

    from matplotlib.axes import Axes

    from matplotlib.figure import Figure

    from matplotlib.patches import FancyBboxPatch, Patch, Rectangle

    from thermal_heterogeneity.figures import (
        CONTEXT_COLORS as SHARED_CONTEXT_COLORS,
        DEMAND_CMAP,
        VARIANCE_COLORS,
        PALETTE,
        create_panel_figure,
        data_safe_legend,
        draw_world_land,
        style_panel,
    )

    CONTEXT_ORDER = ["Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019"]

    CONTEXT_LABELS = {
        "Miami": "Miami",
        "NewDelhi": "New Delhi",
        "PortAngeles_small2019": "Port Angeles",
        "Seattle_small2019": "Seattle",
    }

    CONTEXT_LEGEND_LABELS = {
        "Miami": "Miami",
        "NewDelhi": "New Delhi",
        "PortAngeles_small2019": "Port Angeles",
        "Seattle_small2019": "Seattle",
    }

    CONTEXT_COLORS = SHARED_CONTEXT_COLORS

    POPULATION_LABELS = {
        "us_archive_stratum": "United States",
        "multinational_source_balanced": "Multinational",
        "india_archive_stratum": "India",
    }

    OUTCOME_ORDER = [
        "same_direction_decrease",
        "same_direction_increase",
        "all_tied",
        "direction_not_stable",
        "match_lost",
        "all_unmatched",
    ]

    OUTCOME_LABELS = {
        "same_direction_decrease": "Decrease",
        "same_direction_increase": "Increase",
        "all_tied": "Tie",
        "direction_not_stable": "Unstable",
        "match_lost": "Unmatched",
        "all_unmatched": "Unmatched",
    }

    OUTCOME_COLORS = {
        "same_direction_decrease": PALETTE["blue"],
        "same_direction_increase": PALETTE["vermillion"],
        "all_tied": PALETTE["grey"],
        "direction_not_stable": PALETTE["purple"],
        "match_lost": PALETTE["light_grey"],
        "all_unmatched": PALETTE["light_grey"],
    }

    def _read_csv(root: Path, panel_id: str, required: set[str]) -> pd.DataFrame:
        path = root / f"{panel_id}.csv"
        frame = pd.read_csv(path)
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"{panel_id} missing columns: {sorted(missing)}")
        if frame.empty or set(frame["panel_id"]) != {panel_id}:
            raise ValueError(f"{panel_id} is empty or carries the wrong panel_id")
        return frame

    def _read_json(root: Path, filename: str, panel_id: str) -> dict:
        payload = json.loads((root / filename).read_text(encoding="utf-8"))
        if payload.get("panel_id") != panel_id:
            raise ValueError(f"{filename} carries the wrong panel_id")
        return payload

    def _finish_axis(axis: Axes, letter: str, title: str, grid_axis: str | None = "y") -> None:
        style_panel(axis, letter, title, grid_axis=grid_axis)

    def _context_line_summary(frame: pd.DataFrame, value: str) -> pd.DataFrame:
        matched = frame.loc[frame["matched"].astype(str).str.lower().eq("true")].copy()
        matched[value] = pd.to_numeric(matched[value], errors="coerce")
        matched = matched.dropna(subset=[value])
        return (
            matched.groupby(["context", "delta_c"], sort=False)[value]
            .agg(
                median="median",
                q1=lambda series: series.quantile(0.25),
                q3=lambda series: series.quantile(0.75),
                n="size",
            )
            .reset_index()
        )

    def build_figure_3(panel_root: Path | str) -> Figure:
        """Build Figure 3: annual energy/carbon redistribution under three controls."""

        root = Path(panel_root)
        a = _read_json(root, "Fig3a_meta.json", "Fig3a")
        b = _read_csv(root, "Fig3b", {"population", "context", "seed", "n_assignments", "centre_c", "p10_c", "p90_c", "sd_c"})
        c = _read_json(root, "Fig3c_meta.json", "Fig3c")
        d = _read_csv(root, "Fig3d", {"context", "population", "seed", "service_budget", "delta_c", "delta_energy_mwh", "matched"})
        e = _read_csv(root, "Fig3e", {"context", "population", "seed", "service_budget", "delta_c", "delta_co2_t", "matched", "carbon_factor_version"})
        f = _read_csv(root, "Fig3f", {"context", "population", "seed", "service_budget", "delta_c", "matched", "heating_electricity_change_mwh_signed", "cooling_electricity_change_mwh_signed", "fan_electricity_change_mwh_signed", "gas_hhv_change_mwh_signed", "heating_electricity_reference_mwh", "heating_electricity_target_mwh"})
        g = _read_csv(root, "Fig3g", {"pair_id", "context", "alternative_control", "energy_direction", "co2_direction", "matched", "griffiths"})
        h = _read_csv(root, "Fig3h", {"metric", "sensitivity_type", "sensitivity_value", "classification", "count", "denominator"})

        figure, axes = create_panel_figure()
        figure.subplots_adjust(left=0.14, bottom=0.15, wspace=0.52, hspace=0.50)
        ax = axes.ravel()

        # a: common building boundary and three control families.
        ax[0].set_xlim(0, 3)
        ax[0].set_ylim(0, 2.1)
        ax[0].set_xticks([])
        ax[0].set_yticks([])
        ax[0].axis("off")
        labels = [("Uniform average", "one shared\ndemand"), ("Personal centre", "individual\ncentres"), ("Personal range", "individual\nranges")]
        colors = [PALETTE["uniform"], PALETTE["orange"], PALETTE["green"]]
        for index, ((label, sublabel), color) in enumerate(zip(labels, colors)):
            box = FancyBboxPatch((index + 0.08, 0.84), 0.84, 0.72, boxstyle="round,pad=0.03", facecolor=color, alpha=0.20, edgecolor=color, linewidth=1.1)
            ax[0].add_patch(box)
            ax[0].text(index + 0.5, 1.33, label.replace(" ", "\n", 1), ha="center", va="center")
            ax[0].text(index + 0.5, 1.02, sublabel, ha="center", va="center")
            ax[0].annotate("", xy=(index + 0.5, 0.61), xytext=(index + 0.5, 0.83), arrowprops={"arrowstyle": "->", "lw": 0.8, "color": color})
        footer = "Same building and operating permissions"
        common = FancyBboxPatch((0.12, 0.16), 2.76, 0.42, boxstyle="round,pad=0.03", facecolor="#F1F1F1", edgecolor="#777777", linewidth=0.7)
        ax[0].add_patch(common)
        ax[0].text(1.5, 0.37, footer, ha="center", va="center")
        if len(a["items"]) != 3:
            raise ValueError("Fig3a must define exactly three controls")
        _finish_axis(ax[0], "a", "Three control families", None)

        # b: 12 population-context rows; all three seeds remain as thin intervals/points.
        ylabels, ypositions = [], []
        cursor = 0
        population_order = ["us_archive_stratum", "multinational_source_balanced", "india_archive_stratum"]
        for context in CONTEXT_ORDER:
            for population in population_order:
                group = b.loc[(b["context"] == context) & (b["population"] == population)].sort_values("seed")
                if len(group) != 3:
                    raise ValueError("Fig3b requires three frozen seeds per population-context")
                yy = cursor
                for offset, row in zip([-0.15, 0, 0.15], group.itertuples()):
                    ax[1].hlines(yy + offset, row.p10_c, row.p90_c, color=CONTEXT_COLORS[context], linewidth=0.55, alpha=0.55)
                    ax[1].scatter(row.centre_c, yy + offset, s=9, color=CONTEXT_COLORS[context], zorder=3)
                ax[1].hlines(yy, group["p10_c"].median(), group["p90_c"].median(), color=CONTEXT_COLORS[context], linewidth=2.4)
                ax[1].scatter(group["centre_c"].median(), yy, s=25, color=CONTEXT_COLORS[context], edgecolors="white", linewidths=0.4, zorder=4)
                ypositions.append(yy)
                ylabels.append(f"{CONTEXT_LABELS[context]} · {POPULATION_LABELS[population]}")
                cursor += 1
        ax[1].set_yticks([1, 4, 7, 10], ["Miami", "New Delhi", "Port Angeles", "Seattle"])
        ax[1].invert_yaxis()
        ax[1].set_xlim(18, 31)
        ax[1].set_xticks([20, 24, 28])
        ax[1].set_xlabel("Assigned demand temperature (°C)")
        _finish_axis(ax[1], "b", "Assigned population demand", "x")

        # c: the pre-specified 5 x 4 orthogonal grid, shown without simulated outcomes.
        deltas = c["items"][0]["delta_c"]
        lambdas = c["items"][0]["lambda"]
        for yy, lam in enumerate(lambdas):
            for xx, delta in enumerate(deltas):
                length = 0.07 + 0.12 * float(lam)
                ax[2].hlines(yy, xx - length, xx + length, color=PALETTE["blue"], linewidth=1.3 + float(lam))
                face = "white" if (delta == 0 and lam == 1) else PALETTE["orange"]
                ax[2].scatter(xx, yy, s=38, facecolor=face, edgecolor=PALETTE["orange"], linewidth=1.0, zorder=3)
        ax[2].set_xticks(range(len(deltas)), [f"{value:+g}" for value in deltas])
        ax[2].set_yticks(range(len(lambdas)), [f"{value:g}" for value in lambdas])
        ax[2].set_xlabel("Demand-location shift Δ (°C)")
        ax[2].set_ylabel("Spread scale λ")
        ax[2].text(0.98, 0.04, "open = reference (0, 1)", ha="right", va="bottom", transform=ax[2].transAxes)
        _finish_axis(ax[2], "c", "Centre × spread perturbation", "both")

        # d/e: matched medians and IQR only; matched n values use an aligned legend table.
        def plot_context_lines(panel_axis: Axes, frame: pd.DataFrame, value: str, ylabel: str, letter: str, title: str) -> None:
            summary = _context_line_summary(frame, value)
            n_vectors: dict[str, str] = {}
            for context in CONTEXT_ORDER:
                group = summary.loc[summary["context"] == context].sort_values("delta_c")
                xvals = group["delta_c"].to_numpy(float)
                med = group["median"].to_numpy(float)
                q1 = group["q1"].to_numpy(float)
                q3 = group["q3"].to_numpy(float)
                n_vector = "·".join(str(int(value)) for value in group["n"])
                n_vectors[context] = n_vector
                panel_axis.plot(
                    xvals,
                    med,
                    marker="o",
                    linewidth=1.2,
                    color=CONTEXT_COLORS[context],
                    label=CONTEXT_LEGEND_LABELS[context],
                )
                panel_axis.fill_between(xvals, q1, q3, color=CONTEXT_COLORS[context], alpha=0.24)
            panel_axis.axhline(0, color="#333333", linewidth=0.7)
            panel_axis.set_xticks([-2, -1, 1, 2])
            panel_axis.set_xlabel("Demand-centre shift Δ (°C)")
            panel_axis.set_ylabel(ylabel)
            _finish_axis(panel_axis, letter, title, "y")
            lower, original_upper = panel_axis.get_ylim()
            original_range = original_upper - lower
            panel_axis.set_ylim(lower, original_upper + original_range * 1.70)
            header = panel_axis.text(
                0.5,
                0.93,
                "matched n by Δ",
                ha="center",
                va="center",
                transform=panel_axis.transAxes,
            )
            header.set_gid(f"fig3{letter}-n-table")
            table_positions = [(0.02, 0.81), (0.02, 0.69), (0.02, 0.57), (0.02, 0.45)]
            table_labels = ["Miami", "New Delhi", "Port Angeles", "Seattle"]
            for context, label, (x_position, y_position) in zip(
                CONTEXT_ORDER,
                table_labels,
                table_positions,
            ):
                table_text = panel_axis.text(
                    x_position,
                    y_position,
                    f"{label} {n_vectors[context]}",
                    ha="left",
                    va="center",
                    color=CONTEXT_COLORS[context],
                    transform=panel_axis.transAxes,
                )
                table_text.set_gid(f"fig3{letter}-n-table")
            if value == "delta_energy_mwh":
                panel_axis.set_yticks([-50, 0, 50, 100])
            else:
                panel_axis.set_yticks([-25, 0, 25, 50, 75])

        plot_context_lines(ax[3], d, "delta_energy_mwh", "Δ energy (MWh yr⁻¹)", "d", "Annual energy redistribution")
        plot_context_lines(ax[4], e, "delta_co2_t", "Δ CO₂ (t yr⁻¹)", "e", "Annual carbon redistribution")

        # f: median signed changes come directly from amended fields; unmatched rows excluded and counted.
        matched = f.loc[f["matched"].astype(str).str.lower().eq("true")].copy()
        carrier_fields = [
            ("heating_electricity_change_mwh_signed", "Heating", PALETTE["orange"]),
            ("cooling_electricity_change_mwh_signed", "Cooling", PALETTE["blue"]),
            ("fan_electricity_change_mwh_signed", "Fans", PALETTE["green"]),
            ("gas_hhv_change_mwh_signed", "Gas", PALETTE["purple"]),
        ]
        x = np.arange(len(CONTEXT_ORDER))
        pos = np.zeros(len(CONTEXT_ORDER))
        neg = np.zeros(len(CONTEXT_ORDER))
        for field, label, color in carrier_fields:
            values = np.array([matched.loc[matched["context"] == context, field].median() for context in CONTEXT_ORDER])
            bottoms = np.where(values >= 0, pos, neg)
            ax[5].bar(x, values, bottom=bottoms, color=color, width=0.68, label=label)
            pos += np.where(values >= 0, values, 0)
            neg += np.where(values < 0, values, 0)
        energy_summary = _context_line_summary(d.loc[d["delta_c"] == 2], "delta_energy_mwh").set_index("context")
        context_ns: list[int] = []
        for index, context in enumerate(CONTEXT_ORDER):
            total = float(energy_summary.loc[context, "median"])
            n = int((matched["context"] == context).sum())
            context_ns.append(n)
            ax[5].scatter(index, total, marker="D", s=30, facecolor="white", edgecolor="#222222", zorder=4)
        ax[5].axhline(0, color="#333333", linewidth=0.7)
        context_tick_labels = [
            f"{label}\nn={n}"
            for label, n in zip(["Miami", "New\nDelhi", "Port\nAngeles", "Seattle"], context_ns)
        ]
        ax[5].set_xticks(x, context_tick_labels)
        ax[5].set_yticks([-20, -10, 0, 10])
        ax[5].set_ylabel("Median signed change (MWh yr⁻¹)")
        _finish_axis(ax[5], "f", "Carrier redistribution at Δ=+2°C", "y")
        data_safe_legend(ax[5], ncol=2)
        ax[5].set_ylim(-22, 40)
        match_note = ax[5].text(
            0.98,
            0.54,
            "59 matched · 85 unmatched",
            ha="right",
            va="center",
            transform=ax[5].transAxes,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 0.2},
        )
        match_note.set_gid("fig3f-match-note")

        # g: retain every one of the 130 pair definitions, including unmatched cases.
        bars = []
        for alternative in ["personal_center", "ltpref_interval"]:
            for context in CONTEXT_ORDER:
                bars.append((alternative, context))
        counts = g.groupby(["alternative_control", "context", "energy_direction"]).size()
        x = np.arange(len(bars))
        bottoms = np.zeros(len(bars))
        for outcome in OUTCOME_ORDER[:5]:
            values = np.array([counts.get((alternative, context, outcome), 0) for alternative, context in bars])
            short_outcome = {"same_direction_decrease": "Decrease", "same_direction_increase": "Increase", "all_tied": "Tie", "direction_not_stable": "Unstable", "match_lost": "Unmatched"}[outcome]
            ax[6].bar(x, values, bottom=bottoms, color=OUTCOME_COLORS[outcome], width=0.76, label=short_outcome, hatch="//" if outcome == "match_lost" else None)
            for index, (value, bottom) in enumerate(zip(values, bottoms)):
                if value > 0:
                    ax[6].text(index, bottom + value / 2, str(int(value)), ha="center", va="center")
            bottoms += values
        labels = [f"{'Personal centre' if alt == 'personal_center' else 'Personal range'}\n{CONTEXT_LABELS[ctx]}" for alt, ctx in bars]
        ax[6].set_xticks(x, labels, rotation=90, ha="center")
        for label in ax[6].get_xticklabels():
            label.set_linespacing(0.8)
        ax[6].set_ylabel("Paired definitions (count)")
        ax[6].set_ylim(0, max(bottoms) + 20)
        ax[6].legend(
            loc="upper center",
            ncol=3,
            frameon=False,
            bbox_to_anchor=(0.5, 0.99),
            handlelength=0.8,
            columnspacing=0.7,
            handletextpad=0.3,
        )
        _finish_axis(ax[6], "g", "Three-control paired outcomes", "y")

        # h: three frozen sensitivity families, with their explicit denominators.
        sensitivity_order = ["griffiths_energy", "griffiths_carbon", "interval_width_energy"]
        sensitivity_labels = ["g · energy", "g · CO₂", "width · energy"]
        grouped = h.groupby(["sensitivity_type", "classification"], as_index=False).agg(count=("count", "sum"), denominator=("denominator", "first"))
        yy = np.arange(len(sensitivity_order))
        left = np.zeros(len(sensitivity_order))
        for outcome in OUTCOME_ORDER:
            values = []
            for sensitivity in sensitivity_order:
                row = grouped.loc[(grouped["sensitivity_type"] == sensitivity) & (grouped["classification"] == outcome)]
                values.append(0.0 if row.empty else float(row.iloc[0]["count"]) / float(row.iloc[0]["denominator"]) * 100)
            values_array = np.asarray(values)
            ax[7].barh(yy, values_array, left=left, color=OUTCOME_COLORS[outcome], height=0.62, hatch="//" if outcome in {"match_lost", "all_unmatched"} else None)
            left += values_array
        denominators = [int(h.loc[h["sensitivity_type"] == value, "denominator"].iloc[0]) for value in sensitivity_order]
        ax[7].set_yticks([])
        ax[7].invert_yaxis()
        ax[7].set_xlim(-25, 100)
        for y_value, label, denominator in zip(yy, sensitivity_labels, denominators):
            ax[7].text(-1.5, y_value, f"{label}\nn={denominator}", ha="right", va="center")
        ax[7].set_xticks([0, 50, 100])
        ax[7].set_xlabel("Outcome share (%)")
        _finish_axis(ax[7], "h", "Human-demand sensitivity", "x")

        return figure

    CAPTIONS = {
        1: (
            "**Figure 1 | Observed human demand is geographically structured but dominated by "
            "within-building heterogeneity.** **a,** The complete office archive contains 21,418 records "
            "from 12,130 analysis units; the executable set retains 21,399 records from 12,113 "
            "analysis units in 187 buildings, 40 cities and 15 countries. **b,c,** Observed-city warm- "
            "and cold-season demand centres. Filled circles denote season-supported cities, hollow "
            "circles observed but unsupported cities, circle area the number of analysis units and "
            "colour the demand centre on the same 19.5–30.6 °C scale; unsampled geography is left "
            "blank and no spatial interpolation is applied. **d,** Empirical P10–P90 intervals and "
            "medians for city- and source-level warm/cold aggregates under analysis-unit-equal "
            "weighting; labels give the number of aggregates and median interval width. **e,** "
            "Hierarchical variance allocation across country (22.3%), city within country (7.7%), "
            "building within city (5.8%), analysis unit within building (56.1%) and repeated "
            "measurements (8.2%). **f,** Weighted demand records below, within and above nine bilateral "
            "policy or design ranges at Griffiths coefficient 0.50; labels give the within-range "
            "share and eligible-record denominator, and whiskers are nested-bootstrap 95% confidence "
            "intervals for the within-range share. **g,** Empirical P10–P90 (thin), P25–P75 (thick) and "
            "median (point) of demand centre minus campaign target for three prespecified campaigns; "
            "positive values indicate a demand centre warmer than the target, and negative values "
            "indicate a demand centre cooler than the target. **h,** Leave-one-source sensitivity ranges "
            "and medians across 12 coefficient/weighting specifications, each based on 27 source "
            "removals. Policy documents in **f,g** are external temperature coordinates rather than "
            "individual comfort ground truth."
        ),
        2: (
            "**Figure 2 | Independent shared-room votes reveal demand disagreement hidden by "
            "averages.** **a,** Prespecified construction of the NUS 30-min validation cohort from "
            "4,378 raw rows to 63 temperature-homogeneous room-windows containing 225 valid "
            "person-window votes. **b,** Cooler, no-change and warmer vote shares in all 63 "
            "room-windows, ordered by net vote and disagreement; 49 windows contain more than one "
            "response category and six contain warmer and cooler requests within the same 30-min "
            "room-window. **c,** Vote counts in the six room-windows with within-window opposing "
            "requests; grey circles denote no-change votes and labels give vote count, mean room "
            "temperature and within-window temperature span. **d,** Full request burden, mean "
            "absolute vote, versus burden after averaging, absolute mean vote, for each window. "
            "Hollow orange points are exact cancellations; the dashed line is equality and the "
            "median compression gap is zero despite a positive mean gap. **e,** The four exact-"
            "cancellation windows, in which non-zero full burden is represented as zero after "
            "averaging. **f,** False acceptance and false rejection at burden thresholds 0, 0.25, "
            "0.50 and 0.75; all four thresholds retain zero false rejections and 2–5 false "
            "acceptances among 63 windows. **g,** Disagreement, within-window opposing requests and "
            "mean Jensen gap under 15-, 30- and 60-min windows; whiskers are crossed room/date "
            "cluster-bootstrap 95% confidence intervals and n is the eligible-window denominator. "
            "**h,** Independent ENTH replication (34 windows, 81 votes) beside NUS (63 windows, 225 "
            "votes). Windows, rather than votes, are the inferential unit throughout."
        ),
        3: (
            "**Figure 3 | Demand location redistributes annual energy and carbon under three "
            "controls.** **a,** Uniform-average, personal-centre and personal-range control "
            "families generate physical actions in the same building under identical operating "
            "permissions. These controls are distinct from the mean and full-distribution "
            "qualification representations used to score the same actions. **b,** Assigned P10–P90 "
            "demand intervals, centres and seed variation for three population compositions in four "
            "building/climate contexts. **c,** Prespecified orthogonal transformation "
            "T′=μ+Δ+λ(T−μ), with demand-location shift Δ from −2 to +2 °C and spread scale λ from 0 "
            "to 1.5; the open point is the Δ=0, λ=1 reference. **d,e,** Median and interquartile "
            "range of matched service-qualified changes in purchased space-HVAC site energy and "
            "operational CO₂ relative to the reference. The coloured denominator tables give "
            "matched n for Miami, New Delhi, Port Angeles and Seattle in left-to-right Δ order (−2, "
            "−1, +1 and +2 °C). **f,** Median signed target-minus-reference changes in heating "
            "electricity, cooling electricity, fan electricity and gas HHV for Δ=+2 °C, using the "
            "amended reference/target carrier fields directly; diamonds show median total-energy "
            "change and the two-line context labels give 22, 20, 8 and 9 matched cells. The artwork "
            "states the full accounting boundary of 59 matched and 85 unmatched design rows; "
            "unmatched rows contribute no fabricated delta. **g,** Energy direction for all 130 "
            "default uniform-versus-alternative paired definitions, separated by control and "
            "context: 54 increases, 3 decreases, 36 ties, 7 direction-unstable comparisons and 30 "
            "unmatched comparisons. **h,** Outcome shares across Griffiths energy, Griffiths carbon "
            "and personal-range-width sensitivity families with explicit denominators and the same outcome "
            "colour/hatch key as panel g. Population assignments are controlled archive scenarios "
            "rather than claims of local population representativeness; carbon factors are fixed "
            "regional scenarios."
        ),
    }

    ALT_TEXT = {
        1: (
            "Eight-panel portrait figure. An archive flow retains 21,399 of 21,418 records. Two "
            "observed-city maps show 31 warm-season and 26 cold-season supported cities without "
            "interpolation. Ranked empirical intervals show broad within-city and within-source "
            "demand ranges. A variance bar assigns the largest share, 56.1%, to analysis units "
            "within buildings. Policy-range bars show wide variation in observed-demand coverage. "
            "Panel g plots demand centre minus campaign target: positive values mean that the demand "
            "centre is warmer than the target, and negative values mean that it is cooler. "
            "Leave-one-source sensitivity preserves the importance of within-building heterogeneity, "
            "while showing that its magnitude depends on weighting and the Griffiths coefficient."
        ),
        2: (
            "Eight-panel portrait figure. A filtering flow yields 63 shared-room windows and 225 "
            "direct votes. A 63-row composition chart shows disagreement in 49 windows, including "
            "six with warmer and cooler requests within the same 30-min room-window. Six diverging "
            "bars detail those within-window opposing requests. A scatter shows that averaging can "
            "only preserve or reduce represented request burden and exactly cancels four non-neutral "
            "windows. Threshold bars show false acceptance but no false rejection. Results persist "
            "across 15-, 30- and 60-min windows and replicate in a second ENTH dataset, although "
            "confidence intervals are wider there."
        ),
        3: (
            "Eight-panel portrait figure. Panel a compares three control families—uniform-average, "
            "personal-centre and personal-range controls—under the same building and operating "
            "permissions. Mean and full-distribution are separate qualification representations "
            "used to score generated actions, not additional controls. Population scenarios differ "
            "in both demand centre and width. A centre-by-spread grid defines independent "
            "perturbations. Energy and carbon response curves change sign across contexts: warmer "
            "demand reduces median annual energy and carbon in Miami and New Delhi but increases "
            "them in Port Angeles and Seattle. Coloured tables above both curves report the matched "
            "denominator at each demand shift without covering the data. Carrier bars show "
            "concurrent shifts among heating electricity, cooling electricity, fans and gas and "
            "state that 59 rows match while 85 remain unmatched. Stacked outcome bars retain "
            "increases, decreases, ties, unstable directions and unmatched cases. Sensitivity bars "
            "reuse that outcome key and show that direction and match availability depend on demand "
            "assumptions, so personalization does not imply universal savings."
        ),
    }
    from render_revised_main_figures import restructure, readable_display_labels
    from thermal_heterogeneity.figures import apply_figure_style, apply_data_axis_style
    DPI = 600
    apply_figure_style()
    PANEL_ROOT = RESULTS / "panel_data"
    figure = build_figure_3(PANEL_ROOT)
    figure._stage7_panel_root = PANEL_ROOT
    figure._stage7_decision_value = RESULTS / "decision_value/MAIN_FIG6G_Q5_REGRET.csv"
    restructure(figure, 3)
    readable_display_labels(figure, 3)
    apply_data_axis_style(figure, map_axes=figure.axes[:2] if 3 == 1 else [])
    save_plot(figure, LABEL, dpi=DPI, exact_pixels=True)


draw_figure_s09()

In [ ]:
# Figure_S10: edit this cell and rerun to update this figure.
def draw_figure_s10():
    LABEL = "Figure_S10"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import argparse

    import hashlib

    import json

    from pathlib import Path

    import matplotlib

    import matplotlib.pyplot as plt

    from matplotlib.font_manager import FontProperties, findfont

    from matplotlib.lines import Line2D

    from matplotlib.text import Text

    from PIL import Image

    import numpy as np

    import pandas as pd

    from matplotlib.colors import LinearSegmentedColormap

    from matplotlib.ticker import MaxNLocator

    FONT = "Times New Roman"

    WIDTH_MM, HEIGHT_MM = 180, 183

    CONTROL_ORDER = ("uniform_average", "personal_center", "ltpref_interval")

    CONTROL_LABELS = {
        "uniform_average": "Uniform average",
        "personal_center": "Personal centre",
        "ltpref_interval": "Personal range",
    }

    CONTROL_COLORS = {
        "uniform_average": "#7F7F7F",
        "personal_center": "#C8A36A",
        "ltpref_interval": "#7FA995",
    }

    CONTEXT_ORDER = ("Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019")

    CONTEXT_COLORS = dict(zip(CONTEXT_ORDER, ("#C78470", "#AF94AD", "#6F9EBC", "#7FA995")))

    CONTEXT_LABELS = dict(zip(CONTEXT_ORDER, ("Miami", "New Delhi", "Port Angeles", "Seattle")))

    CONTEXT_SHORT = ("Miami", "New\nDelhi", "Port\nAngeles", "Seattle")

    CONTROL_MARKERS = dict(zip(CONTROL_ORDER, ("o", "s", "^")))

    ALTERNATIVES = CONTROL_ORDER[1:]

    DATA_BOUNDS_MM = tuple(
        (x, y, 67.0, 35.0) for y in (125.0, 72.0, 19.0) for x in (19.0, 109.0)
    )

    def configure_style() -> dict[str, str]:
        fonts = {}
        for weight in ("normal", "bold"):
            prop = FontProperties(family=FONT, weight=weight)
            path = Path(findfont(prop, fallback_to_default=False))
            if FontProperties(fname=str(path)).get_name() != FONT:
                raise ValueError(f"Exact {FONT} {weight} is required")
            fonts[weight] = str(path)
        plt.rcParams.update({
            "font.family": FONT, "font.size": 11, "text.color": "black",
            "axes.labelsize": 11, "axes.labelcolor": "black", "axes.labelpad": 6,
            "axes.titlesize": 12, "axes.titleweight": "bold",
            "axes.spines.top": False, "axes.spines.right": False,
            "axes.edgecolor": "#777777", "axes.linewidth": 0.75,
            "xtick.labelsize": 11, "ytick.labelsize": 11,
            "xtick.major.pad": 4, "ytick.major.pad": 4,
            "xtick.color": "black", "ytick.color": "black",
            "legend.fontsize": 11, "legend.frameon": False,
            "figure.facecolor": "white", "savefig.facecolor": "white",
            "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
            "svg.hashsalt": "thermal-heterogeneity-stage8",
        })
        return fonts

    def six_panel_canvas(titles: list[str], *, control_key: bool = True, bounds=None, height=None):
        """Create equal data frames with separate title, axis and shared-key bands."""
        if len(titles) != 6:
            raise ValueError("The main result template has exactly six data panels")
        configure_style()
        page_height = height or HEIGHT_MM
        fig = plt.figure(figsize=(WIDTH_MM / 25.4, page_height / 25.4))
        axes = []
        for index, (title, panel_bounds) in enumerate(zip(titles, bounds or DATA_BOUNDS_MM)):
            x, y, width, height_mm = panel_bounds
            ax = fig.add_axes((x / WIDTH_MM, y / page_height, width / WIDTH_MM, height_mm / page_height))
            ax.set_title(f"({chr(97 + index)}) {title}", loc="left", pad=8)
            ax.grid(False)
            ax.set_axisbelow(True)
            axes.append(ax)
        if control_key:
            handles = [Line2D([], [], marker=marker, linestyle="none", color=CONTROL_COLORS[key],
                              markersize=5, label=CONTROL_LABELS[key])
                       for key, marker in zip(CONTROL_ORDER, ("o", "s", "^"))]
            fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.995),
                       ncol=3, handletextpad=0.3, columnspacing=0.9)
        return fig, axes
    from matplotlib.lines import Line2D
    DPI = 600
    source = pd.read_csv(RESULTS/"Figure_sources/Supplementary_Figure_S21_source.csv")
    fig, axes = six_panel_canvas(["Port Angeles electricity", "Seattle electricity",
        "Port Angeles temperature", "Seattle temperature", "Port Angeles tail deviation",
        "Seattle tail deviation"], control_key=False)
    fig.legend(handles=[Line2D([], [], color=CONTROL_COLORS["uniform_average"], label="Own baseline"),
                        Line2D([], [], color=CONTROL_COLORS["personal_center"], linestyle="--", label="Response")],
               loc="upper center", bbox_to_anchor=(.5,.995), ncol=2)
    for column, context in enumerate(("PortAngeles_small2019","Seattle_small2019")):
        for operation,color,linestyle in (("Own baseline",CONTROL_COLORS["uniform_average"],"-"),
                                         ("Response",CONTROL_COLORS["personal_center"],"--")):
            data = source[source.context.eq(context)&source.operation.eq(operation)].sort_values("hour_after_event_start")
            for index,(metric,label) in enumerate((("hvac_electric_kw","HVAC electricity (kW)"),
                ("occupied_mean_operative_temperature_c","Operative temperature (°C)"),
                ("fixed_tail_reference_mismatch_c","Tail mismatch (°C)"))):
                ax = axes[index*2+column]
                ax.plot(data.hour_after_event_start.to_numpy()+.5,data[metric].to_numpy(),
                        color=color,linestyle=linestyle,linewidth=1.45)
                ax.set_ylabel(label)
    for ax in axes:
        ax.axvspan(0,2,color="#C8A36A",alpha=.2,zorder=0)
        ax.set_xlim(0,50)
        ax.set_xticks([0,24,48])
        ax.set_xlabel("Hours from event start")
    for ax in (axes[2],axes[4]):
        ax.set_xlim(0,6)
        ax.set_xticks([0,2,4,6])
    save_plot(fig,LABEL,dpi=DPI)


draw_figure_s10()

In [ ]:
# Figure_S11: edit this cell and rerun to update this figure.
def draw_figure_s11():
    LABEL = "Figure_S11"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import argparse

    from itertools import combinations

    import json

    from pathlib import Path

    import matplotlib

    import matplotlib.pyplot as plt

    from matplotlib import font_manager

    from matplotlib.font_manager import FontProperties

    from matplotlib.lines import Line2D

    from matplotlib.text import Text

    from matplotlib.ticker import MaxNLocator, StrMethodFormatter

    import numpy as np

    import pandas as pd

    COLORS = {"uniform_average": "#7F7F7F", "personal_center": "#C8A36A", "ltpref_interval": "#7FA995"}

    LABELS = {"uniform_average": "Uniform average", "personal_center": "Personal centre", "ltpref_interval": "Personal range"}

    MARKERS = {"uniform_average": "o", "personal_center": "s", "ltpref_interval": "D"}

    CONTEXTS = [("Miami", "Miami"), ("NewDelhi", "New Delhi"), ("PortAngeles_small2019", "Port Angeles"), ("Seattle_small2019", "Seattle")]

    CLASSES = ["original_maximum", "median", "high90_nonmaximum"]

    CLASS_LABELS = ["Peak load", "Median load", "90th\npercentile"]

    CAPTION = (
        "Supplementary Fig. 13 | July demand response across three source load classes. "
        "Panels show the Miami (a), New Delhi (b), Port Angeles (c) and Seattle (d) representative building contexts "
        "for composition 2026102001, random mixing and zone-level control. The peak date maximizes common-25 "
        "baseline weekday electrical HVAC use from 14:00 to 16:00; added dates are nearest the median and 90th "
        "percentile, excluding the peak. Dates use the source simulation calendar. Faint dashed curves show the "
        "physical response to a +1 °C shift. Solid curves show the largest positive service-qualified response "
        "among +0.5/+1 °C actions, with zero for a qualified baseline lacking a qualified action. Mean and fixed "
        "annual-tail limits apply separately to the event and next 48 hours. Each context has its own vertical scale. "
        "HVAC denotes heating, ventilation and air conditioning."
    )
    DPI = 600
    source = pd.read_csv(RESULTS/'Figure_sources/TYPICAL_DAY_DR_source.csv')
    normal_path = font_manager.findfont(FontProperties(family="Times New Roman", style="normal", weight="normal"), fallback_to_default=False)
    bold_path = font_manager.findfont(FontProperties(family="Times New Roman", style="normal", weight="bold"), fallback_to_default=False)
    normal = FontProperties(fname=normal_path, family="Times New Roman", weight="normal", size=11)
    bold = FontProperties(fname=bold_path, family="Times New Roman", weight="bold", size=12)
    if normal.get_name() != "Times New Roman" or bold.get_name() != "Times New Roman":
        raise ValueError("Exact Times New Roman normal and bold fonts are required")
    plt.rcParams.update({"font.family": "Times New Roman", "font.size": 11, "text.color": "black",
                         "axes.labelcolor": "black", "xtick.color": "black", "ytick.color": "black",
                         "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none", "axes.unicode_minus": False})
    fig = plt.figure(figsize=(180 / 25.4, 174 / 25.4), facecolor="white")
    axes, titles = [], []
    for index, (context, label) in enumerate(CONTEXTS):
        left, bottom = [.145, .635][index % 2], [.565, .18][index // 2]
        ax = fig.add_axes([left, bottom, .31, .25])
        axes.append(ax)
        rows = source[source.context.eq(context)]
        max_value = float(rows[["physical_1c_response_kw", "service_qualified_available_response_kw"]].to_numpy().max())
        ticks = MaxNLocator(nbins=4, min_n_ticks=4).tick_values(0, max_value * 1.12)
        upper = float(ticks[-1])
        ax.set_ylim(-upper * .065, upper)
        ax.set_yticks(ticks[ticks >= 0])
        ax.yaxis.set_major_formatter(StrMethodFormatter("{x:g}"))
        for offset, control in zip([-.14, 0., .14], COLORS):
            subset = rows[rows.control.eq(control)].set_index("date_class").loc[CLASSES]
            x = np.arange(3) + offset
            ax.plot(x, subset.physical_1c_response_kw, linestyle=(0, (3, 2)), linewidth=1.1,
                    color=COLORS[control], alpha=.46, marker=MARKERS[control], markersize=5.2,
                    markerfacecolor="white", markeredgewidth=.9, zorder=2)
            ax.plot(x, subset.service_qualified_available_response_kw, linewidth=1.15, color=COLORS[control],
                    marker=MARKERS[control], markersize=4.5, markeredgewidth=.5, markeredgecolor="white", zorder=3)
        days = rows.drop_duplicates("date_class").set_index("date_class").loc[CLASSES].event_day.astype(int)
        ax.set_xticks(np.arange(3), [f"{label}\n{day} July" for label, day in zip(CLASS_LABELS, days)])
        ax.set_xlim(-.37, 2.37)
        ax.set_ylabel("Two-hour reduction (kW)", fontproperties=normal, labelpad=5)
        ax.axhline(0, color="#B0B0B0", linewidth=.6, zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_linewidth(.75)
        ax.tick_params(axis="both", direction="out", length=3, width=.7, pad=3)
        for text in ax.get_xticklabels() + ax.get_yticklabels():
            text.set_fontproperties(normal)
        title = fig.text(left, bottom + .276, f"{chr(97 + index)}  {label}", ha="left", va="bottom", fontproperties=bold)
        titles.append(title)
    control_handles = [Line2D([], [], color=COLORS[control], marker=MARKERS[control], linewidth=1.1, markersize=4.6,
                              label=LABELS[control]) for control in COLORS]
    control_legend = fig.legend(handles=control_handles, loc="upper center", bbox_to_anchor=(.545, .985),
                               ncol=3, frameon=False, prop=normal, handlelength=1.4, columnspacing=1.25, handletextpad=.45)
    meaning_handles = [Line2D([], [], color="#777777", linestyle=(0, (3, 2)), alpha=.46, marker="o", markerfacecolor="white", label="+1 °C physical"),
                       Line2D([], [], color="#777777", marker="o", label="Service-qualified availability")]
    meaning_legend = fig.legend(handles=meaning_handles, loc="upper center", bbox_to_anchor=(.545, .935), ncol=2,
                               frameon=False, prop=normal, handlelength=1.9, columnspacing=1.4, handletextpad=.5)
    footer = fig.text(.545, .071, "Baseline load class and date", ha="center", va="center", fontproperties=normal)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    text_artists = [text for text in fig.findobj(Text) if text.get_visible() and text.get_text()]
    outside, intersections = [], []
    supported_glyphs = set(font_manager.get_font(normal_path).get_charmap())
    for text in text_artists:
        if any(not char.isspace() and ord(char) not in supported_glyphs for char in text.get_text()):
            raise ValueError("Exact Times New Roman lacks a required text glyph")
        box = text.get_window_extent(renderer)
        if box.x0 < -1 or box.y0 < -1 or box.x1 > fig.bbox.x1 + 1 or box.y1 > fig.bbox.y1 + 1:
            outside.append(text.get_text())
        if text.get_color() != "black" or text.get_fontproperties().get_name() != "Times New Roman":
            raise ValueError("Figure text style differs from the locked font/colour contract")
        if abs(text.get_fontsize() - (12 if text in titles else 11)) > 1e-9:
            raise ValueError("Figure text size differs from the locked 11/12-point contract")
    for first, second in combinations(text_artists, 2):
        a, b = first.get_window_extent(renderer), second.get_window_extent(renderer)
        if a.overlaps(b):
            intersection_width, intersection_height = min(a.x1, b.x1) - max(a.x0, b.x0), min(a.y1, b.y1) - max(a.y0, b.y0)
            if intersection_width > 1.2 and intersection_height > 1.2:
                intersections.append({"first": first.get_text(), "second": second.get_text()})
    if outside or intersections:
        raise ValueError(f"Figure text geometry failed: outside={outside}, intersections={intersections}")

    save_plot(fig,LABEL,dpi=DPI,round_pixels=False)


draw_figure_s11()

In [ ]:
# Figure_S12: edit this cell and rerun to update this figure.
def draw_figure_s12():
    LABEL = "Figure_S12"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import hashlib

    import json

    from pathlib import Path

    from typing import Callable

    import matplotlib.pyplot as plt

    import numpy as np

    import pandas as pd

    from matplotlib.axes import Axes

    from matplotlib.lines import Line2D

    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Polygon, Rectangle

    from thermal_heterogeneity.figures import (
        CATEGORY_STYLES,
        CONTEXT_COLORS,
        COUNT_CMAP,
        PALETTE,
        REPRESENTATION_COLORS,
        SIGNED_CMAP,
        ExportBundle,
        create_panel_figure,
        save_figure_bundle,
        style_panel,
    )

    CONTEXTS = ["Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019"]

    CONTEXT_LABELS = {
        "Miami": "Miami",
        "NewDelhi": "New Delhi",
        "PortAngeles_small2019": "Port Angeles",
        "Seattle_small2019": "Seattle",
    }

    CONTEXT_COLOURS = CONTEXT_COLORS

    LAMBDA_COLOURS = {
        0.0: PALETTE["blue"],
        0.5: PALETTE["sky"],
        1.0: PALETTE["orange"],
        1.5: PALETTE["vermillion"],
    }

    LAMBDA_MARKERS = {0.0: "o", 0.5: "s", 1.0: "^", 1.5: "D"}

    CLASS_COLOURS = {
        "mean-safe": PALETTE["sky"],
        "distribution-required": PALETTE["vermillion"],
        "finite-library-unresolved": PALETTE["light_grey"],
    }

    def _sha256(path: Path) -> str:
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def verify_panel_inputs(panel_directory: Path | str, figure_number: int) -> dict[str, str]:
        """Verify manifest hashes and critical frozen denominators for one figure."""

        panel_directory = Path(panel_directory)
        manifest_path = panel_directory / "PANEL_DATA_V2_MANIFEST.csv"
        manifest = pd.read_csv(manifest_path)
        selected = manifest[manifest["panel_id"].str.startswith(f"Fig{figure_number}")]
        if len(selected) != 8:
            raise ValueError(f"Figure {figure_number} requires exactly eight manifest rows")
        hashes: dict[str, str] = {}
        for row in selected.itertuples(index=False):
            path = panel_directory / row.relative_path
            observed = _sha256(path)
            if observed != row.sha256:
                raise ValueError(f"panel hash mismatch: {path.name}")
            hashes[row.panel_id] = observed

        if figure_number == 4:
            counts = pd.read_csv(panel_directory / "Fig4c.csv")
            full = counts[counts["representation"] == "full"].sort_values("lambda")
            mean = counts[counts["representation"] == "mean"].sort_values("lambda")
            if full["selected_count"].tolist() != [128, 120, 74, 21]:
                raise ValueError("Fig4c full-demand counts differ from the freeze")
            if mean["selected_count"].tolist() != [128, 128, 128, 128]:
                raise ValueError("Fig4c mean-demand counts differ from the freeze")
        elif figure_number == 5:
            case = json.loads((panel_directory / "Fig5bc_case_lock.json").read_text(encoding="utf-8"))
            if not case["mean_full_same_action"]:
                raise ValueError("Fig5b/c must show the same frozen action")
            boundary = pd.read_csv(panel_directory / "Fig5h.csv")
            feb = boundary[boundary["month"] == 2]
            if set(feb["reference_positive_selected_count"]) != {0}:
                raise ValueError("February central reference must remain 0/144")
            if set(feb["month_grid_positive_selected_count"]) != {210}:
                raise ValueError("February full grid must remain 210/2880")
        elif figure_number == 6:
            phase = pd.read_csv(panel_directory / "Fig6c.csv").set_index("classification")
            expected = {
                "mean-safe": 275,
                "distribution-required": 331,
                "finite-library-unresolved": 402,
            }
            if phase["count"].to_dict() != expected:
                raise ValueError("Fig6c phase counts differ from the freeze")
            gate = pd.read_csv(panel_directory / "Fig6f.csv")
            retained = gate["true_positive_candidates"] / gate["full_eligible_candidates"]
            covered = gate["full_available_and_certified_decisions"] / gate["full_available_decisions"]
            if not np.allclose(retained, gate["true_action_retention"], rtol=0, atol=1e-12):
                raise ValueError("Fig6f action-retention denominators do not reproduce")
            if not np.allclose(covered, gate["decision_coverage"], rtol=0, atol=1e-12):
                raise ValueError("Fig6f decision-coverage denominators do not reproduce")
        else:
            raise ValueError("only Figures 4, 5 and 6 are implemented here")
        return hashes

    def _panel(axis: Axes, letter: str, title: str, *, grid_axis: str | None = "y") -> None:
        style_panel(axis, letter, title, grid_axis=grid_axis)

    def _blank(axis: Axes, letter: str, title: str) -> None:
        _panel(axis, letter, title, grid_axis=None)
        axis.set_xlim(0, 1)
        axis.set_ylim(0, 1)
        axis.set_xticks([])
        axis.set_yticks([])
        for spine in axis.spines.values():
            spine.set_visible(False)

    def _child(parent: Axes, bounds: list[float]) -> Axes:
        axis = parent.inset_axes(bounds)
        axis.tick_params(width=0.65, length=2.5, pad=1, color="#333333")
        for side in ("top", "right"):
            axis.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            axis.spines[side].set_linewidth(0.65)
            axis.spines[side].set_color("#333333")
        axis.grid(True, axis="y", color="#D9D9D9", linewidth=0.35, zorder=0)
        return axis

    def _parent_for_children(axis: Axes, letter: str, title: str) -> None:
        _blank(axis, letter, title)

    def _critical(text, name: str):
        """Tag a layout-critical text artist for figure-specific geometry tests."""

        text.set_gid(f"figures-4-6-layout-critical:{name}")
        return text

    def _inset_panel_axis(
        axis: Axes,
        *,
        left: float = 0.0,
        right: float = 0.0,
        bottom: float = 0.0,
        top: float = 0.0,
    ) -> None:
        """Move one top-level data rectangle within its locked panel cell."""

        position = axis.get_position()
        axis.set_position(
            [
                position.x0 + left,
                position.y0 + bottom,
                position.width - left - right,
                position.height - bottom - top,
            ]
        )

    def _event_span(axis: Axes) -> None:
        axis.axvspan(0, 2, color=PALETTE["orange"], alpha=0.16, linewidth=0)
        axis.axvspan(2, 50, color=PALETTE["sky"], alpha=0.08, linewidth=0)

    def build_figure_5(panel_directory: Path | str):
        """Build Figure 5: event flexibility under service and recovery constraints."""

        panel_directory = Path(panel_directory)
        verify_panel_inputs(panel_directory, 5)
        figure, axes = create_panel_figure()
        a, b, c, d, e, f, g, h = axes.ravel()

        # a: frozen event and recovery contract.
        _blank(a, "a", "Event and recovery contract")
        a.plot([0.08, 0.93], [0.60, 0.60], color="#555555", linewidth=1.1)
        a.add_patch(Rectangle((0.10, 0.48), 0.16, 0.24, facecolor=PALETTE["orange"], alpha=0.35, edgecolor="none"))
        a.add_patch(Rectangle((0.26, 0.48), 0.64, 0.24, facecolor=PALETTE["sky"], alpha=0.18, edgecolor="none"))
        a.text(0.18, 0.60, "2-h event", ha="center", va="center")
        a.text(0.58, 0.60, "48-h recovery", ha="center", va="center")
        a.text(0.10, 0.37, "0 h", ha="center")
        a.text(0.26, 0.37, "2 h", ha="center")
        a.text(0.90, 0.37, "50 h", ha="center")
        a.text(0.18, 0.84, "response", ha="center", color=PALETTE["orange"])
        a.text(0.58, 0.84, "rebound + recovery service", ha="center", color=PALETTE["blue"])
        a.text(0.50, 0.17, "50-h net energy = event saving - recovery extra", ha="center")

        # b: same action baseline and perturbed power traces.
        power = pd.read_csv(panel_directory / "Fig5b_power_trace.csv")
        for state, color, linestyle in [("baseline", PALETTE["dark_grey"], "--"), ("perturbed", PALETTE["blue"], "-")]:
            rows = power[power["state"] == state].sort_values("hour")
            b.plot(rows["hour"], rows["hvac_electric_power_kw"], color=color, linestyle=linestyle, label=state.capitalize())
        _event_span(b)
        base = power[power["state"] == "baseline"].sort_values("hour")
        pert = power[power["state"] == "perturbed"].sort_values("hour")
        difference = base["hvac_electric_power_kw"].to_numpy() - pert["hvac_electric_power_kw"].to_numpy()
        response_kw = float(difference[:2].mean())
        rebound_kw = float((-difference[2:]).max())
        net_kwh = float(difference.sum())
        b.text(0.98, 0.96, f"same action\nresponse {response_kw:.1f} kW\nrebound {rebound_kw:.1f} kW\nnet {net_kwh:.1f} kWh", transform=b.transAxes, ha="right", va="top")
        b.set_xlim(0, 49)
        b.set_ylim(0, max(power["hvac_electric_power_kw"]) * 1.18)
        b.set_xlabel("Hour from event start")
        b.set_ylabel("HVAC power (kW)")
        _panel(b, "b", "Fixed-action power trajectory")
        b.legend(frameon=False, loc="lower right", ncol=2, handlelength=1.5)
        _inset_panel_axis(b, left=0.030, bottom=0.010)

        # c: temperature envelope and service trajectories on aligned child axes.
        trace = pd.read_csv(panel_directory / "Fig5c_temperature_service_trace.csv")
        _parent_for_children(c, "c", "Temperature and service")
        temp_ax = _child(c, [0.18, 0.55, 0.79, 0.31])
        service_ax = _child(c, [0.18, 0.12, 0.79, 0.31])
        for state, color, linestyle in [("baseline", PALETTE["dark_grey"], "--"), ("perturbed", PALETTE["blue"], "-")]:
            rows = trace[trace["state"] == state].sort_values("hour")
            temp_ax.fill_between(rows["hour"], rows["zone_min_c"], rows["zone_max_c"], color=color, alpha=0.10, linewidth=0)
            temp_ax.plot(rows["hour"], rows["occupancy_weighted_temp_c"], color=color, linestyle=linestyle)
            service_ax.plot(rows["hour"], rows["full_mean_service"], color=color, linestyle=linestyle)
            service_ax.plot(rows["hour"], rows["fixed27_tail_service"], color=color, linestyle=":")
        mean_budget = float(trace["mean_service_budget_c"].iloc[0])
        tail_budget = float(trace["tail_service_budget_c"].iloc[0])
        service_ax.axhline(mean_budget, color=PALETTE["dark_grey"], linewidth=0.65)
        service_ax.axhline(tail_budget, color=PALETTE["vermillion"], linewidth=0.65)
        for child in (temp_ax, service_ax):
            _event_span(child)
            child.set_xlim(0, 49)
            child.set_xticks([0, 10, 20, 30, 40])
        temp_ax.set_yticks([23, 25, 27])
        service_ax.set_yticks([2, 4, 6])
        temp_ax.set_ylabel("Temp. (°C)")
        temp_ax.set_xticklabels([])
        service_ax.set_ylabel("Service (°C)")
        service_ax.set_xlabel("Hour from event start")
        _critical(temp_ax.text(0.98, 0.93, "line: occupied mean\nshade: zone range", transform=temp_ax.transAxes, ha="right", va="top"), "5c-temperature-key")
        _critical(service_ax.text(0.98, 0.93, "mean service; tail dotted", transform=service_ax.transAxes, ha="right", va="top"), "5c-service-key")

        # d/e: population-level qualification and false acceptance.
        july = pd.read_csv(panel_directory / "Fig5d.csv")
        for rep, label in [("full", "Full demand"), ("mean", "Mean")]:
            rows = july[july["representation"] == rep].sort_values("lambda")
            style = CATEGORY_STYLES["full_demand" if rep == "full" else "mean"]
            d.plot(rows["lambda"], rows["positive_selected_count"], label=label, **style)
            for row in rows.to_dict("records"):
                offset: int | tuple[int, int] = -15 if rep == "full" and row["lambda"] <= 0.5 else (7 if rep == "full" else 12)
                horizontal_alignment = "center"
                if rep == "mean" and row["lambda"] <= 0.5:
                    offset = 13
                if rep == "full" and row["lambda"] == 1.5:
                    offset = (0, 11)
                xytext = (0, offset) if isinstance(offset, int) else offset
                _critical(d.annotate(f"{row['positive_selected_count']}/144", (row["lambda"], row["positive_selected_count"]), xytext=xytext, textcoords="offset points", ha=horizontal_alignment, va="center", clip_on=True), f"5d-{rep}-{row['lambda']}")
        d.set_xlim(-0.20, 1.70)
        d.set_ylim(0, 220)
        d.set_xticks([0, 0.5, 1, 1.5])
        d.set_xlabel("Demand spread λ")
        d.set_ylabel("Positive selections (count)")
        _panel(d, "d", "July response availability")
        legend = d.legend(
            frameon=False,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.98),
            ncol=2,
            handlelength=1.5,
        )
        legend.set_gid("figure-5d-legend")
        d.set_yticks([0, 50, 100, 150])
        _inset_panel_axis(d, left=0.035, bottom=0.008)

        july_false = pd.read_csv(panel_directory / "Fig5e.csv").sort_values("lambda")
        bars = e.bar(july_false["lambda"], july_false["false_accept_count"], width=0.28, color=PALETTE["vermillion"], edgecolor="white")
        for bar, row in zip(bars, july_false.itertuples(index=False)):
            e.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 4, f"{row.false_accept_count}/122\n{100*row.false_accept_rate:.0f}%", ha="center", va="bottom")
        e.set_xlim(-0.25, 1.75)
        e.set_ylim(0, 140)
        e.set_xticks([0, 0.5, 1, 1.5])
        e.text(0.98, 0.04, "λ", transform=e.transAxes, ha="right", va="bottom")
        e.set_ylabel("False accepts (count)")
        _panel(e, "e", "False flexibility acceptance")

        # f: paired response changes in the 12 common cells.
        paired = pd.read_csv(panel_directory / "Fig5f.csv")
        wide = paired.pivot(index="decision_id", columns="lambda", values="selected_response_kw")
        for _, row in wide.sort_values(1.0).iterrows():
            color = PALETTE["blue"] if row[1.5] < row[1.0] - 1e-9 else PALETTE["grey"]
            f.plot([1, 1.5], [row[1.0], row[1.5]], color=color, alpha=0.75, linewidth=1.0)
            f.scatter([1, 1.5], [row[1.0], row[1.5]], color=color, s=14)
        deltas = wide[1.5] - wide[1.0]
        f.text(0.04, 0.96, f"5 decrease; 7 tie\nmax decrease {abs(deltas.min()):.3f} kW", transform=f.transAxes, ha="left", va="top")
        f.set_xlim(0.88, 1.62)
        f.set_xticks([1, 1.5], ["λ=1", "λ=1.5"])
        f.set_ylabel("Response (kW)")
        _panel(f, "f", "Response in common cells")
        _inset_panel_axis(f, left=0.025, bottom=0.005)

        # g: four recovery/energy endpoints in aligned microplots.
        recovery = pd.read_csv(panel_directory / "Fig5g.csv")
        _parent_for_children(g, "g", "Recovery and net energy")
        metrics = [
            ("delta_response_kw", "Response (kW)"),
            ("delta_recovery_peak_kw", "Recovery peak (kW)"),
            ("delta_recovery_extra_kwh", "Recovery extra (kWh)"),
            ("delta_50h_net_saving_kwh", "50-h net (kWh)"),
        ]
        bounds = [[0.16, 0.55, 0.32, 0.15], [0.61, 0.55, 0.36, 0.15], [0.16, 0.07, 0.32, 0.15], [0.61, 0.07, 0.36, 0.15]]
        header_positions = [(0.32, 0.90), (0.79, 0.90), (0.32, 0.40), (0.79, 0.40)]
        for metric_index, ((metric, label), bound) in enumerate(zip(metrics, bounds)):
            child = _child(g, bound)
            values = recovery[metric].to_numpy(float)
            yy = np.linspace(-0.12, 0.12, len(values))
            child.axvline(0, color="#333333", linewidth=0.65)
            child.scatter(values, yy, color=np.where(values < -1e-9, PALETTE["blue"], np.where(values > 1e-9, PALETTE["vermillion"], PALETTE["grey"])), s=13)
            child.set_yticks([])
            child.set_ylim(-0.25, 0.25)
            limit = max(1.0, float(np.max(np.abs(values))))
            tick_limit = float(np.ceil(limit / 5) * 5)
            child.set_xlim(-1.15 * tick_limit, 1.15 * tick_limit)
            child.set_xticks([-tick_limit, 0, tick_limit], [f"{-tick_limit:g}", "0", f"{tick_limit:g}"])
            child.set_xlabel("")
            lower = int((values < -1e-9).sum())
            tied = int((np.abs(values) <= 1e-9).sum())
            higher = int((values > 1e-9).sum())
            hx, hy = header_positions[metric_index]
            _critical(g.text(hx, hy, f"{label}\n{lower}− | {tied}= | {higher}+", ha="center", va="top"), f"5g-header-{metric_index}")
            child.grid(True, axis="x", color="#D9D9D9", linewidth=0.35)

        # h: July and February counterfactual grids.
        boundary = pd.read_csv(panel_directory / "Fig5h.csv")
        _parent_for_children(h, "h", "Season-location boundary")
        heat_bounds = [[0.05, 0.44, 0.41, 0.42], [0.56, 0.44, 0.41, 0.42]]
        vmax = int(boundary["full_positive_selected_count"].max())
        for month, bound, label in [(7, heat_bounds[0], "July"), (2, heat_bounds[1], "February")]:
            child = _child(h, bound)
            sub = boundary[boundary["month"] == month].pivot(index="lambda", columns="delta_c", values="full_positive_selected_count").sort_index().sort_index(axis=1)
            child.imshow(sub.to_numpy(), cmap=COUNT_CMAP, vmin=0, vmax=vmax, origin="lower", aspect="auto")
            child.set_xticks(range(len(sub.columns)), [f"{x:g}" for x in sub.columns])
            child.set_yticks(range(len(sub.index)), [f"{x:g}" for x in sub.index])
            child.text(0.03, 0.97, label, transform=child.transAxes, va="top", color="white")
            child.grid(False)
        h.text(0.50, 0.28, "Demand shift (°C); rows are λ", ha="center", va="center")
        h.text(0.50, 0.18, "July: ref 51/144; grid 1,247/2,880", ha="center", va="center")
        h.text(0.50, 0.02, "February: ref 0/144; grid 210/2,880", ha="center", va="bottom", bbox={"facecolor": "white", "edgecolor": "#555555", "pad": 1})
        return figure

    CAPTIONS = {
        4: (
            "Figure 4 | Demand spread contracts service-qualified annual action space. "
            "a, The same fixed action is audited under a compressed mean and the complete demand distribution. "
            "b, Fixed-centre spread perturbations for the pre-specified Miami multinational composition; horizontal segments show P10-P90. "
            "c, Full-demand and mean-based annual selections among 144 design cells at each spread multiplier. "
            "d, Mean-selected actions that fail full-demand review (denominator 128). "
            "e, Full-service eligible-action counts across all 144 cells per multiplier, retaining zero-action cells. "
            "f, Median and interquartile annual-energy change for bilateral full-service selections; the right-hand table gives matched/unmatched cells for λ=0, 0.5 and 1.5. "
            "g, Carbon intervals and median signed carrier changes for the same bilateral comparisons; colours and markers retain the λ encoding from panel f. "
            "h, Median non-additive eligibility interaction across contexts on one shared diverging action-count scale; selected-energy interactions are resolved in 585 of 1,728 cells."
        ),
        5: (
            "Figure 5 | Demand heterogeneity contracts service-qualified flexibility under event and recovery constraints. "
            "a, Pre-specified 2-h event, 48-h recovery and 50-h energy-accounting contract. "
            "b, Baseline and perturbed HVAC power for the same pre-specified action; the panel does not compare mean- and full-selected actions. "
            "c, Zone temperature envelope, occupancy-weighted temperature and full-distribution mean and fixed-27 tail-service trajectories for that case; horizontal lines are the fixed mean- and tail-service budgets. "
            "d, July cells with a service-qualified positive response under full and mean demand. "
            "e, Mean-positive selections that fail full-demand review (denominator 122). "
            "f, Selected response for 12 common full-service cells at spread multipliers 1 and 1.5. "
            "g, Response, recovery peak, recovery electricity and 50-h net-saving changes for those 12 cells; each header reports decrease, tie and increase counts. "
            "h, Positive full-service selections across demand location and spread; the February central reference (0/144) is distinct from the complete February grid (210/2,880)."
        ),
        6: (
            "Figure 6 | Protected anonymous demand summaries recover service-qualified decisions beyond a single mean. "
            "a, Schematic conditions governing whether an average is sufficient; the vertical axis is the service-budget multiplier, for which larger values permit looser service-error allowances, and the drawn regions do not define an empirical monotonic boundary. "
            "b, Exact sampled design cells classified across four contexts without spatial interpolation. "
            "c, Counts among all 1,008 design units and the 606 resolvable units. "
            "d, False-safe and false-required errors from the lowest-error 1.059 °C in-sample scanned diagnostic threshold; it did not meet the prespecified total-error, worst-context-error, sensitivity and specificity standards, and denominators are context-specific. "
            "e, Median and interquartile service-reconstruction errors for mean, three-quantile and five-quantile summaries. "
            "f, Held-out certification trade-off; marker shape identifies the anonymous summary and marker size encodes decision coverage. The highlighted q5 plus training-only pooled-q95 rule has zero observed false acceptance, retains 14,709/18,766 true-qualified candidates and covers 542/606 resolvable decisions in the context-seed split. "
            "g, Local-update learning curves; points and error bars show updated scores and their hierarchical-bootstrap 95% intervals. In the paired improvement analysis at 40 observations, both city-held-out improvement intervals are above zero, whereas both source-held-out improvement intervals cross zero. "
            "h, Anonymous-summary workflow from archive prior to qualified energy, carbon and flexibility reporting; anonymity of the summaries is not a formal privacy guarantee."
        ),
    }

    ALT_TEXT = {
        4: (
            "Eight-panel figure comparing one fixed annual action library under mean and full thermal-demand service checks. "
            "At fixed mean demand, increasing spread λ contracts full-qualified selections from 128/144 to 21/144 while mean-based selections stay 128/144; false acceptance rises from 0/128 to 128/128. "
            "Eligible action counts collapse, and matched annual-energy and carbon effects differ by context and carrier. The matched/unmatched comparison contains 169/263 cells. "
            "Centre×spread eligibility interactions are non-zero in 1,261/1,728 cells; energy interactions are resolved for 585/1,728 cells, with 1,143 unresolved."
        ),
        5: (
            "Eight-panel figure comparing service-qualified demand-response flexibility with a pre-specified event, recovery and energy-accounting contract. "
            "Panels b and c show baseline and perturbed physical trajectories for the same pre-specified action, with 28.3 kW response, 13.1 kW rebound and 30.3 kWh net saving; they do not compare mean- and full-selected actions. "
            "Across July design cells, full-qualified positive selections fall from 122/144 to 12/144 as λ rises, while mean-based selections remain 122/144 and false acceptance reaches 115/122. "
            "Only 12 cells are common at λ=1 and 1.5; five lose response. February has 0/144 positive selections at the central reference but 210/2,880 across the complete grid."
        ),
        6: (
            "Eight-panel figure testing anonymous demand summaries and a protected qualification rule. "
            "Panel a is a schematic separation of mean-safe, distribution-required and unresolved regimes by demand heterogeneity and service-budget multiplier; larger multipliers permit looser service-error allowances, and the drawn regions are not an empirical monotonic boundary. "
            "Panel b maps the three regimes for the exact sampled cells in Miami, New Delhi, Port Angeles and Seattle. "
            "Panel c shows that, of 1,008 design units, 275 are mean-safe, 331 require the demand distribution and 402 are unresolved; the resolved denominator is 606. "
            "Panel d shows that the lowest-error 1.059 °C in-sample scanned diagnostic threshold makes 92/606 errors and does not meet the prespecified performance standards. "
            "Panel e shows lower median tail-service and eligibility errors for three- and five-quantile summaries than for the mean alone. "
            "Panel f shows that, in the context-seed holdout, q5 plus a training-only pooled-q95 margin yields zero observed false accepts, retains 14,709/18,766 true-qualified candidates and covers 542/606 resolvable decisions; this is held-out evidence, not a universal guarantee. "
            "Panel g shows local-feedback learning curves for dimensionless normalized ordinal RPS and tail-share MAE. Their points and error bars are updated scores and their hierarchical-bootstrap 95% intervals. In the paired improvement analysis at 40 city-held-out observations, the RPS and tail-share error improvement intervals are above zero, whereas at 40 source-held-out observations both corresponding improvement intervals cross zero, including the tail-share error interval. "
            "Panel h traces an anonymous-summary qualified-reporting workflow from an archive prior through local updating, five anonymous quantiles and a protected q95 qualification rule to energy, carbon, flexibility and recovery reporting; it makes no formal privacy guarantee."
        ),
    }
    from render_revised_main_figures import restructure, readable_display_labels
    from thermal_heterogeneity.figures import apply_figure_style, apply_data_axis_style
    DPI = 600
    apply_figure_style()
    PANEL_ROOT = RESULTS / "panel_data"
    figure = build_figure_5(PANEL_ROOT)
    figure._stage7_panel_root = PANEL_ROOT
    figure._stage7_decision_value = RESULTS / "decision_value/MAIN_FIG6G_Q5_REGRET.csv"
    restructure(figure, 5)
    readable_display_labels(figure, 5)
    apply_data_axis_style(figure, map_axes=figure.axes[:2] if 5 == 1 else [])
    save_plot(figure, LABEL, dpi=DPI, exact_pixels=True)


draw_figure_s12()

In [ ]:
def draw_s13():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S13"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    qt = read("expected/quantile_certification_gate_reference.csv")

    fig, a = canvas(
        [
            "Context–seed holdout",
            "Population holdout",
            "Qualification margins",
            "Protection energy penalty",
        ]
    )

    for ax, (scheme, d) in zip(a[:2], qt.groupby("split_scheme", sort=True)):
        for j, (rep, q) in enumerate(d.groupby("representation")):
            ax.scatter(
                q.retained_full_eligible_fraction * 100,
                q.false_accept_candidates + 1,
                color=REPRESENTATION_COLORS[rep],
                marker=["o", "s", "^"][j],
                s=35,
                label={"mean_only": "Mean only", "quantile3": "Three bins", "quantile5": "Five bins"}[rep],
            )
        ax.set_yscale("log")
        ax.set_xlabel("Qualified actions retained (%)")
        ax.set_ylabel("False accepts + 1 (log)")
        ax.set_xlim(-3, 103)
        ax.set_title(
            f"({chr(97+a.index(ax))}) "
            + scheme.replace("leave_one_", "").replace("_out", "").replace("_", " ").capitalize(),
            loc="left",
            pad=18,
        )

    a[0].legend(frameon=False)

    margins = read("expected/quantile_fold_margins_reference.csv")

    q = margins[
        margins.representation.astype(str).str.contains("5")
        & margins.strategy.astype(str).str.contains("95")
    ]

    if q.empty:
        q = margins[margins.representation.astype(str).str.contains("5")]

    box(
        a[2],
        [q.mean_margin_c.values, q.tail_margin_c.values],
        ["Mean", "Tail"],
        "Training-derived margin (°C)",
        colors=[PALETTE["dark_grey"], PALETTE["vermillion"]],
    )

    q = qt[qt.representation.astype(str).str.contains("5")]

    a[3].scatter(
        q.retained_full_eligible_fraction * 100,
        q.energy_penalty_median_kwh / 1000,
        s=35,
        c=REPRESENTATION_COLORS["quantile5"],
    )

    a[3].set_xlabel("Qualified actions retained (%)")

    a[3].set_ylabel("Median energy penalty (MWh)")

    manifest.append(
        save(
            fig,
            out,
            8,
            "Figure S8 | Anonymous-summary rule comparisons. a,b, Mean-only, three-bin and five-bin summaries under all tested protection strategies for context–seed and population holdout, respectively. Grey/gold/green identify the summaries. Three-bin and five-bin summaries use at most three and five near-equal-count bins per zone with bin means, bin counts and fixed-tail membership counts. One is added to false-accept counts for logarithmic display. c, Training-fold mean and tail margins for five-bin summaries with 95th-percentile protection. Boxes show medians and interquartile ranges, with whiskers at 1.5 interquartile ranges. d, Median selection-energy penalty versus qualified-action retention for the tested five-bin protection strategies.",
            [
                "expected/quantile_certification_gate_reference.csv",
                "expected/quantile_fold_margins_reference.csv",
            ],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S08"
    final_stem = "Figure_S13"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S13.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s13()

In [ ]:
# Figure_S14: edit this cell and rerun to update this figure.
def draw_figure_s14():
    LABEL = "Figure_S14"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import hashlib

    import json

    from pathlib import Path

    from typing import Callable

    import matplotlib.pyplot as plt

    import numpy as np

    import pandas as pd

    from matplotlib.axes import Axes

    from matplotlib.lines import Line2D

    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Polygon, Rectangle

    from thermal_heterogeneity.figures import (
        CATEGORY_STYLES,
        CONTEXT_COLORS,
        COUNT_CMAP,
        PALETTE,
        REPRESENTATION_COLORS,
        SIGNED_CMAP,
        ExportBundle,
        create_panel_figure,
        save_figure_bundle,
        style_panel,
    )

    CONTEXTS = ["Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019"]

    CONTEXT_LABELS = {
        "Miami": "Miami",
        "NewDelhi": "New Delhi",
        "PortAngeles_small2019": "Port Angeles",
        "Seattle_small2019": "Seattle",
    }

    CONTEXT_COLOURS = CONTEXT_COLORS

    LAMBDA_COLOURS = {
        0.0: PALETTE["blue"],
        0.5: PALETTE["sky"],
        1.0: PALETTE["orange"],
        1.5: PALETTE["vermillion"],
    }

    LAMBDA_MARKERS = {0.0: "o", 0.5: "s", 1.0: "^", 1.5: "D"}

    CLASS_COLOURS = {
        "mean-safe": PALETTE["sky"],
        "distribution-required": PALETTE["vermillion"],
        "finite-library-unresolved": PALETTE["light_grey"],
    }

    def _sha256(path: Path) -> str:
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def verify_panel_inputs(panel_directory: Path | str, figure_number: int) -> dict[str, str]:
        """Verify manifest hashes and critical frozen denominators for one figure."""

        panel_directory = Path(panel_directory)
        manifest_path = panel_directory / "PANEL_DATA_V2_MANIFEST.csv"
        manifest = pd.read_csv(manifest_path)
        selected = manifest[manifest["panel_id"].str.startswith(f"Fig{figure_number}")]
        if len(selected) != 8:
            raise ValueError(f"Figure {figure_number} requires exactly eight manifest rows")
        hashes: dict[str, str] = {}
        for row in selected.itertuples(index=False):
            path = panel_directory / row.relative_path
            observed = _sha256(path)
            if observed != row.sha256:
                raise ValueError(f"panel hash mismatch: {path.name}")
            hashes[row.panel_id] = observed

        if figure_number == 4:
            counts = pd.read_csv(panel_directory / "Fig4c.csv")
            full = counts[counts["representation"] == "full"].sort_values("lambda")
            mean = counts[counts["representation"] == "mean"].sort_values("lambda")
            if full["selected_count"].tolist() != [128, 120, 74, 21]:
                raise ValueError("Fig4c full-demand counts differ from the freeze")
            if mean["selected_count"].tolist() != [128, 128, 128, 128]:
                raise ValueError("Fig4c mean-demand counts differ from the freeze")
        elif figure_number == 5:
            case = json.loads((panel_directory / "Fig5bc_case_lock.json").read_text(encoding="utf-8"))
            if not case["mean_full_same_action"]:
                raise ValueError("Fig5b/c must show the same frozen action")
            boundary = pd.read_csv(panel_directory / "Fig5h.csv")
            feb = boundary[boundary["month"] == 2]
            if set(feb["reference_positive_selected_count"]) != {0}:
                raise ValueError("February central reference must remain 0/144")
            if set(feb["month_grid_positive_selected_count"]) != {210}:
                raise ValueError("February full grid must remain 210/2880")
        elif figure_number == 6:
            phase = pd.read_csv(panel_directory / "Fig6c.csv").set_index("classification")
            expected = {
                "mean-safe": 275,
                "distribution-required": 331,
                "finite-library-unresolved": 402,
            }
            if phase["count"].to_dict() != expected:
                raise ValueError("Fig6c phase counts differ from the freeze")
            gate = pd.read_csv(panel_directory / "Fig6f.csv")
            retained = gate["true_positive_candidates"] / gate["full_eligible_candidates"]
            covered = gate["full_available_and_certified_decisions"] / gate["full_available_decisions"]
            if not np.allclose(retained, gate["true_action_retention"], rtol=0, atol=1e-12):
                raise ValueError("Fig6f action-retention denominators do not reproduce")
            if not np.allclose(covered, gate["decision_coverage"], rtol=0, atol=1e-12):
                raise ValueError("Fig6f decision-coverage denominators do not reproduce")
        else:
            raise ValueError("only Figures 4, 5 and 6 are implemented here")
        return hashes

    def _panel(axis: Axes, letter: str, title: str, *, grid_axis: str | None = "y") -> None:
        style_panel(axis, letter, title, grid_axis=grid_axis)

    def _blank(axis: Axes, letter: str, title: str) -> None:
        _panel(axis, letter, title, grid_axis=None)
        axis.set_xlim(0, 1)
        axis.set_ylim(0, 1)
        axis.set_xticks([])
        axis.set_yticks([])
        for spine in axis.spines.values():
            spine.set_visible(False)

    def _box(
        axis: Axes,
        xy: tuple[float, float],
        width: float,
        height: float,
        text: str,
        *,
        facecolor: str,
        edgecolor: str = "#555555",
        hatch: str | None = None,
    ) -> None:
        patch = FancyBboxPatch(
            xy,
            width,
            height,
            boxstyle="round,pad=0.012,rounding_size=0.018",
            facecolor=facecolor,
            edgecolor=edgecolor,
            linewidth=0.8,
            hatch=hatch,
        )
        axis.add_patch(patch)
        axis.text(xy[0] + width / 2, xy[1] + height / 2, text, ha="center", va="center")

    def _arrow(axis: Axes, start: tuple[float, float], end: tuple[float, float]) -> None:
        axis.add_patch(
            FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=10, linewidth=0.8, color="#555555")
        )

    def _child(parent: Axes, bounds: list[float]) -> Axes:
        axis = parent.inset_axes(bounds)
        axis.tick_params(width=0.65, length=2.5, pad=1, color="#333333")
        for side in ("top", "right"):
            axis.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            axis.spines[side].set_linewidth(0.65)
            axis.spines[side].set_color("#333333")
        axis.grid(True, axis="y", color="#D9D9D9", linewidth=0.35, zorder=0)
        return axis

    def _parent_for_children(axis: Axes, letter: str, title: str) -> None:
        _blank(axis, letter, title)

    def _critical(text, name: str):
        """Tag a layout-critical text artist for figure-specific geometry tests."""

        text.set_gid(f"figures-4-6-layout-critical:{name}")
        return text

    def _inset_panel_axis(
        axis: Axes,
        *,
        left: float = 0.0,
        right: float = 0.0,
        bottom: float = 0.0,
        top: float = 0.0,
    ) -> None:
        """Move one top-level data rectangle within its locked panel cell."""

        position = axis.get_position()
        axis.set_position(
            [
                position.x0 + left,
                position.y0 + bottom,
                position.width - left - right,
                position.height - bottom - top,
            ]
        )

    def build_figure_6(panel_directory: Path | str):
        """Build Figure 6: protected anonymous demand summaries."""

        panel_directory = Path(panel_directory)
        verify_panel_inputs(panel_directory, 6)
        figure, axes = create_panel_figure()
        a, b, c, d, e, f, g, h = axes.ravel()

        # a: conceptual sufficiency phase space.
        _blank(a, "a", "Is the mean sufficient?")
        a.add_patch(Polygon([[0.12, 0.14], [0.86, 0.14], [0.12, 0.82]], closed=True, facecolor=CLASS_COLOURS["mean-safe"], alpha=0.65, edgecolor="white"))
        a.add_patch(Polygon([[0.86, 0.14], [0.86, 0.82], [0.12, 0.82]], closed=True, facecolor=CLASS_COLOURS["distribution-required"], alpha=0.65, edgecolor="white"))
        a.add_patch(Rectangle((0.12, 0.70), 0.74, 0.12, facecolor=CLASS_COLOURS["finite-library-unresolved"], hatch="///", alpha=0.85, edgecolor="#777777"))
        a.plot([0.12, 0.86], [0.14, 0.14], color="#333333", linewidth=0.65)
        a.plot([0.12, 0.12], [0.14, 0.82], color="#333333", linewidth=0.65)
        a.text(0.49, 0.05, "Demand heterogeneity SD (°C)", ha="center")
        a.text(
            0.02,
            0.48,
            "Service-budget multiplier\n(looser allowance ↑)",
            rotation=90,
            va="center",
        )
        a.text(0.30, 0.31, "mean-safe", ha="center")
        a.text(0.64, 0.55, "distribution-required", ha="center")
        a.text(0.49, 0.76, "unresolved", ha="center")

        # b: exact simulated design cells in four contexts.
        phase = pd.read_csv(panel_directory / "Fig6b.csv")
        _parent_for_children(b, "b", "Sufficiency phase map")
        child_bounds = [[0.07, 0.45, 0.40, 0.22], [0.57, 0.45, 0.40, 0.22], [0.07, 0.10, 0.40, 0.22], [0.57, 0.10, 0.40, 0.22]]
        markers = {"mean-safe": "o", "distribution-required": "s", "finite-library-unresolved": "x"}
        for panel_index, (context, bound) in enumerate(zip(CONTEXTS, child_bounds)):
            child = _child(b, bound)
            rows = phase[phase["context"] == context]
            for classification in ["mean-safe", "distribution-required", "finite-library-unresolved"]:
                sub = rows[rows["classification"] == classification]
                child.scatter(sub["heterogeneity_sd_c"], sub["service_fraction"], s=9, color=CLASS_COLOURS[classification], marker=markers[classification], alpha=0.90 if classification == "finite-library-unresolved" else 0.85, linewidths=0.5)
            child.set_xlim(-0.08, 4.35)
            child.set_ylim(0.46, 1.04)
            child.set_title(CONTEXT_LABELS[context], loc="left", fontsize=11, fontweight="normal", pad=0)
            child.set_xticks([0, 2, 4])
            child.set_yticks([0.5, 0.75, 1.0])
            if panel_index < 2:
                child.set_xticklabels([])
            if panel_index % 2 == 1:
                child.set_yticklabels([])
        counts = phase["classification"].value_counts()
        b.legend(
            handles=[
                Line2D([], [], color=CLASS_COLOURS[key], marker=markers[key], linestyle="", markersize=5)
                for key in ["mean-safe", "distribution-required", "finite-library-unresolved"]
            ],
            labels=[
                f"Safe {counts['mean-safe']}",
                f"Distribution {counts['distribution-required']}",
                f"Unresolved {counts['finite-library-unresolved']}",
            ],
            frameon=False,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.92),
            ncol=3,
            handlelength=0.8,
            handletextpad=0.2,
            columnspacing=0.45,
        )

        # c: all cells and resolved subset.
        summary = pd.read_csv(panel_directory / "Fig6c.csv").set_index("classification")
        _panel(c, "c", "Outcome shares", grid_axis="x")
        orders = ["mean-safe", "distribution-required", "finite-library-unresolved"]
        labels = {"mean-safe": "Mean-safe", "distribution-required": "Distribution", "finite-library-unresolved": "Unresolved"}
        left = 0.0
        for key in orders:
            count = int(summary.loc[key, "count"])
            width = 100 * count / 1008
            c.barh([1], [width], left=[left], color=CLASS_COLOURS[key], hatch="///" if key == "finite-library-unresolved" else None, edgecolor="white", label=labels[key])
            c.text(left + width / 2, 1, f"{count}\n{width:.1f}%", ha="center", va="center")
            left += width
        left = 0.0
        for key in ["mean-safe", "distribution-required"]:
            count = int(summary.loc[key, "count"])
            width = 100 * count / 606
            c.barh([0], [width], left=[left], color=CLASS_COLOURS[key], edgecolor="white")
            c.text(left + width / 2, 0, f"{count}\n{width:.1f}%", ha="center", va="center")
            left += width
        _inset_panel_axis(c, left=0.025, bottom=0.025)
        c.set_yticks([1, 0], ["All", "Resolved"])
        c.set_xlim(0, 100)
        c.set_xlabel("Design units (%)")

        # d: threshold errors retain direction and context denominator.
        threshold = pd.read_csv(panel_directory / "Fig6d.csv")
        contexts = [x for x in CONTEXTS if x in set(threshold["context"])] + ["overall"]
        y = np.arange(len(contexts))[::-1]
        false_safe = []
        false_required = []
        denominators = []
        for context in contexts:
            row = threshold[threshold["context"] == context].iloc[0]
            false_safe.append(100 * row["false_safe"] / row["denominator"])
            false_required.append(100 * row["false_required"] / row["denominator"])
            denominators.append(int(row["denominator"]))
        d.barh(y, false_safe, color=PALETTE["vermillion"], label="False-safe")
        d.barh(y, false_required, left=false_safe, color=PALETTE["sky"], label="False-required")
        overall = float(threshold.loc[threshold["context"] == "overall", "error_rate"].iloc[0] * 100)
        d.axvline(overall, color="#333333", linewidth=0.8, linestyle="--")
        d.set_yticks(y, ["MIA", "NDL", "PTA", "SEA", "Overall"])
        d.set_xlim(0, 24)
        d.set_xlabel("Classification error (%)")
        d.set_ylabel("")
        for yy, fs, fr, den in zip(y, false_safe, false_required, denominators):
            d.text(fs + fr + 0.4, yy, f"n={den}", va="center")
        d.set_ylim(-0.6, 5.2)
        d.text(
            0.98,
            0.94,
            "in-sample scan 1.059 °C | all 15.2% | max 20.4%",
            transform=d.transAxes,
            ha="right",
            va="top",
        )
        _panel(d, "d", "Threshold residual error", grid_axis="x")
        _inset_panel_axis(d, left=0.045, bottom=0.025)

        # e: approximation quality in two aligned metrics.
        approx = pd.read_csv(panel_directory / "Fig6e.csv")
        _parent_for_children(e, "e", "Approximation quality")
        reps = ["mean_only", "quantile3", "quantile5"]
        rep_labels = ["Mean", "q3", "q5"]
        rep_colours = [REPRESENTATION_COLORS[key] for key in ["mean_only", "quantile3", "quantile5"]]
        for metric, label, bound, scale in [
            ("candidate_tail_mae_c", "Tail-service MAE (°C)", [0.05, 0.13, 0.42, 0.72], 1.0),
            ("eligibility_disagreement", "Eligibility disagreement (%)", [0.56, 0.13, 0.42, 0.72], 100.0),
        ]:
            child = _child(e, bound)
            for index, (rep, color) in enumerate(zip(reps, rep_colours)):
                values = approx.loc[approx["representation"] == rep, metric].to_numpy(float) * scale
                q1, med, q3 = np.quantile(values, [0.25, 0.5, 0.75])
                child.plot([index, index], [q1, q3], color=color, linewidth=2)
                child.scatter([index], [med], color=color, s=26, zorder=3)
                child.text(index, q3, f"{med:.2f}", ha="center", va="bottom")
            child.set_xticks(range(3), rep_labels)
            child.set_xlim(-0.45, 2.45)
            child.set_ylim(0, child.get_ylim()[1] * 1.22)
        e.text(0.24, 0.93, "Tail MAE (°C)", ha="center", va="top")
        e.text(0.76, 0.93, "Eligibility error (%)", ha="center", va="top")

        # f: protected certification trade-off, split facets.
        gate = pd.read_csv(panel_directory / "Fig6f.csv")
        _parent_for_children(f, "f", "Protected-quantile trade-off")
        split_bounds = [[0.09, 0.14, 0.38, 0.71], [0.58, 0.14, 0.40, 0.71]]
        split_labels = {
            "leave_one_context_seed_out": "Context-seed",
            "leave_one_population_out": "Population",
        }
        rep_styles = {
            "mean_only": (REPRESENTATION_COLORS["mean_only"], "s"),
            "quantile3": (REPRESENTATION_COLORS["quantile3"], "o"),
            "quantile5": (REPRESENTATION_COLORS["quantile5"], "^"),
        }
        for split, bound in zip(split_labels, split_bounds):
            child = _child(f, bound)
            rows = gate[gate["split_scheme"] == split]
            for rep, (color, marker) in rep_styles.items():
                sub = rows[rows["representation"] == rep]
                sizes = 15 + 65 * sub["decision_coverage"].to_numpy(float)
                yvals = np.log10(sub["false_accept_candidates"].to_numpy(float) + 1)
                child.scatter(100 * sub["true_action_retention"], yvals, s=sizes, color=color, marker=marker, alpha=0.72, edgecolors="white", linewidths=0.4)
            highlight = rows[(rows["representation"] == "quantile5") & (rows["margin_rule"] == "pooled_q95")].iloc[0]
            hx = 100 * highlight["true_action_retention"]
            hy = np.log10(highlight["false_accept_candidates"] + 1)
            child.scatter([hx], [hy], s=105, facecolors="none", edgecolors="#111111", linewidths=1.0)
            child.annotate(f"q5+q95\n{int(highlight['true_positive_candidates']):,}/{int(highlight['full_eligible_candidates']):,}\n{int(highlight['full_available_and_certified_decisions'])}/{int(highlight['full_available_decisions'])}", (hx, hy), xytext=(-3, 7), textcoords="offset points", ha="right", va="bottom")
            child.set_xlim(-4, 104)
            child.set_xticks([0, 50, 100])
            child.set_ylim(-0.15, np.log10(gate["false_accept_candidates"].max() + 1) + 0.15)
            child.set_yticks([0, 1, 2, 3, 4], ["0", "9", "99", "999", "9,999"])
            child.set_xlabel("")
            child.set_ylabel("")
            child.text(0.03, 0.97, split_labels[split], transform=child.transAxes, ha="left", va="top")
        f.text(0.50, 0.94, "x retention (%) | y false accepts (log) | size coverage", ha="center", va="top")

        # g: local anonymous update learning curves.
        learning = pd.read_csv(panel_directory / "Fig6g.csv")
        _parent_for_children(g, "g", "Local feedback")
        feedback_deltas = []
        for metric, label, bound in [("CRPS", "Normalized RPS", [0.10, 0.32, 0.36, 0.35]), ("tail_MAE", "Tail-share MAE", [0.61, 0.32, 0.36, 0.35])]:
            child = _child(g, bound)
            for split, color, marker in [("city", PALETTE["blue"], "o"), ("source", PALETTE["orange"], "s")]:
                rows = learning[(learning["metric"] == metric) & (learning["transfer_split"] == split)].sort_values("feedback_n")
                child.errorbar(rows["feedback_n"], rows["estimate"], yerr=[rows["estimate"] - rows["ci_low"], rows["ci_high"] - rows["estimate"]], color=color, marker=marker, capsize=2, label=split.capitalize())
                row0 = rows[rows["feedback_n"] == 0].iloc[0]
                row40 = rows[rows["feedback_n"] == 40].iloc[0]
                if split == "city":
                    feedback_deltas.append(float(row40["estimate"] - row0["estimate"]))
            child.set_xticks([0, 5, 10, 20, 40])
            child.set_xlabel("")
            child.set_ylabel("")
        g.text(0.28, 0.73, "Normalized RPS", ha="center", va="center")
        g.text(0.79, 0.73, "Tail-share MAE", ha="center", va="center")
        g.legend(
            handles=[
                Line2D([], [], color=PALETTE["blue"], marker="o", label="City"),
                Line2D([], [], color=PALETTE["orange"], marker="s", label="Source"),
            ],
            frameon=False,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.94),
            ncol=2,
            handlelength=1.2,
            columnspacing=0.8,
        )
        _critical(g.text(0.50, 0.035, f"City Δ40: RPS {feedback_deltas[0]:+.3f}; tail {feedback_deltas[1]:+.3f}", ha="center", va="bottom"), "6g-feedback-footer")

        # h: constructive anonymous-summary workflow without a formal privacy guarantee.
        workflow = json.loads((panel_directory / "Fig6h_meta.json").read_text(encoding="utf-8"))["steps"]
        _blank(h, "h", "Qualified reporting workflow")
        positions = [(0.03, 0.61), (0.37, 0.61), (0.71, 0.61), (0.12, 0.29), (0.62, 0.29)]
        short = ["Archive prior\nanonymous", "Local update\nroom/building", "Five quantiles\nanonymous", "Protected rule\nq95 margin", "Report outcomes\n+ recovery"]
        colours = ["#F2F2F2", "#E8F2F8", "#DDF2EB", "#DDF2EB", "#FDE9DF"]
        for (x, y), label, color, _step in zip(positions, short, colours, workflow):
            _box(h, (x, y), 0.26, 0.20, label, facecolor=color)
        _arrow(h, (0.29, 0.71), (0.37, 0.71))
        _arrow(h, (0.63, 0.71), (0.71, 0.71))
        _arrow(h, (0.84, 0.60), (0.37, 0.50))
        _arrow(h, (0.38, 0.39), (0.62, 0.39))
        h.text(0.50, 0.10, "Report representation + mean/tail service\naction range + recovery window", ha="center", va="center")
        return figure

    CAPTIONS = {
        4: (
            "Figure 4 | Demand spread contracts service-qualified annual action space. "
            "a, The same fixed action is audited under a compressed mean and the complete demand distribution. "
            "b, Fixed-centre spread perturbations for the pre-specified Miami multinational composition; horizontal segments show P10-P90. "
            "c, Full-demand and mean-based annual selections among 144 design cells at each spread multiplier. "
            "d, Mean-selected actions that fail full-demand review (denominator 128). "
            "e, Full-service eligible-action counts across all 144 cells per multiplier, retaining zero-action cells. "
            "f, Median and interquartile annual-energy change for bilateral full-service selections; the right-hand table gives matched/unmatched cells for λ=0, 0.5 and 1.5. "
            "g, Carbon intervals and median signed carrier changes for the same bilateral comparisons; colours and markers retain the λ encoding from panel f. "
            "h, Median non-additive eligibility interaction across contexts on one shared diverging action-count scale; selected-energy interactions are resolved in 585 of 1,728 cells."
        ),
        5: (
            "Figure 5 | Demand heterogeneity contracts service-qualified flexibility under event and recovery constraints. "
            "a, Pre-specified 2-h event, 48-h recovery and 50-h energy-accounting contract. "
            "b, Baseline and perturbed HVAC power for the same pre-specified action; the panel does not compare mean- and full-selected actions. "
            "c, Zone temperature envelope, occupancy-weighted temperature and full-distribution mean and fixed-27 tail-service trajectories for that case; horizontal lines are the fixed mean- and tail-service budgets. "
            "d, July cells with a service-qualified positive response under full and mean demand. "
            "e, Mean-positive selections that fail full-demand review (denominator 122). "
            "f, Selected response for 12 common full-service cells at spread multipliers 1 and 1.5. "
            "g, Response, recovery peak, recovery electricity and 50-h net-saving changes for those 12 cells; each header reports decrease, tie and increase counts. "
            "h, Positive full-service selections across demand location and spread; the February central reference (0/144) is distinct from the complete February grid (210/2,880)."
        ),
        6: (
            "Figure 6 | Protected anonymous demand summaries recover service-qualified decisions beyond a single mean. "
            "a, Schematic conditions governing whether an average is sufficient; the vertical axis is the service-budget multiplier, for which larger values permit looser service-error allowances, and the drawn regions do not define an empirical monotonic boundary. "
            "b, Exact sampled design cells classified across four contexts without spatial interpolation. "
            "c, Counts among all 1,008 design units and the 606 resolvable units. "
            "d, False-safe and false-required errors from the lowest-error 1.059 °C in-sample scanned diagnostic threshold; it did not meet the prespecified total-error, worst-context-error, sensitivity and specificity standards, and denominators are context-specific. "
            "e, Median and interquartile service-reconstruction errors for mean, three-quantile and five-quantile summaries. "
            "f, Held-out certification trade-off; marker shape identifies the anonymous summary and marker size encodes decision coverage. The highlighted q5 plus training-only pooled-q95 rule has zero observed false acceptance, retains 14,709/18,766 true-qualified candidates and covers 542/606 resolvable decisions in the context-seed split. "
            "g, Local-update learning curves; points and error bars show updated scores and their hierarchical-bootstrap 95% intervals. In the paired improvement analysis at 40 observations, both city-held-out improvement intervals are above zero, whereas both source-held-out improvement intervals cross zero. "
            "h, Anonymous-summary workflow from archive prior to qualified energy, carbon and flexibility reporting; anonymity of the summaries is not a formal privacy guarantee."
        ),
    }

    ALT_TEXT = {
        4: (
            "Eight-panel figure comparing one fixed annual action library under mean and full thermal-demand service checks. "
            "At fixed mean demand, increasing spread λ contracts full-qualified selections from 128/144 to 21/144 while mean-based selections stay 128/144; false acceptance rises from 0/128 to 128/128. "
            "Eligible action counts collapse, and matched annual-energy and carbon effects differ by context and carrier. The matched/unmatched comparison contains 169/263 cells. "
            "Centre×spread eligibility interactions are non-zero in 1,261/1,728 cells; energy interactions are resolved for 585/1,728 cells, with 1,143 unresolved."
        ),
        5: (
            "Eight-panel figure comparing service-qualified demand-response flexibility with a pre-specified event, recovery and energy-accounting contract. "
            "Panels b and c show baseline and perturbed physical trajectories for the same pre-specified action, with 28.3 kW response, 13.1 kW rebound and 30.3 kWh net saving; they do not compare mean- and full-selected actions. "
            "Across July design cells, full-qualified positive selections fall from 122/144 to 12/144 as λ rises, while mean-based selections remain 122/144 and false acceptance reaches 115/122. "
            "Only 12 cells are common at λ=1 and 1.5; five lose response. February has 0/144 positive selections at the central reference but 210/2,880 across the complete grid."
        ),
        6: (
            "Eight-panel figure testing anonymous demand summaries and a protected qualification rule. "
            "Panel a is a schematic separation of mean-safe, distribution-required and unresolved regimes by demand heterogeneity and service-budget multiplier; larger multipliers permit looser service-error allowances, and the drawn regions are not an empirical monotonic boundary. "
            "Panel b maps the three regimes for the exact sampled cells in Miami, New Delhi, Port Angeles and Seattle. "
            "Panel c shows that, of 1,008 design units, 275 are mean-safe, 331 require the demand distribution and 402 are unresolved; the resolved denominator is 606. "
            "Panel d shows that the lowest-error 1.059 °C in-sample scanned diagnostic threshold makes 92/606 errors and does not meet the prespecified performance standards. "
            "Panel e shows lower median tail-service and eligibility errors for three- and five-quantile summaries than for the mean alone. "
            "Panel f shows that, in the context-seed holdout, q5 plus a training-only pooled-q95 margin yields zero observed false accepts, retains 14,709/18,766 true-qualified candidates and covers 542/606 resolvable decisions; this is held-out evidence, not a universal guarantee. "
            "Panel g shows local-feedback learning curves for dimensionless normalized ordinal RPS and tail-share MAE. Their points and error bars are updated scores and their hierarchical-bootstrap 95% intervals. In the paired improvement analysis at 40 city-held-out observations, the RPS and tail-share error improvement intervals are above zero, whereas at 40 source-held-out observations both corresponding improvement intervals cross zero, including the tail-share error interval. "
            "Panel h traces an anonymous-summary qualified-reporting workflow from an archive prior through local updating, five anonymous quantiles and a protected q95 qualification rule to energy, carbon, flexibility and recovery reporting; it makes no formal privacy guarantee."
        ),
    }
    from render_revised_main_figures import restructure, readable_display_labels
    from thermal_heterogeneity.figures import apply_figure_style, apply_data_axis_style
    DPI = 600
    apply_figure_style()
    PANEL_ROOT = RESULTS / "panel_data"
    figure = build_figure_6(PANEL_ROOT)
    figure._stage7_panel_root = PANEL_ROOT
    figure._stage7_decision_value = RESULTS / "decision_value/MAIN_FIG6G_Q5_REGRET.csv"
    restructure(figure, 6)
    readable_display_labels(figure, 6)
    apply_data_axis_style(figure, map_axes=figure.axes[:2] if 6 == 1 else [])
    save_plot(figure, LABEL, dpi=DPI, exact_pixels=True)


draw_figure_s14()

In [ ]:
def draw_s15():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S15"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    learn = read("expected/local_update_learning_curve_reference.csv")

    fig, a = canvas(
        [
            "City transfer: ordinal RPS",
            "Source transfer: ordinal RPS",
            "City transfer: tail-share error",
            "Source transfer: tail-share error",
        ]
    )

    for ax, (split, metric) in zip(
        a, [("city", "crps"), ("source", "crps"), ("city", "tail_mae"), ("source", "tail_mae")]
    ):
        d = learn[learn.split.astype(str).str.contains(split)].sort_values("feedback_n")
        key = "updated_" + metric
        ax.errorbar(
            d.feedback_n,
            d[key],
            yerr=[d[key] - d[key + "_ci_low"], d[key + "_ci_high"] - d[key]],
            fmt="o-",
            capsize=3,
            color=PALETTE["blue"] if split == "city" else PALETTE["orange"],
        )
        ax.plot(d.feedback_n, d["base_" + metric], ls="--", color="gray", label="Prior")
        ax.set_xlabel("Local feedback observations")
        ax.set_ylabel("Ordinal RPS (unitless)" if metric == "crps" else "Tail-share MAE (fraction)")
        ax.set_xticks([0, 10, 20, 40])

    a[0].legend(frameon=False)

    manifest.append(
        save(
            fig,
            out,
            9,
            "Figure S9 | Local-update learning curves. a,b, Analysis-unit-balanced ordinal ranked probability score (RPS, unitless) under city-held-out and source-held-out transfer. c,d, Mean absolute error of predicted versus observed warmer/cooler vote shares (fraction) under the same splits. Dashed lines show the prior. Points and whiskers show updated estimates and hierarchical-bootstrap 95% intervals. The horizontal coordinate is the number of local feedback observations.",
            ["expected/local_update_learning_curve_reference.csv"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S09"
    final_stem = "Figure_S15"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S15.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s15()

In [ ]:
def draw_s16():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S16"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    human = read("human/human_profiles_minimized.csv.gz")
    chinese_reference = pd.read_csv(RESULTS / "CTCD_external/CTCD_REFERENCE_HISTOGRAM.csv")
    chinese_widths = pd.read_csv(RESULTS / "CTCD_external/CTCD_INTERVAL_HISTOGRAM.csv")

    fig, a = canvas(
        [
            "Temperature and votes",
            "Reference sensitivity",
            "Records per unit",
            "Personal range widths",
        ]
    )

    h = a[0].hist2d(
        human.observed_temperature_c, human.thermal_sensation, bins=[35, 7], cmap=COUNT_CMAP
    )

    a[0].set_xlabel("Observed temperature (°C)")

    a[0].set_ylabel("Thermal-sensation vote")

    # Keep the count key outside the data axes and leave a clear right gutter.
    cax = fig.add_axes([0.48, 0.59, 0.012, 0.32])

    bar = fig.colorbar(h[3], cax=cax, ticks=[0, 750, 1500])

    bar.ax.set_title("Records", fontsize=11, weight="normal", pad=6)

    for g, c, label in zip(
        ["reference_g033_c", "reference_g050_c", "reference_g067_c"],
        G_COLORS,
        ["ASHRAE g = 0.33", "ASHRAE g = 0.50", "ASHRAE g = 0.67"],
    ):
        a[1].hist(
            human[g],
            bins=np.arange(10, 38, 0.5),
            histtype="step",
            density=True,
            color=c,
            label=label,
        )

    chinese_primary = chinese_reference[chinese_reference.griffiths_coefficient.eq(0.50)]
    chinese_edges = np.r_[chinese_primary.bin_lower_c, chinese_primary.bin_upper_c.iloc[-1]]
    a[1].stairs(chinese_primary.density_per_c, chinese_edges,
                color=COLORS[3], linewidth=1.6, linestyle="--", label="CTCD g = 0.50")

    a[1].set_xlabel("Reference centre (°C)")

    a[1].set_ylabel("Record density")
    a[1].yaxis.set_label_coords(-.175, .5)

    a[1].set_ylim(0, 0.41)
    a[1].set_yticks([0, .1, .2, .3, .4])
    a[1].legend(frameon=False, loc="upper left", ncol=1, handlelength=1.1,
                columnspacing=0.7, handletextpad=0.4, labelspacing=0.25)

    counts = human.groupby("release_unit_id").size()

    a[2].hist(
        counts, bins=np.arange(0, min(counts.max(), 40) + 2) - 0.5, color=COLORS[0], alpha=0.9
    )

    a[2].set_yscale("log")

    a[2].set_xlabel("Records per analysis unit")

    a[2].set_ylabel("Analysis units (log)")

    intervals = read("human/ltpref_intervals_minimized.csv.gz")

    width_cols = [c for c in intervals if "width" in c and "multiplier" not in c]

    lo = next((c for c in intervals if "lower" in c and c.endswith("_c")), None)

    hi = next((c for c in intervals if "upper" in c and c.endswith("_c")), None)

    if lo and hi:
        widths = intervals[hi] - intervals[lo]
    elif width_cols:
        widths = intervals[width_cols[0]]
    else:
        raise ValueError("No interval-width fields")

    width_edges = np.r_[chinese_widths.bin_lower_c, chinese_widths.bin_upper_c.iloc[-1]]
    a[3].hist(widths.dropna(), bins=width_edges, density=True,
              color=COLORS[2], alpha=0.50, label="ASHRAE")
    a[3].stairs(chinese_widths.density_per_c, width_edges, color=COLORS[3],
                linewidth=1.6, linestyle="--", label="CTCD")
    a[3].legend(frameon=False, loc="upper right", handlelength=1.4)

    a[3].set_xlabel("Model-derived width (°C)")

    a[3].set_ylabel("Record density")

    caption = 'Supplementary Figure S16 | Observation and derived-input diagnostics. a, Observed temperature and thermal-sensation votes in the ASHRAE office records. Colour gives records per bin. b, Reconstructed reference centres at three Griffiths coefficients in ASHRAE records, with the coefficient-0.50 distribution of 12,427 overlap-screened CTCD office records collected during active heating or cooling. Each histogram is normalized within its plotted temperature range. c, Records per ASHRAE analysis unit, with unit counts on a logarithmic axis. Analysis units include identified subject-within-building groups and single-record units without a usable subject identifier. The CTCD records have no verified subject identifier. d, Normalized model-generated Personal range widths for 21,399 ASHRAE and 12,404 executable CTCD office records. The same Griffiths-centre and 19-scene numerical range-construction workflow is applied to both datasets. These ranges are model outputs, not observed individual acceptable ranges.'

    manifest.append(
        save(
            fig,
            out,
            16,
            caption,
            ["human/human_profiles_minimized.csv.gz", "human/ltpref_intervals_minimized.csv.gz"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S16"
    final_stem = "Figure_S16"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    (RESULTS / "captions/Figure_S16.txt").write_text(caption + "\n", encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s16()


In [ ]:
def draw_s17():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S17"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    input_root = RESULTS / "energy_service"

    engineering = input_root / "engineering"

    profile_path = engineering / "PROFILE_ASSIGNMENTS.csv.gz"

    command_path = engineering / "CONTROL_REFERENCE.json"

    group_path = engineering / "ZONE_EXECUTION_GROUPS.csv"

    profiles = pd.read_csv(profile_path)

    commands = json.loads(command_path.read_text(encoding="utf-8"))

    groups = pd.read_csv(group_path)

    example = profiles[(profiles.context == "Miami") &
                       (profiles.layout == "random_mixing")].copy()

    composition_ids = sorted(example.composition_id.unique())

    if len(composition_ids) != 3 or any(
        len(example[example.composition_id == key]) != 267 for key in composition_ids
    ):
        raise ValueError("Expected the three fixed compositions of 267 anonymous profiles")

    source = []

    for row in commands:
        if row["variant"] != 0:
            continue
        for zone, target in row["targets_c"].items():
            source.append({**{k: row[k] for k in ("context", "composition_id", "layout",
                                                "permission", "control", "variant")},
                           "zone_id": zone, "target_c": target})

    source = pd.DataFrame(source)

    source.to_csv(out / "actual_control_target_source.csv", index=False)

    example.to_csv(out / "actual_assigned_profile_source.csv", index=False)

    colors = [PALETTE["blue"], PALETTE["purple"], PALETTE["green"]]

    fig, axes = canvas(["Assigned interval spectra", "Demand centres and widths",
                        "Executed zone commands", "Range across controlled zones"])

    for offset, (composition, color) in enumerate(zip(composition_ids, colors)):
        local = example[example.composition_id == composition].sort_values("centre_c")
        y = np.arange(267) + 267 * offset
        axes[0].hlines(y, local.lower_c, local.upper_c, color=color, lw=.42, alpha=.45)
        axes[0].scatter(local.centre_c, y, s=2.5, color=color, linewidths=0, zorder=3)
        axes[1].scatter(local.centre_c, local.upper_c - local.lower_c, s=10,
                        color=color, alpha=.65, linewidths=0,
                        label=f"Composition {offset + 1}")

    axes[0].set_yticks([133, 400, 667], ["1", "2", "3"])

    axes[0].set_ylim(-10, 811)

    axes[0].set_xlim(16, max(32, np.ceil(example.centre_c.max())))

    axes[0].set_xlabel("Assigned demand interval (°C)")

    axes[0].set_ylabel("Composition (267 profiles each)")

    axes[1].set_xlabel("Demand centre (°C)")

    axes[1].set_ylabel("Model-derived width (°C)")

    axes[1].legend(frameon=False, loc="upper right", handletextpad=.3)

    native = source[source.permission == "native_zone"]

    context_names = ["Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019"]

    controls = ["uniform_average", "personal_center", "ltpref_interval"]

    control_colors = [CONTROL_COLORS["uniform"], CONTROL_COLORS["personal_center"],
                      CONTROL_COLORS["ltpref_interval"]]

    for control_index, (control, color) in enumerate(zip(controls, control_colors)):
        for context_index, context in enumerate(context_names):
            local = native[(native.context == context) & (native.control == control)]
            position = context_index + (control_index - 1) * .24
            bp = axes[2].boxplot([local.target_c.to_numpy()], positions=[position], widths=.20,
                                patch_artist=True, manage_ticks=False, showfliers=False,
                                medianprops={"color": "black", "linewidth": .8},
                                boxprops={"linewidth": .65}, whiskerprops={"linewidth": .65},
                                capprops={"linewidth": .65})
            bp["boxes"][0].set_facecolor(color)
            bp["boxes"][0].set_alpha(.85)
            points = np.sort(local.target_c.to_numpy())
            jitter = (np.arange(len(points)) % 7 - 3) * .018
            axes[2].scatter(position + jitter, points, s=3, color=color, alpha=.55,
                            linewidths=0, zorder=3)

    axes[2].set_xticks(range(4), ["Miami", "New\nDelhi", "Port\nAngeles", "Seattle"])

    axes[2].set_ylabel("Occupied target (°C)")

    axes[2].set_xlabel("Building–weather context")

    ranges = (source.groupby(["context", "composition_id", "layout", "permission", "control"])
              .target_c.agg(lambda value: value.max() - value.min()).reset_index(name="span_c"))

    group_lookup = {}

    for context, local in groups.groupby("context"):
        group_lookup[(context, "building")] = 1
        group_lookup[(context, "floor")] = local.floor.nunique()
        group_lookup[(context, "native_zone")] = local.zone.nunique()

    ranges["independent_command_groups"] = [group_lookup[(row.context, row.permission)]
                                           for row in ranges.itertuples()]

    ranges.to_csv(out / "actual_control_command_range_source.csv", index=False)

    for i, permission in enumerate(["building", "floor", "native_zone"]):
        for j, (control, color) in enumerate(zip(controls, control_colors)):
            values = ranges[(ranges.permission == permission) &
                            (ranges.control == control)].span_c.to_numpy()
            position = i + (j - 1) * .22
            lo, med, hi = np.quantile(values, [.1, .5, .9])
            axes[3].plot([lo, hi], [position, position], color=color, lw=2)
            axes[3].scatter(values, position + (np.arange(len(values)) % 5 - 2) * .025,
                            color=color, s=8, alpha=.55, linewidths=0)
            axes[3].scatter([med], [position], color=color, s=23, edgecolors="white", lw=.4,
                            zorder=4)

    axes[3].set_yticks(range(3), ["Building", "Floor", "Zone"])

    axes[3].invert_yaxis()

    axes[3].set_xlabel("Within-plan target range (°C)")

    axes[3].set_ylabel("Command permission")

    fig.legend(handles=[Patch(facecolor=color, label=label) for color, label in
                         zip(control_colors, ["Uniform average", "Personal centre", "Personal range"])],
               frameon=False, ncol=3, loc="center", bbox_to_anchor=(.53, .505),
               columnspacing=1.5, handlelength=1.2)

    caption = (
        "Figure S13 | Assigned demand inputs and control commands in the main comparison. "
        "a, The three fixed compositions, each containing 267 archive-derived virtual profiles. "
        "Profiles are ordered by reconstructed centre within each composition. Horizontal marks "
        "give model-derived intervals and points give centres. These are the Miami random-mixing "
        "assignment, whose three profile sets are also used in the other layouts and contexts. "
        "b, Centres and model-derived interval widths for the same 801 profile assignments. "
        "c, Distributions of the actually generated occupied zone-level thermostat targets for "
        "the minimum-predicted-mismatch candidate of each control, including all three compositions and layouts. "
        "Boxes give medians and interquartile ranges, whiskers extend to 1.5 interquartile ranges, "
        "and points retain every zone command. d, The maximum minus minimum zone target of each "
        "minimum-predicted-mismatch plan, separated by command permission and control. Points give all declared "
        "plans, thick segments span P10–P90 and larger white-edged points mark medians. "
        "Uniform, personal-centre and personal-range controls are grey, gold and green. Small-building "
        "floor and building permissions coincide. Commands are model inputs, not measured room temperatures."
    )

    metadata = {path.relative_to(input_root).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
                for path in [profile_path, command_path, group_path]}

    (out / "actual_control_input_sha256.json").write_text(json.dumps(metadata, indent=2),
                                                       encoding="utf-8")

    manifest.append(save(fig, out, 13, caption, list(metadata)))

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S13"
    final_stem = "Figure_S17"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = 'Supplementary Figure S17 | Original-archive demand inputs and control-command diagnostics. a, The three original ASHRAE compositions, each containing 267 archive-derived virtual profiles. Profiles are ordered by reconstructed centre within each composition. Horizontal marks give model-derived intervals and points give centres. These are the Miami random-mixing assignment, whose three profile sets are also used in the other layouts and contexts. b, Centres and model-derived interval widths for the same 801 profile assignments. c, Distributions of the actually generated occupied zone-level thermostat targets for the minimum-predicted-mismatch candidate of each control, including all three compositions and layouts. Boxes give medians and interquartile ranges, whiskers extend to 1.5 interquartile ranges, and points retain every zone command. d, The maximum minus minimum zone target of each minimum-predicted-mismatch plan, separated by command permission and control. Points give all declared plans, thick segments span 10th–90th percentiles and larger white-edged points mark medians. Uniform, personal-centre and personal-range controls are grey, gold and green. Small-building floor and building permissions coincide. Commands are model inputs, not measured room temperatures. Personal range uses model-generated individual temperature ranges. All controls choose shared targets within the permitted spaces.'
    (RESULTS / "captions/Figure_S17.txt").write_text(caption, encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s17()

In [ ]:
# Figure_S18: edit this cell and rerun to update this figure.
def draw_figure_s18():
    LABEL = "Figure_S18"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import argparse

    import hashlib

    import json

    from pathlib import Path

    import matplotlib

    import matplotlib.pyplot as plt

    from matplotlib.font_manager import FontProperties, findfont

    from matplotlib.lines import Line2D

    from matplotlib.text import Text

    from PIL import Image

    import numpy as np

    import pandas as pd

    from matplotlib.colors import LinearSegmentedColormap

    from matplotlib.ticker import MaxNLocator

    FONT = "Times New Roman"

    WIDTH_MM, HEIGHT_MM = 180, 183

    CONTROL_ORDER = ("uniform_average", "personal_center", "ltpref_interval")

    CONTROL_LABELS = {
        "uniform_average": "Uniform average",
        "personal_center": "Personal centre",
        "ltpref_interval": "Personal range",
    }

    CONTROL_COLORS = {
        "uniform_average": "#7F7F7F",
        "personal_center": "#C8A36A",
        "ltpref_interval": "#7FA995",
    }

    CONTEXT_ORDER = ("Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019")

    CONTEXT_COLORS = dict(zip(CONTEXT_ORDER, ("#C78470", "#AF94AD", "#6F9EBC", "#7FA995")))

    CONTEXT_LABELS = dict(zip(CONTEXT_ORDER, ("Miami", "New Delhi", "Port Angeles", "Seattle")))

    CONTEXT_SHORT = ("Miami", "New\nDelhi", "Port\nAngeles", "Seattle")

    CONTROL_MARKERS = dict(zip(CONTROL_ORDER, ("o", "s", "^")))

    ALTERNATIVES = CONTROL_ORDER[1:]

    DATA_BOUNDS_MM = tuple(
        (x, y, 67.0, 35.0) for y in (125.0, 72.0, 19.0) for x in (19.0, 109.0)
    )

    def configure_style() -> dict[str, str]:
        fonts = {}
        for weight in ("normal", "bold"):
            prop = FontProperties(family=FONT, weight=weight)
            path = Path(findfont(prop, fallback_to_default=False))
            if FontProperties(fname=str(path)).get_name() != FONT:
                raise ValueError(f"Exact {FONT} {weight} is required")
            fonts[weight] = str(path)
        plt.rcParams.update({
            "font.family": FONT, "font.size": 11, "text.color": "black",
            "axes.labelsize": 11, "axes.labelcolor": "black", "axes.labelpad": 6,
            "axes.titlesize": 12, "axes.titleweight": "bold",
            "axes.spines.top": False, "axes.spines.right": False,
            "axes.edgecolor": "#777777", "axes.linewidth": 0.75,
            "xtick.labelsize": 11, "ytick.labelsize": 11,
            "xtick.major.pad": 4, "ytick.major.pad": 4,
            "xtick.color": "black", "ytick.color": "black",
            "legend.fontsize": 11, "legend.frameon": False,
            "figure.facecolor": "white", "savefig.facecolor": "white",
            "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
            "svg.hashsalt": "thermal-heterogeneity-stage8",
        })
        return fonts

    def six_panel_canvas(titles: list[str], *, control_key: bool = True, bounds=None, height=None):
        """Create equal data frames with separate title, axis and shared-key bands."""
        if len(titles) != 6:
            raise ValueError("The main result template has exactly six data panels")
        configure_style()
        page_height = height or HEIGHT_MM
        fig = plt.figure(figsize=(WIDTH_MM / 25.4, page_height / 25.4))
        axes = []
        for index, (title, panel_bounds) in enumerate(zip(titles, bounds or DATA_BOUNDS_MM)):
            x, y, width, height_mm = panel_bounds
            ax = fig.add_axes((x / WIDTH_MM, y / page_height, width / WIDTH_MM, height_mm / page_height))
            ax.set_title(f"({chr(97 + index)}) {title}", loc="left", pad=8)
            ax.grid(False)
            ax.set_axisbelow(True)
            axes.append(ax)
        if control_key:
            handles = [Line2D([], [], marker=marker, linestyle="none", color=CONTROL_COLORS[key],
                              markersize=5, label=CONTROL_LABELS[key])
                       for key, marker in zip(CONTROL_ORDER, ("o", "s", "^"))]
            fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.995),
                       ncol=3, handletextpad=0.3, columnspacing=0.9)
        return fig, axes

    def point_summary(ax, frame, metric, ylabel, *, scale=1.0, controls=CONTROL_ORDER,
                      signed=False, percentage=False):
        """Show every finite scenario plus medians/IQRs, never a selected favourable subset."""
        records = []
        offsets = np.linspace(-0.23, 0.23, len(controls))
        for context_index, context in enumerate(CONTEXT_ORDER):
            cases = frame[frame.context.eq(context)]
            if all(column in cases for column in ("composition_id", "layout")):
                for _, case in cases.groupby(["composition_id", "layout"], sort=True):
                    x_line, y_line = [], []
                    for control, offset in zip(controls, offsets):
                        part = case[case.control.eq(control)]
                        if len(part) == 1 and pd.notna(part.iloc[0][metric]):
                            x_line.append(context_index + offset)
                            y_line.append(float(part.iloc[0][metric]) * scale)
                    if len(y_line) > 1:
                        ax.plot(x_line, y_line, color="#BDB9B2", linewidth=.6, alpha=.65, zorder=1)
            for control, offset in zip(controls, offsets):
                selected = frame[frame.context.eq(context) & frame.control.eq(control)].copy()
                values = pd.to_numeric(selected[metric], errors="raise").to_numpy(dtype=float) * scale
                finite = values[np.isfinite(values)]
                if finite.size:
                    x = context_index + offset
                    jitter = np.linspace(-0.045, 0.045, finite.size) if finite.size > 1 else np.zeros(1)
                    ax.scatter(x + jitter, finite, s=12, color=CONTROL_COLORS[control], alpha=0.78,
                               marker=CONTROL_MARKERS[control], linewidths=.2, edgecolors="white", zorder=2)
                    low, median, high = np.quantile(finite, [0.25, 0.5, 0.75])
                    ax.errorbar(x, median, yerr=[[median - low], [high - median]],
                                fmt=CONTROL_MARKERS[control], color=CONTROL_COLORS[control],
                                markersize=4.5, linewidth=1.5, capsize=2.3, zorder=3)
                else:
                    low = median = high = np.nan
                records.append({"context": context, "control": control, "metric": metric,
                                "declared_cases": len(selected), "finite_cases": len(finite),
                                "q25_display": low, "median_display": median, "q75_display": high})
        if signed:
            ax.axhline(0, color="#A5A5A5", linewidth=0.65, zorder=0)
        if percentage:
            finite_values = pd.to_numeric(frame[metric], errors="raise").to_numpy(float) * scale
            finite_values = finite_values[np.isfinite(finite_values)]
            if len(finite_values):
                span = max(np.ptp(finite_values), 8)
                ax.set_ylim(max(-2, float(finite_values.min()) - span * .12),
                            min(102, float(finite_values.max()) + span * .12))
        if ax.get_position().width * ax.figure.get_figwidth() * 25.4 < 55:
            ax.set_xticks(range(4), list(CONTEXT_LABELS.values()), rotation=75, ha="right")
        else:
            ax.set_xticks(range(4), CONTEXT_SHORT)
        ax.set_xlim(-0.55, 3.55)
        ax.set_ylabel(ylabel)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.ticklabel_format(axis="y", style="plain", useOffset=False)
        return records
    def render_monthly_support(builder, args, mode):
        """Monthly summaries retain all qualified trajectories in the joint scope."""
        from matplotlib.lines import Line2D
        from matplotlib.patches import Patch
        table = pd.read_csv(args.seasonal_root / 'PERIOD_TOTALS.csv')
        selected = table[table.comparison_mode.eq(mode) & table.period_type.eq('month')
            & table.permission.eq('native_zone') & table.data_status.eq('qualified')].copy()
        if table.composition_id.nunique() != 6 or set(table.layout) != {'random_mixing', 'demand_ordered'}:
            raise ValueError('Seasonal evidence must retain the six-composition, two-layout design')
        selected['month_number'] = selected.period.astype(int)
        keys = ['context', 'composition_id', 'layout', 'control', 'run_id']
        for _, trajectory in selected.groupby(keys):
            if sorted(trajectory.month_number) != list(range(1, 13)) or abs(trajectory.hours.sum() - 8760) > 1e-6:
                raise ValueError('Every included annual trajectory must contain all twelve months and 8760 hours')
        selected.to_csv(args.output / (LABEL + '_monthly_source.csv'), index=False)
        height = 230.0
        figure = builder.plt.figure(figsize=(180 / 25.4, height / 25.4))
        axes, counts = [], []
        stack_keys = ['heating_electric_kwh', 'cooling_electric_kwh', 'fans_electric_kwh', 'gas_hhv_kwh']
        closure = (selected[stack_keys].sum(axis=1) - selected.purchased_site_kwh).abs().max()
        if closure > 1e-8:
            raise ValueError('Space-conditioning components must close to the fixed purchased-energy endpoint')
        stack_colors = ['#C78470', '#6F9EBC', '#C8A36A', '#AF94AD']
        for i, context in enumerate(builder.CONTEXT_ORDER):
            label = builder.CONTEXT_LABELS[context]
            left = figure.add_axes(np.array([19, 184-i*54, 67, 30]) / [180, height, 180, height])
            right = figure.add_axes(np.array([109, 184-i*54, 67, 30]) / [180, height, 180, height])
            axes.extend([left, right])
            left.set_title(f'({chr(97+2*i)}) {label}: service', loc='left', pad=8)
            right.set_title(f'({chr(98+2*i)}) {label}: energy', loc='left', pad=8)
            block = selected[selected.context.eq(context)]
            baseline = block[block.control.eq('uniform_average')].groupby('month_number')[stack_keys].mean()
            if len(baseline) == 12:
                right.stackplot(baseline.index.to_numpy(), *[baseline[k].to_numpy()/1000 for k in stack_keys],
                                colors=stack_colors, alpha=.85, linewidth=0, zorder=1)
            n = []
            for control in builder.CONTROL_ORDER:
                values = block[block.control.eq(control)]
                n.append(values[['composition_id', 'layout']].drop_duplicates().shape[0])
                counts.append(dict(context=context, control=control, qualified_conditions=n[-1], designed_conditions=12))
                if values.empty:
                    continue
                grouped = values.groupby('month_number')
                size = {'uniform_average': 4.5, 'personal_center': 3.7, 'ltpref_interval': 3.0}[control]
                for ax, metric, scale in [(left, 'mean_reference_mismatch_c', 1), (right, 'purchased_site_kwh', .001)]:
                    summary = grouped[metric].agg(['mean', 'min', 'max']) * scale
                    ax.fill_between(summary.index, summary['min'], summary['max'],
                                    color=builder.CONTROL_COLORS[control], alpha=.10, linewidth=0, zorder=2)
                    ax.plot(summary.index, summary['mean'], color=builder.CONTROL_COLORS[control],
                            marker=builder.CONTROL_MARKERS[control], linewidth=1.3, markersize=size,
                            markerfacecolor='white', markeredgewidth=.7, zorder=4)
            left.text(.02, .98, 'n = ' + ' / '.join(map(str, n)), transform=left.transAxes,
                      ha='left', va='top', fontsize=11)
            low, high = left.get_ylim()
            left.set_ylim(low, high + .25 * max(high-low, .1))
            left.set_ylabel('Mismatch (°C)')
            right.set_ylabel('Energy (MWh)')
            for ax in (left, right):
                ax.set_xlim(.8, 12.2)
                ax.set_xticks([1, 4, 7, 10], ['January', 'April', 'July', 'October'], rotation=30, ha='right')
                ax.grid(False)
                ax.yaxis.set_major_locator(builder.MaxNLocator(3))
        figure.legend([Line2D([], [], color=builder.CONTROL_COLORS[c], marker=builder.CONTROL_MARKERS[c],
                             linestyle='-', lw=1.3, markersize=4, markerfacecolor='white') for c in builder.CONTROL_ORDER],
                      [builder.CONTROL_LABELS[c] for c in builder.CONTROL_ORDER], loc='upper center',
                      bbox_to_anchor=(.5, .995), ncol=3, frameon=False, columnspacing=.9, handletextpad=.3)
        figure.legend([Patch(facecolor=color, alpha=.85) for color in stack_colors],
                      ['Heating', 'Cooling', 'Fans', 'Gas'], loc='lower center',
                      bbox_to_anchor=(.5, .005), ncol=4, frameon=False, columnspacing=.8, handletextpad=.3,
                      handlelength=1.1)
        mode_label = 'direct control operation' if mode == 'direct' else 'common-service annual selections'
        number = 18 if mode == 'direct' else 19
        selection_description = ('Direct operation uses each control’s minimum-predicted-mismatch candidate.'
            if mode == 'direct' else 'Common-service choices use the reference annual mean and fixed-tail limits; their monthly service outcomes can vary.')
        caption = (
            f'Supplementary Figure S{number} | Monthly thermal service and energy: {mode_label}. '
            'All six fixed demand compositions and both random and demand-ordered layouts are '
            'included with zone-level commands in four building-weather contexts. '
            'a,c,e,g, Occupied population-weighted absolute temperature-reference mismatch. '
            'b,d,f,h, Purchased energy for heating, ventilation and air conditioning. Lines show '
            'means across engineering-qualified composition-layout conditions for each control, '
            'and shading gives their full minimum-to-maximum range. Counts in the service panels '
            'give the included conditions in legend order, from 12 designed conditions per control '
            'and context. The energy background stacks mean uniform-average heating electricity, '
            'cooling electricity, fan electricity and heating gas. Gas uses '
            'its higher heating value. Total lines include all these carriers. The same underlying '
            '8760-hour trajectories supply all twelve months. '
            f'{selection_description} '
            'Each context retains its own prototype energy scale and occupancy weights. '
            'Personal range denotes model-generated individual demand ranges.'
        )
        (RESULTS / 'captions' / f'{LABEL}.txt').write_text(caption, encoding='utf-8')
        pd.DataFrame(counts).to_csv(args.output / (LABEL + '_counts.csv'), index=False)
        return figure

    DPI = 600
    builder = SimpleNamespace(plt=plt,CONTEXT_ORDER=CONTEXT_ORDER,CONTEXT_LABELS=CONTEXT_LABELS,
        CONTROL_ORDER=CONTROL_ORDER,CONTROL_COLORS=CONTROL_COLORS,CONTROL_MARKERS=CONTROL_MARKERS,
        CONTROL_LABELS=CONTROL_LABELS,MaxNLocator=MaxNLocator)
    args = SimpleNamespace(seasonal_root=RESULTS/"Joint_main/seasonal",output=INTERMEDIATE/LABEL)
    args.output.mkdir(parents=True,exist_ok=True)
    configure_style()
    fig = render_monthly_support(builder,args,"direct")
    save_plot(fig,LABEL,dpi=DPI)


draw_figure_s18()

In [ ]:
# Figure_S19: edit this cell and rerun to update this figure.
def draw_figure_s19():
    LABEL = "Figure_S19"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import argparse

    import hashlib

    import json

    from pathlib import Path

    import matplotlib

    import matplotlib.pyplot as plt

    from matplotlib.font_manager import FontProperties, findfont

    from matplotlib.lines import Line2D

    from matplotlib.text import Text

    from PIL import Image

    import numpy as np

    import pandas as pd

    from matplotlib.colors import LinearSegmentedColormap

    from matplotlib.ticker import MaxNLocator

    FONT = "Times New Roman"

    WIDTH_MM, HEIGHT_MM = 180, 183

    CONTROL_ORDER = ("uniform_average", "personal_center", "ltpref_interval")

    CONTROL_LABELS = {
        "uniform_average": "Uniform average",
        "personal_center": "Personal centre",
        "ltpref_interval": "Personal range",
    }

    CONTROL_COLORS = {
        "uniform_average": "#7F7F7F",
        "personal_center": "#C8A36A",
        "ltpref_interval": "#7FA995",
    }

    CONTEXT_ORDER = ("Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019")

    CONTEXT_COLORS = dict(zip(CONTEXT_ORDER, ("#C78470", "#AF94AD", "#6F9EBC", "#7FA995")))

    CONTEXT_LABELS = dict(zip(CONTEXT_ORDER, ("Miami", "New Delhi", "Port Angeles", "Seattle")))

    CONTEXT_SHORT = ("Miami", "New\nDelhi", "Port\nAngeles", "Seattle")

    CONTROL_MARKERS = dict(zip(CONTROL_ORDER, ("o", "s", "^")))

    ALTERNATIVES = CONTROL_ORDER[1:]

    DATA_BOUNDS_MM = tuple(
        (x, y, 67.0, 35.0) for y in (125.0, 72.0, 19.0) for x in (19.0, 109.0)
    )

    def configure_style() -> dict[str, str]:
        fonts = {}
        for weight in ("normal", "bold"):
            prop = FontProperties(family=FONT, weight=weight)
            path = Path(findfont(prop, fallback_to_default=False))
            if FontProperties(fname=str(path)).get_name() != FONT:
                raise ValueError(f"Exact {FONT} {weight} is required")
            fonts[weight] = str(path)
        plt.rcParams.update({
            "font.family": FONT, "font.size": 11, "text.color": "black",
            "axes.labelsize": 11, "axes.labelcolor": "black", "axes.labelpad": 6,
            "axes.titlesize": 12, "axes.titleweight": "bold",
            "axes.spines.top": False, "axes.spines.right": False,
            "axes.edgecolor": "#777777", "axes.linewidth": 0.75,
            "xtick.labelsize": 11, "ytick.labelsize": 11,
            "xtick.major.pad": 4, "ytick.major.pad": 4,
            "xtick.color": "black", "ytick.color": "black",
            "legend.fontsize": 11, "legend.frameon": False,
            "figure.facecolor": "white", "savefig.facecolor": "white",
            "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
            "svg.hashsalt": "thermal-heterogeneity-stage8",
        })
        return fonts

    def six_panel_canvas(titles: list[str], *, control_key: bool = True, bounds=None, height=None):
        """Create equal data frames with separate title, axis and shared-key bands."""
        if len(titles) != 6:
            raise ValueError("The main result template has exactly six data panels")
        configure_style()
        page_height = height or HEIGHT_MM
        fig = plt.figure(figsize=(WIDTH_MM / 25.4, page_height / 25.4))
        axes = []
        for index, (title, panel_bounds) in enumerate(zip(titles, bounds or DATA_BOUNDS_MM)):
            x, y, width, height_mm = panel_bounds
            ax = fig.add_axes((x / WIDTH_MM, y / page_height, width / WIDTH_MM, height_mm / page_height))
            ax.set_title(f"({chr(97 + index)}) {title}", loc="left", pad=8)
            ax.grid(False)
            ax.set_axisbelow(True)
            axes.append(ax)
        if control_key:
            handles = [Line2D([], [], marker=marker, linestyle="none", color=CONTROL_COLORS[key],
                              markersize=5, label=CONTROL_LABELS[key])
                       for key, marker in zip(CONTROL_ORDER, ("o", "s", "^"))]
            fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.995),
                       ncol=3, handletextpad=0.3, columnspacing=0.9)
        return fig, axes

    def point_summary(ax, frame, metric, ylabel, *, scale=1.0, controls=CONTROL_ORDER,
                      signed=False, percentage=False):
        """Show every finite scenario plus medians/IQRs, never a selected favourable subset."""
        records = []
        offsets = np.linspace(-0.23, 0.23, len(controls))
        for context_index, context in enumerate(CONTEXT_ORDER):
            cases = frame[frame.context.eq(context)]
            if all(column in cases for column in ("composition_id", "layout")):
                for _, case in cases.groupby(["composition_id", "layout"], sort=True):
                    x_line, y_line = [], []
                    for control, offset in zip(controls, offsets):
                        part = case[case.control.eq(control)]
                        if len(part) == 1 and pd.notna(part.iloc[0][metric]):
                            x_line.append(context_index + offset)
                            y_line.append(float(part.iloc[0][metric]) * scale)
                    if len(y_line) > 1:
                        ax.plot(x_line, y_line, color="#BDB9B2", linewidth=.6, alpha=.65, zorder=1)
            for control, offset in zip(controls, offsets):
                selected = frame[frame.context.eq(context) & frame.control.eq(control)].copy()
                values = pd.to_numeric(selected[metric], errors="raise").to_numpy(dtype=float) * scale
                finite = values[np.isfinite(values)]
                if finite.size:
                    x = context_index + offset
                    jitter = np.linspace(-0.045, 0.045, finite.size) if finite.size > 1 else np.zeros(1)
                    ax.scatter(x + jitter, finite, s=12, color=CONTROL_COLORS[control], alpha=0.78,
                               marker=CONTROL_MARKERS[control], linewidths=.2, edgecolors="white", zorder=2)
                    low, median, high = np.quantile(finite, [0.25, 0.5, 0.75])
                    ax.errorbar(x, median, yerr=[[median - low], [high - median]],
                                fmt=CONTROL_MARKERS[control], color=CONTROL_COLORS[control],
                                markersize=4.5, linewidth=1.5, capsize=2.3, zorder=3)
                else:
                    low = median = high = np.nan
                records.append({"context": context, "control": control, "metric": metric,
                                "declared_cases": len(selected), "finite_cases": len(finite),
                                "q25_display": low, "median_display": median, "q75_display": high})
        if signed:
            ax.axhline(0, color="#A5A5A5", linewidth=0.65, zorder=0)
        if percentage:
            finite_values = pd.to_numeric(frame[metric], errors="raise").to_numpy(float) * scale
            finite_values = finite_values[np.isfinite(finite_values)]
            if len(finite_values):
                span = max(np.ptp(finite_values), 8)
                ax.set_ylim(max(-2, float(finite_values.min()) - span * .12),
                            min(102, float(finite_values.max()) + span * .12))
        if ax.get_position().width * ax.figure.get_figwidth() * 25.4 < 55:
            ax.set_xticks(range(4), list(CONTEXT_LABELS.values()), rotation=75, ha="right")
        else:
            ax.set_xticks(range(4), CONTEXT_SHORT)
        ax.set_xlim(-0.55, 3.55)
        ax.set_ylabel(ylabel)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.ticklabel_format(axis="y", style="plain", useOffset=False)
        return records
    def render_monthly_support(builder, args, mode):
        """Monthly summaries retain all qualified trajectories in the joint scope."""
        from matplotlib.lines import Line2D
        from matplotlib.patches import Patch
        table = pd.read_csv(args.seasonal_root / 'PERIOD_TOTALS.csv')
        selected = table[table.comparison_mode.eq(mode) & table.period_type.eq('month')
            & table.permission.eq('native_zone') & table.data_status.eq('qualified')].copy()
        if table.composition_id.nunique() != 6 or set(table.layout) != {'random_mixing', 'demand_ordered'}:
            raise ValueError('Seasonal evidence must retain the six-composition, two-layout design')
        selected['month_number'] = selected.period.astype(int)
        keys = ['context', 'composition_id', 'layout', 'control', 'run_id']
        for _, trajectory in selected.groupby(keys):
            if sorted(trajectory.month_number) != list(range(1, 13)) or abs(trajectory.hours.sum() - 8760) > 1e-6:
                raise ValueError('Every included annual trajectory must contain all twelve months and 8760 hours')
        selected.to_csv(args.output / (LABEL + '_monthly_source.csv'), index=False)
        height = 230.0
        figure = builder.plt.figure(figsize=(180 / 25.4, height / 25.4))
        axes, counts = [], []
        stack_keys = ['heating_electric_kwh', 'cooling_electric_kwh', 'fans_electric_kwh', 'gas_hhv_kwh']
        closure = (selected[stack_keys].sum(axis=1) - selected.purchased_site_kwh).abs().max()
        if closure > 1e-8:
            raise ValueError('Space-conditioning components must close to the fixed purchased-energy endpoint')
        stack_colors = ['#C78470', '#6F9EBC', '#C8A36A', '#AF94AD']
        for i, context in enumerate(builder.CONTEXT_ORDER):
            label = builder.CONTEXT_LABELS[context]
            left = figure.add_axes(np.array([19, 184-i*54, 67, 30]) / [180, height, 180, height])
            right = figure.add_axes(np.array([109, 184-i*54, 67, 30]) / [180, height, 180, height])
            axes.extend([left, right])
            left.set_title(f'({chr(97+2*i)}) {label}: service', loc='left', pad=8)
            right.set_title(f'({chr(98+2*i)}) {label}: energy', loc='left', pad=8)
            block = selected[selected.context.eq(context)]
            baseline = block[block.control.eq('uniform_average')].groupby('month_number')[stack_keys].mean()
            if len(baseline) == 12:
                right.stackplot(baseline.index.to_numpy(), *[baseline[k].to_numpy()/1000 for k in stack_keys],
                                colors=stack_colors, alpha=.85, linewidth=0, zorder=1)
            n = []
            for control in builder.CONTROL_ORDER:
                values = block[block.control.eq(control)]
                n.append(values[['composition_id', 'layout']].drop_duplicates().shape[0])
                counts.append(dict(context=context, control=control, qualified_conditions=n[-1], designed_conditions=12))
                if values.empty:
                    continue
                grouped = values.groupby('month_number')
                size = {'uniform_average': 4.5, 'personal_center': 3.7, 'ltpref_interval': 3.0}[control]
                for ax, metric, scale in [(left, 'mean_reference_mismatch_c', 1), (right, 'purchased_site_kwh', .001)]:
                    summary = grouped[metric].agg(['mean', 'min', 'max']) * scale
                    ax.fill_between(summary.index, summary['min'], summary['max'],
                                    color=builder.CONTROL_COLORS[control], alpha=.10, linewidth=0, zorder=2)
                    ax.plot(summary.index, summary['mean'], color=builder.CONTROL_COLORS[control],
                            marker=builder.CONTROL_MARKERS[control], linewidth=1.3, markersize=size,
                            markerfacecolor='white', markeredgewidth=.7, zorder=4)
            left.text(.02, .98, 'n = ' + ' / '.join(map(str, n)), transform=left.transAxes,
                      ha='left', va='top', fontsize=11)
            low, high = left.get_ylim()
            left.set_ylim(low, high + .25 * max(high-low, .1))
            left.set_ylabel('Mismatch (°C)')
            right.set_ylabel('Energy (MWh)')
            for ax in (left, right):
                ax.set_xlim(.8, 12.2)
                ax.set_xticks([1, 4, 7, 10], ['January', 'April', 'July', 'October'], rotation=30, ha='right')
                ax.grid(False)
                ax.yaxis.set_major_locator(builder.MaxNLocator(3))
        figure.legend([Line2D([], [], color=builder.CONTROL_COLORS[c], marker=builder.CONTROL_MARKERS[c],
                             linestyle='-', lw=1.3, markersize=4, markerfacecolor='white') for c in builder.CONTROL_ORDER],
                      [builder.CONTROL_LABELS[c] for c in builder.CONTROL_ORDER], loc='upper center',
                      bbox_to_anchor=(.5, .995), ncol=3, frameon=False, columnspacing=.9, handletextpad=.3)
        figure.legend([Patch(facecolor=color, alpha=.85) for color in stack_colors],
                      ['Heating', 'Cooling', 'Fans', 'Gas'], loc='lower center',
                      bbox_to_anchor=(.5, .005), ncol=4, frameon=False, columnspacing=.8, handletextpad=.3,
                      handlelength=1.1)
        mode_label = 'direct control operation' if mode == 'direct' else 'common-service annual selections'
        number = 18 if mode == 'direct' else 19
        caption = (
            f'Supplementary Figure S{number} | Monthly thermal service and energy: {mode_label}. '
            'All six fixed demand compositions and both random and demand-ordered layouts are '
            'included with zone-level commands in four building-weather contexts. '
            'a,c,e,g, Occupied population-weighted absolute temperature-reference mismatch. '
            'b,d,f,h, Purchased energy for heating, ventilation and air conditioning. Lines show '
            'means across engineering-qualified composition-layout conditions for each control, '
            'and shading gives their full minimum-to-maximum range. Counts in the service panels '
            'give the included conditions in legend order, from 12 designed conditions per control '
            'and context. The energy background stacks mean uniform-average heating electricity, '
            'cooling electricity, fan electricity and heating gas. Gas uses '
            'its higher heating value. Total lines include all these carriers. The same underlying '
            '8760-hour trajectories supply all twelve months. Common-service choices use the '
            'reference annual mean and fixed-tail limits; their monthly service outcomes can vary. '
            'Each context retains its own prototype energy scale and occupancy weights. '
            'Personal range denotes model-generated individual demand ranges.'
        )
        (RESULTS / 'captions' / f'{LABEL}.txt').write_text(caption, encoding='utf-8')
        pd.DataFrame(counts).to_csv(args.output / (LABEL + '_counts.csv'), index=False)
        return figure

    DPI = 600
    builder = SimpleNamespace(plt=plt,CONTEXT_ORDER=CONTEXT_ORDER,CONTEXT_LABELS=CONTEXT_LABELS,
        CONTROL_ORDER=CONTROL_ORDER,CONTROL_COLORS=CONTROL_COLORS,CONTROL_MARKERS=CONTROL_MARKERS,
        CONTROL_LABELS=CONTROL_LABELS,MaxNLocator=MaxNLocator)
    args = SimpleNamespace(seasonal_root=RESULTS/"Joint_main/seasonal",output=INTERMEDIATE/LABEL)
    args.output.mkdir(parents=True,exist_ok=True)
    configure_style()
    fig = render_monthly_support(builder,args,"equal_service")
    save_plot(fig,LABEL,dpi=DPI)


draw_figure_s19()

In [ ]:
def draw_s20():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S20"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    annual = read("expected/annual_decisions_reference.csv.gz")

    fig, a = canvas(
        [
            "Annual false acceptance",
            "Qualified annual choices",
            "Eligible annual actions",
            "Selected annual energy",
        ]
    )

    base = annual[annual.delta_c == 0]

    for j, (budget, d) in enumerate(base.groupby("service_fraction")):
        rows = []
        for lam, q in d.groupby("lambda"):
            mean = q.mean_selected_action_id.notna()
            false = q.mean_selected_full_reaudit_status.eq("mean_selected_false_accept")
            rows.append(
                (lam, int(false.sum()), int(mean.sum()), q.full_selected_action_id.notna().sum())
            )
        z = pd.DataFrame(rows, columns=["lambda", "false", "mean", "full"])
        # Status spelling is checked below against actual non-empty selections.
        if z["false"].sum() == 0:
            for k, (lam, q) in enumerate(d.groupby("lambda")):
                selected = q.mean_selected_action_id.notna()
                z.loc[k, "false"] = (
                    selected & ~q.mean_selected_full_eligible.fillna(False).astype(bool)
                ).sum()
        a[0].plot(
            z["lambda"],
            100 * z["false"] / z["mean"].replace(0, np.nan),
            marker="o",
            color=COLORS[j],
            label=f"Limit {budget:g}",
        )
        a[1].plot(z["lambda"], z["full"], marker="o", color=COLORS[j], label=f"{budget:g}")

    a[0].set_ylabel("False acceptance (%)")

    a[0].set_xlabel("Demand spread λ")

    a[0].legend(frameon=False, fontsize=11)

    a[1].set_ylabel("Selections among 36 cells")

    a[1].set_xlabel("Demand spread λ")

    for city, col in CONTEXT_COLORS.items():
        d = base[base.city == city].groupby("lambda").full_eligible_action_count.median()
        a[2].plot(d.index, d, marker="o", color=col, label=city)

    a[2].set_ylabel("Median eligible actions")

    a[2].set_xlabel("Demand spread λ")

    a[2].legend(loc="upper right", frameon=False, fontsize=11, handlelength=.6,
                handletextpad=.35, borderaxespad=.35, labelspacing=.35)

    q = base[
        (base["lambda"] == 1)
        & base.full_selected_site_kwh.notna()
        & base.mean_selected_site_kwh.notna()
    ]

    box(
        a[3],
        [
            (
                q.loc[q.city == c, "full_selected_site_kwh"]
                - q.loc[q.city == c, "mean_selected_site_kwh"]
            )
            / 1000
            for c in CONTEXTS
        ],
        ["Miami", "New\nDelhi", "Port\nAngeles", "Seattle"],
        "Full minus mean energy (MWh)",
        colors=list(CONTEXT_COLORS.values()),
    )

    a[3].axhline(0, color="gray", lw=0.7)

    manifest.append(
        save(
            fig,
            out,
            6,
            "Figure S6 | Annual service-budget and decision diagnostics. a, Fraction of mean-selected cases failing full-distribution review under four nested service budgets. b, Full-distribution selections among the thirty-six composition–seed–context cells per budget. c, Median full-qualified action count in each context. Terracotta, mauve, blue and green denote Miami, New Delhi, Port Angeles and Seattle, respectively. d, Full-minus-mean selected annual energy among bilateral selections at archive spread. Boxes show medians and interquartile ranges, whiskers extend to 1.5 interquartile ranges, and outliers remain in the underlying data. Budgets share the same underlying scenarios.",
            ["expected/annual_decisions_reference.csv.gz"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S06"
    final_stem = "Figure_S20"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S20.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s20()

In [ ]:
# Figure_S21: edit this cell and rerun to update this figure.
def draw_figure_s21():
    LABEL = "Figure_S21"
    import matplotlib.pyplot as plt
    plt.rcdefaults()
    import hashlib

    import json

    from pathlib import Path

    from typing import Callable

    import matplotlib.pyplot as plt

    import numpy as np

    import pandas as pd

    from matplotlib.axes import Axes

    from matplotlib.lines import Line2D

    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Polygon, Rectangle

    from thermal_heterogeneity.figures import (
        CATEGORY_STYLES,
        CONTEXT_COLORS,
        COUNT_CMAP,
        PALETTE,
        REPRESENTATION_COLORS,
        SIGNED_CMAP,
        ExportBundle,
        create_panel_figure,
        save_figure_bundle,
        style_panel,
    )

    CONTEXTS = ["Miami", "NewDelhi", "PortAngeles_small2019", "Seattle_small2019"]

    CONTEXT_LABELS = {
        "Miami": "Miami",
        "NewDelhi": "New Delhi",
        "PortAngeles_small2019": "Port Angeles",
        "Seattle_small2019": "Seattle",
    }

    CONTEXT_COLOURS = CONTEXT_COLORS

    LAMBDA_COLOURS = {
        0.0: PALETTE["blue"],
        0.5: PALETTE["sky"],
        1.0: PALETTE["orange"],
        1.5: PALETTE["vermillion"],
    }

    LAMBDA_MARKERS = {0.0: "o", 0.5: "s", 1.0: "^", 1.5: "D"}

    CLASS_COLOURS = {
        "mean-safe": PALETTE["sky"],
        "distribution-required": PALETTE["vermillion"],
        "finite-library-unresolved": PALETTE["light_grey"],
    }

    def _sha256(path: Path) -> str:
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def verify_panel_inputs(panel_directory: Path | str, figure_number: int) -> dict[str, str]:
        """Verify manifest hashes and critical frozen denominators for one figure."""

        panel_directory = Path(panel_directory)
        manifest_path = panel_directory / "PANEL_DATA_V2_MANIFEST.csv"
        manifest = pd.read_csv(manifest_path)
        selected = manifest[manifest["panel_id"].str.startswith(f"Fig{figure_number}")]
        if len(selected) != 8:
            raise ValueError(f"Figure {figure_number} requires exactly eight manifest rows")
        hashes: dict[str, str] = {}
        for row in selected.itertuples(index=False):
            path = panel_directory / row.relative_path
            observed = _sha256(path)
            if observed != row.sha256:
                raise ValueError(f"panel hash mismatch: {path.name}")
            hashes[row.panel_id] = observed

        if figure_number == 4:
            counts = pd.read_csv(panel_directory / "Fig4c.csv")
            full = counts[counts["representation"] == "full"].sort_values("lambda")
            mean = counts[counts["representation"] == "mean"].sort_values("lambda")
            if full["selected_count"].tolist() != [128, 120, 74, 21]:
                raise ValueError("Fig4c full-demand counts differ from the freeze")
            if mean["selected_count"].tolist() != [128, 128, 128, 128]:
                raise ValueError("Fig4c mean-demand counts differ from the freeze")
        elif figure_number == 5:
            case = json.loads((panel_directory / "Fig5bc_case_lock.json").read_text(encoding="utf-8"))
            if not case["mean_full_same_action"]:
                raise ValueError("Fig5b/c must show the same frozen action")
            boundary = pd.read_csv(panel_directory / "Fig5h.csv")
            feb = boundary[boundary["month"] == 2]
            if set(feb["reference_positive_selected_count"]) != {0}:
                raise ValueError("February central reference must remain 0/144")
            if set(feb["month_grid_positive_selected_count"]) != {210}:
                raise ValueError("February full grid must remain 210/2880")
        elif figure_number == 6:
            phase = pd.read_csv(panel_directory / "Fig6c.csv").set_index("classification")
            expected = {
                "mean-safe": 275,
                "distribution-required": 331,
                "finite-library-unresolved": 402,
            }
            if phase["count"].to_dict() != expected:
                raise ValueError("Fig6c phase counts differ from the freeze")
            gate = pd.read_csv(panel_directory / "Fig6f.csv")
            retained = gate["true_positive_candidates"] / gate["full_eligible_candidates"]
            covered = gate["full_available_and_certified_decisions"] / gate["full_available_decisions"]
            if not np.allclose(retained, gate["true_action_retention"], rtol=0, atol=1e-12):
                raise ValueError("Fig6f action-retention denominators do not reproduce")
            if not np.allclose(covered, gate["decision_coverage"], rtol=0, atol=1e-12):
                raise ValueError("Fig6f decision-coverage denominators do not reproduce")
        else:
            raise ValueError("only Figures 4, 5 and 6 are implemented here")
        return hashes

    def _panel(axis: Axes, letter: str, title: str, *, grid_axis: str | None = "y") -> None:
        style_panel(axis, letter, title, grid_axis=grid_axis)

    def _blank(axis: Axes, letter: str, title: str) -> None:
        _panel(axis, letter, title, grid_axis=None)
        axis.set_xlim(0, 1)
        axis.set_ylim(0, 1)
        axis.set_xticks([])
        axis.set_yticks([])
        for spine in axis.spines.values():
            spine.set_visible(False)

    def _box(
        axis: Axes,
        xy: tuple[float, float],
        width: float,
        height: float,
        text: str,
        *,
        facecolor: str,
        edgecolor: str = "#555555",
        hatch: str | None = None,
    ) -> None:
        patch = FancyBboxPatch(
            xy,
            width,
            height,
            boxstyle="round,pad=0.012,rounding_size=0.018",
            facecolor=facecolor,
            edgecolor=edgecolor,
            linewidth=0.8,
            hatch=hatch,
        )
        axis.add_patch(patch)
        axis.text(xy[0] + width / 2, xy[1] + height / 2, text, ha="center", va="center")

    def _arrow(axis: Axes, start: tuple[float, float], end: tuple[float, float]) -> None:
        axis.add_patch(
            FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=10, linewidth=0.8, color="#555555")
        )

    def _child(parent: Axes, bounds: list[float]) -> Axes:
        axis = parent.inset_axes(bounds)
        axis.tick_params(width=0.65, length=2.5, pad=1, color="#333333")
        for side in ("top", "right"):
            axis.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            axis.spines[side].set_linewidth(0.65)
            axis.spines[side].set_color("#333333")
        axis.grid(True, axis="y", color="#D9D9D9", linewidth=0.35, zorder=0)
        return axis

    def _parent_for_children(axis: Axes, letter: str, title: str) -> None:
        _blank(axis, letter, title)

    def _critical(text, name: str):
        """Tag a layout-critical text artist for figure-specific geometry tests."""

        text.set_gid(f"figures-4-6-layout-critical:{name}")
        return text

    def _inset_panel_axis(
        axis: Axes,
        *,
        left: float = 0.0,
        right: float = 0.0,
        bottom: float = 0.0,
        top: float = 0.0,
    ) -> None:
        """Move one top-level data rectangle within its locked panel cell."""

        position = axis.get_position()
        axis.set_position(
            [
                position.x0 + left,
                position.y0 + bottom,
                position.width - left - right,
                position.height - bottom - top,
            ]
        )

    def _quantiles(values: pd.Series | np.ndarray) -> tuple[float, float, float]:
        clean = np.asarray(values, dtype=float)
        clean = clean[np.isfinite(clean)]
        if len(clean) == 0:
            return np.nan, np.nan, np.nan
        return tuple(float(x) for x in np.quantile(clean, [0.25, 0.5, 0.75]))

    def _simple_density(values: np.ndarray, grid: np.ndarray) -> np.ndarray:
        values = np.asarray(values, dtype=float)
        if np.ptp(values) < 1e-10:
            width = 0.10
        else:
            width = max(0.13, 0.25 * np.std(values, ddof=1))
        density = np.exp(-0.5 * ((grid[:, None] - values[None, :]) / width) ** 2).sum(axis=1)
        area = np.trapz(density, grid)
        return density / area if area > 0 else density

    def build_figure_4(panel_directory: Path | str):
        """Build Figure 4: spread and annual service-qualified action space."""

        panel_directory = Path(panel_directory)
        verify_panel_inputs(panel_directory, 4)
        figure, axes = create_panel_figure()
        a, b, c, d, e, f, g, h = axes.ravel()

        # a: same action, different demand representations.
        _blank(a, "a", "Two service checks")
        _box(a, (0.03, 0.37), 0.23, 0.26, "Fixed action\nand trajectory", facecolor="#F2F2F2")
        _arrow(a, (0.27, 0.50), (0.39, 0.72))
        _arrow(a, (0.27, 0.50), (0.39, 0.28))
        _box(a, (0.40, 0.58), 0.27, 0.25, "Mean demand\nservice check", facecolor="#E6E6E6")
        _box(a, (0.40, 0.15), 0.27, 0.25, "Full demand\nservice check", facecolor="#DCEEF8")
        _arrow(a, (0.68, 0.705), (0.78, 0.705))
        _arrow(a, (0.68, 0.275), (0.78, 0.275))
        _critical(a.text(0.88, 0.705, "Pass / fail", va="center", ha="center", color=PALETTE["dark_grey"]), "4a-mean-outcome")
        _critical(a.text(0.88, 0.275, "Pass / fail", va="center", ha="center", color=PALETTE["blue"]), "4a-full-outcome")

        # b: spread manipulation at a fixed centre.
        spread = pd.read_csv(panel_directory / "Fig4b.csv")
        xmin = spread["perturbed_demand_c"].min() - 0.5
        xmax = spread["perturbed_demand_c"].max() + 0.5
        grid = np.linspace(xmin, xmax, 320)
        line_styles = {0.0: "-", 0.5: "--", 1.0: "-.", 1.5: ":"}
        for lam in [0.0, 0.5, 1.0, 1.5]:
            vals = spread.loc[spread["lambda"] == lam, "perturbed_demand_c"].to_numpy()
            density = _simple_density(vals, grid)
            b.plot(grid, density, color=LAMBDA_COLOURS[lam], linestyle=line_styles[lam], label=f"λ={lam:g}")
            q10, q90 = np.quantile(vals, [0.1, 0.9])
            b.plot([q10, q90], [0.003 + lam * 0.002] * 2, color=LAMBDA_COLOURS[lam], linewidth=2.0)
        common_mean = float(spread["perturbed_demand_c"].groupby(spread["lambda"]).mean().mean())
        b.axvline(common_mean, color="#333333", linewidth=0.7)
        _critical(b.text(0.98, 0.57, f"shared mean = {common_mean:.1f} °C", transform=b.transAxes, ha="right", va="top"), "4b-shared-mean")
        b.set_xlim(xmin, xmax + 0.8)
        b.set_ylim(0, b.get_ylim()[1] * 1.12)
        b.set_xticks([15, 20, 25, 30, 35])
        b.set_xlabel("Demand temperature (°C)")
        b.set_ylabel("Normalized density")
        _panel(b, "b", "Fixed-mean demand spread")
        b.legend(frameon=False, ncol=2, loc="upper left", handlelength=1.5, columnspacing=0.8)
        _inset_panel_axis(b, left=0.030, bottom=0.010)

        # c: full-demand coverage falls while the mean line is flat.
        coverage = pd.read_csv(panel_directory / "Fig4c.csv")
        for rep, label in [("full", "Full demand"), ("mean", "Mean")]:
            rows = coverage[coverage["representation"] == rep].sort_values("lambda")
            style = CATEGORY_STYLES["full_demand" if rep == "full" else "mean"]
            c.plot(rows["lambda"], rows["selected_count"], label=label, **style)
            for row in rows.to_dict("records"):
                offset = -15 if rep == "full" and row["lambda"] <= 0.5 else (7 if rep == "full" else 11)
                if rep == "mean" and row["lambda"] <= 0.5:
                    offset = 13
                _critical(c.annotate(f"{row['selected_count']}/144", (row["lambda"], row["selected_count"]), xytext=(0, offset), textcoords="offset points", ha="center", va="center", clip_on=True), f"4c-{rep}-{row['lambda']}")
        c.set_xlim(-0.20, 1.70)
        c.set_ylim(0, 172)
        c.set_xticks([0, 0.5, 1, 1.5])
        c.set_yticks([0, 50, 100, 150])
        c.set_ylabel("Selected cells (count)")
        _panel(c, "c", "Qualified selections")
        c.legend(frameon=False, loc="lower left", ncol=2, handlelength=1.5)

        # d: mean-selected actions that fail full review.
        false_accept = pd.read_csv(panel_directory / "Fig4d.csv").sort_values("lambda")
        bars = d.bar(false_accept["lambda"], false_accept["false_accept_count"], width=0.28, color=PALETTE["vermillion"], edgecolor="white")
        for bar, row in zip(bars, false_accept.itertuples(index=False)):
            d.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 4, f"{row.false_accept_count}/128\n{100*row.false_accept_rate:.0f}%", ha="center", va="bottom")
        d.set_xlim(-0.25, 1.75)
        d.set_ylim(0, 150)
        d.set_xticks([0, 0.5, 1, 1.5])
        d.text(0.98, 0.04, "λ", transform=d.transAxes, ha="right", va="bottom")
        d.set_ylabel("False accepts (count)")
        _panel(d, "d", "Mean false acceptance")
        _inset_panel_axis(d, left=0.030, bottom=0.008)

        # e: action-set contraction before optimization.
        eligible = pd.read_csv(panel_directory / "Fig4e.csv")
        groups = [eligible.loc[eligible["lambda"] == lam, "full_eligible_action_count"].to_numpy() for lam in [0, 0.5, 1, 1.5]]
        boxes = e.boxplot(groups, positions=[0, 0.5, 1, 1.5], widths=0.28, patch_artist=True, showfliers=False, whis=(10, 90))
        for patch, color in zip(boxes["boxes"], [LAMBDA_COLOURS[x] for x in [0, 0.5, 1, 1.5]]):
            patch.set_facecolor(color)
            patch.set_alpha(0.35)
            patch.set_edgecolor(color)
        medians_for_display = []
        zeros_for_display = []
        for index, (lam, values) in enumerate(zip([0, 0.5, 1, 1.5], groups)):
            jitter = np.linspace(-0.11, 0.11, len(values))
            e.scatter(lam + jitter, values, s=5, color=LAMBDA_COLOURS[lam], alpha=0.52, linewidths=0)
            unresolved = int((values == 0).sum())
            medians_for_display.append(f"{np.median(values):.0f}")
            zeros_for_display.append(str(unresolved))
        e.set_xlim(-0.25, 1.75)
        e.set_ylim(-3, 82)
        e.set_xticks([0, 0.5, 1, 1.5])
        e.set_yticks([0, 20, 40, 60, 80])
        e.set_ylabel("Full-eligible actions (count)")
        _critical(e.text(0.03, 0.96, f"Median  {' · '.join(medians_for_display)}", transform=e.transAxes, ha="left", va="top"), "4e-medians")
        _critical(e.text(0.03, 0.87, f"Zero      {' · '.join(zeros_for_display)}", transform=e.transAxes, ha="left", va="top"), "4e-zeros")
        _panel(e, "e", "Eligible action counts")

        # f: energy effects only within common-service selections.
        energy = pd.read_csv(panel_directory / "Fig4f.csv")
        f.axvline(0, color="#333333", linewidth=0.65)
        y_base = np.arange(len(CONTEXTS))[::-1]
        offsets = {0.0: -0.20, 0.5: 0.0, 1.5: 0.20}
        for lam in [0.0, 0.5, 1.5]:
            for y, context in zip(y_base, CONTEXTS):
                rows = energy[(energy["context"] == context) & (energy["lambda"] == lam)]
                matched = rows[rows["bilateral_matched"]]
                q1, med, q3 = _quantiles(matched["delta_energy_mwh"])
                yy = y + offsets[lam]
                if np.isfinite(med):
                    interval = f.plot([q1, q3], [yy, yy], color=LAMBDA_COLOURS[lam], linewidth=1.5)[0]
                    interval.set_gid(f"figure-4f-data-interval:{context}:{lam}")
                    marker = f.scatter(
                        [med],
                        [yy],
                        color=LAMBDA_COLOURS[lam],
                        marker=LAMBDA_MARKERS[lam],
                        s=22,
                        zorder=3,
                    )
                    marker.set_gid(f"figure-4f-data-marker:{context}:{lam}")
        denominator_rows = []
        for context in CONTEXTS:
            parts = []
            for lam in [0.0, 0.5, 1.5]:
                rows = energy[(energy["context"] == context) & (energy["lambda"] == lam)]
                matched = int(rows["bilateral_matched"].sum())
                parts.append(f"{matched}/{len(rows)-matched}")
            denominator_rows.append(" · ".join(parts))
        f.set_yticks(y_base, ["MIA", "NDL", "PTA", "SEA"])
        f.set_xlim(-70, 172)
        f.set_xticks([-60, -30, 0, 30, 60])
        f.set_xlabel("Energy change (MWh yr⁻¹)")
        f.set_ylabel("")
        f.axvline(50, color="#D2D2D2", linewidth=0.65)
        for y, label in zip(y_base, denominator_rows):
            _critical(f.text(168, y, label, ha="right", va="center"), f"4f-denominator-{y}")
        _critical(f.text(168, 4.00, "m/u by λ: 0 · .5 · 1.5", ha="right", va="bottom"), "4f-denominator-header")
        f.set_ylim(-0.5, 4.75)
        _panel(f, "f", "Matched annual energy", grid_axis="x")
        legend = f.legend(
            handles=[Line2D([], [], color=LAMBDA_COLOURS[x], marker=LAMBDA_MARKERS[x], linestyle="") for x in [0, 1.5, 0.5]],
            labels=["λ0", "λ1.5", "λ.5"],
            frameon=False,
            loc="upper left",
            ncol=2,
            handlelength=1.0,
            columnspacing=0.7,
            handletextpad=0.3,
        )
        legend.set_gid("figure-4f-legend")
        _inset_panel_axis(f, left=0.040, bottom=0.010)

        # g: CO2 forest plus a signed carrier heatmap inset.
        response = pd.read_csv(panel_directory / "Fig4g.csv")
        _parent_for_children(g, "g", "Carbon and carriers")
        main = _child(g, [0.05, 0.18, 0.37, 0.66])
        carrier = _child(g, [0.64, 0.18, 0.34, 0.66])
        main.axvline(0, color="#333333", linewidth=0.65)
        row_positions: list[float] = []
        row_labels: list[str] = []
        carrier_matrix: list[list[float]] = []
        for context_index, context in enumerate(CONTEXTS):
            base = (len(CONTEXTS) - 1 - context_index) * 3
            for lam_index, lam in enumerate([0.0, 0.5, 1.5]):
                rows = response[(response["context"] == context) & (response["lambda"] == lam) & response["bilateral_matched"]]
                y = base + (2 - lam_index)
                q1, med, q3 = _quantiles(rows["delta_co2_t"])
                main.plot([q1, q3], [y, y], color=LAMBDA_COLOURS[lam], linewidth=1.4)
                main.scatter([med], [y], s=18, color=LAMBDA_COLOURS[lam], marker=LAMBDA_MARKERS[lam], zorder=3)
                row_positions.append(y)
                context_code = {"Miami": "MIA", "NewDelhi": "NDL", "PortAngeles_small2019": "PTA", "Seattle_small2019": "SEA"}[context]
                row_labels.append(context_code if lam == 0.5 else "")
                carrier_matrix.append([
                    float(rows["heating_electricity_delta_mwh"].median()),
                    float(rows["cooling_electricity_delta_mwh"].median()),
                    float(rows["fan_delta_mwh"].median()),
                    float(rows["gas_hhv_delta_mwh"].median()),
                ])
        main.set_yticks(row_positions, row_labels)
        main.set_xticks([-10, 0, 10, 20, 30])
        main.set_xlabel("ΔCO₂ (tCO₂ yr⁻¹)")
        main.grid(True, axis="x", color="#D9D9D9", linewidth=0.35)
        matrix = np.asarray(carrier_matrix)[np.argsort(row_positions)]
        vmax = max(1.0, float(np.nanquantile(np.abs(matrix), 0.95)))
        carrier.imshow(matrix, cmap=SIGNED_CMAP, vmin=-vmax, vmax=vmax, origin="lower", aspect="auto")
        carrier.set_xticks(range(4), ["Heat", "Cool", "Fan", "Gas"], rotation=45, ha="right")
        carrier.set_yticks([])
        carrier.set_xlabel("Δ carrier (MWh yr⁻¹)")
        carrier.grid(False)
        carrier.text(0.98, 0.98, f"±{vmax:.0f}", transform=carrier.transAxes, ha="right", va="top")
        g.text(0.255, 0.89, "Carbon", ha="center", va="center")
        g.text(0.785, 0.89, "Carriers", ha="center", va="center")

        # h: context heatmaps of non-additive eligibility interaction.
        interaction = pd.read_csv(panel_directory / "Fig4h.csv")
        _parent_for_children(h, "h", "Centre×spread interaction")
        child_bounds = [[0.08, 0.66, 0.38, 0.18], [0.58, 0.66, 0.38, 0.18], [0.08, 0.42, 0.38, 0.18], [0.58, 0.42, 0.38, 0.18]]
        medians = interaction.groupby(["context", "lambda", "delta_c"], as_index=False)["eligible_count_interaction"].median()
        vlim = max(1.0, float(medians["eligible_count_interaction"].abs().max()))
        for panel_index, (context, bounds) in enumerate(zip(CONTEXTS, child_bounds)):
            sub = medians[medians["context"] == context].pivot(index="lambda", columns="delta_c", values="eligible_count_interaction").sort_index().sort_index(axis=1)
            child = _child(h, bounds)
            child.imshow(sub.to_numpy(), cmap=SIGNED_CMAP, vmin=-vlim, vmax=vlim, origin="lower", aspect="auto")
            child.set_xticks(range(len(sub.columns)), [f"{x:g}" for x in sub.columns])
            child.set_yticks(range(len(sub.index)), [f"{x:g}" for x in sub.index])
            if panel_index < 2:
                child.set_xticklabels([])
            if panel_index % 2 == 1:
                child.set_yticklabels([])
            child.grid(False)
            child.text(0.02, 0.98, CONTEXT_LABELS[context], transform=child.transAxes, va="top")
        gradient = np.linspace(-vlim, vlim, 256)[None, :]
        h.imshow(
            gradient,
            cmap=SIGNED_CMAP,
            vmin=-vlim,
            vmax=vlim,
            aspect="auto",
            extent=(0.20, 0.80, 0.23, 0.255),
            transform=h.transAxes,
            origin="lower",
        )
        h.add_patch(Rectangle((0.20, 0.23), 0.60, 0.025, fill=False, edgecolor="#555555", linewidth=0.65))
        h.text(0.18, 0.242, f"−{vlim:.0f}", ha="right", va="center")
        h.text(0.50, 0.242, "0", ha="center", va="center")
        h.text(0.82, 0.242, f"+{vlim:.0f}", ha="left", va="center")
        _critical(h.text(0.50, 0.010, "Eligibility ≠0: 1,261/1,728\nEnergy ≠0: 545/585; unresolved 1,143", ha="center", va="bottom"), "4h-denominators")
        return figure

    CAPTIONS = {
        4: (
            "Figure 4 | Demand spread contracts service-qualified annual action space. "
            "a, The same fixed action is audited under a compressed mean and the complete demand distribution. "
            "b, Fixed-centre spread perturbations for the pre-specified Miami multinational composition; horizontal segments show P10-P90. "
            "c, Full-demand and mean-based annual selections among 144 design cells at each spread multiplier. "
            "d, Mean-selected actions that fail full-demand review (denominator 128). "
            "e, Full-service eligible-action counts across all 144 cells per multiplier, retaining zero-action cells. "
            "f, Median and interquartile annual-energy change for bilateral full-service selections; the right-hand table gives matched/unmatched cells for λ=0, 0.5 and 1.5. "
            "g, Carbon intervals and median signed carrier changes for the same bilateral comparisons; colours and markers retain the λ encoding from panel f. "
            "h, Median non-additive eligibility interaction across contexts on one shared diverging action-count scale; selected-energy interactions are resolved in 585 of 1,728 cells."
        ),
        5: (
            "Figure 5 | Demand heterogeneity contracts service-qualified flexibility under event and recovery constraints. "
            "a, Pre-specified 2-h event, 48-h recovery and 50-h energy-accounting contract. "
            "b, Baseline and perturbed HVAC power for the same pre-specified action; the panel does not compare mean- and full-selected actions. "
            "c, Zone temperature envelope, occupancy-weighted temperature and full-distribution mean and fixed-27 tail-service trajectories for that case; horizontal lines are the fixed mean- and tail-service budgets. "
            "d, July cells with a service-qualified positive response under full and mean demand. "
            "e, Mean-positive selections that fail full-demand review (denominator 122). "
            "f, Selected response for 12 common full-service cells at spread multipliers 1 and 1.5. "
            "g, Response, recovery peak, recovery electricity and 50-h net-saving changes for those 12 cells; each header reports decrease, tie and increase counts. "
            "h, Positive full-service selections across demand location and spread; the February central reference (0/144) is distinct from the complete February grid (210/2,880)."
        ),
        6: (
            "Figure 6 | Protected anonymous demand summaries recover service-qualified decisions beyond a single mean. "
            "a, Schematic conditions governing whether an average is sufficient; the vertical axis is the service-budget multiplier, for which larger values permit looser service-error allowances, and the drawn regions do not define an empirical monotonic boundary. "
            "b, Exact sampled design cells classified across four contexts without spatial interpolation. "
            "c, Counts among all 1,008 design units and the 606 resolvable units. "
            "d, False-safe and false-required errors from the lowest-error 1.059 °C in-sample scanned diagnostic threshold; it did not meet the prespecified total-error, worst-context-error, sensitivity and specificity standards, and denominators are context-specific. "
            "e, Median and interquartile service-reconstruction errors for mean, three-quantile and five-quantile summaries. "
            "f, Held-out certification trade-off; marker shape identifies the anonymous summary and marker size encodes decision coverage. The highlighted q5 plus training-only pooled-q95 rule has zero observed false acceptance, retains 14,709/18,766 true-qualified candidates and covers 542/606 resolvable decisions in the context-seed split. "
            "g, Local-update learning curves; points and error bars show updated scores and their hierarchical-bootstrap 95% intervals. In the paired improvement analysis at 40 observations, both city-held-out improvement intervals are above zero, whereas both source-held-out improvement intervals cross zero. "
            "h, Anonymous-summary workflow from archive prior to qualified energy, carbon and flexibility reporting; anonymity of the summaries is not a formal privacy guarantee."
        ),
    }

    ALT_TEXT = {
        4: (
            "Eight-panel figure comparing one fixed annual action library under mean and full thermal-demand service checks. "
            "At fixed mean demand, increasing spread λ contracts full-qualified selections from 128/144 to 21/144 while mean-based selections stay 128/144; false acceptance rises from 0/128 to 128/128. "
            "Eligible action counts collapse, and matched annual-energy and carbon effects differ by context and carrier. The matched/unmatched comparison contains 169/263 cells. "
            "Centre×spread eligibility interactions are non-zero in 1,261/1,728 cells; energy interactions are resolved for 585/1,728 cells, with 1,143 unresolved."
        ),
        5: (
            "Eight-panel figure comparing service-qualified demand-response flexibility with a pre-specified event, recovery and energy-accounting contract. "
            "Panels b and c show baseline and perturbed physical trajectories for the same pre-specified action, with 28.3 kW response, 13.1 kW rebound and 30.3 kWh net saving; they do not compare mean- and full-selected actions. "
            "Across July design cells, full-qualified positive selections fall from 122/144 to 12/144 as λ rises, while mean-based selections remain 122/144 and false acceptance reaches 115/122. "
            "Only 12 cells are common at λ=1 and 1.5; five lose response. February has 0/144 positive selections at the central reference but 210/2,880 across the complete grid."
        ),
        6: (
            "Eight-panel figure testing anonymous demand summaries and a protected qualification rule. "
            "Panel a is a schematic separation of mean-safe, distribution-required and unresolved regimes by demand heterogeneity and service-budget multiplier; larger multipliers permit looser service-error allowances, and the drawn regions are not an empirical monotonic boundary. "
            "Panel b maps the three regimes for the exact sampled cells in Miami, New Delhi, Port Angeles and Seattle. "
            "Panel c shows that, of 1,008 design units, 275 are mean-safe, 331 require the demand distribution and 402 are unresolved; the resolved denominator is 606. "
            "Panel d shows that the lowest-error 1.059 °C in-sample scanned diagnostic threshold makes 92/606 errors and does not meet the prespecified performance standards. "
            "Panel e shows lower median tail-service and eligibility errors for three- and five-quantile summaries than for the mean alone. "
            "Panel f shows that, in the context-seed holdout, q5 plus a training-only pooled-q95 margin yields zero observed false accepts, retains 14,709/18,766 true-qualified candidates and covers 542/606 resolvable decisions; this is held-out evidence, not a universal guarantee. "
            "Panel g shows local-feedback learning curves for dimensionless normalized ordinal RPS and tail-share MAE. Their points and error bars are updated scores and their hierarchical-bootstrap 95% intervals. In the paired improvement analysis at 40 city-held-out observations, the RPS and tail-share error improvement intervals are above zero, whereas at 40 source-held-out observations both corresponding improvement intervals cross zero, including the tail-share error interval. "
            "Panel h traces an anonymous-summary qualified-reporting workflow from an archive prior through local updating, five anonymous quantiles and a protected q95 qualification rule to energy, carbon, flexibility and recovery reporting; it makes no formal privacy guarantee."
        ),
    }
    from render_revised_main_figures import restructure, readable_display_labels
    from thermal_heterogeneity.figures import apply_figure_style, apply_data_axis_style
    DPI = 600
    apply_figure_style()
    PANEL_ROOT = RESULTS / "panel_data"
    figure = build_figure_4(PANEL_ROOT)
    figure._stage7_panel_root = PANEL_ROOT
    figure._stage7_decision_value = RESULTS / "decision_value/MAIN_FIG6G_Q5_REGRET.csv"
    restructure(figure, 4)
    readable_display_labels(figure, 4)
    apply_data_axis_style(figure, map_axes=figure.axes[:2] if 4 == 1 else [])
    (RESULTS / "captions/Figure_S21.txt").write_text('Supplementary Figure S21 | Annual service qualification across demand-spread scenarios. a, Demand distributions at fixed mean, with equal-height densities and 10th–90th percentile segments. b, Mean-based and full-demand selections among 144 cells per spread. c, Mean-selected plans failing full-demand review. d, Qualified-plan counts: dots are cells, boxes show medians/interquartile ranges and whiskers 10th–90th percentiles. e, Median/interquartile annual energy changes among matched full-service selections. f, Corresponding carbon and carrier changes. g, Qualified-action count interactions relative to additive centre and spread effects. Blue/terracotta indicate negative/positive values, with separate shared scales for f,g. Full review applies population-mean and fixed-27 tail mismatch limits. λ is the assigned spread multiplier. Matched counts and references are given in Supplementary Section S10.', encoding="utf-8")
    save_plot(figure, LABEL, dpi=DPI, exact_pixels=True)


draw_figure_s21()

In [ ]:
def draw_s22():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S22"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    severity = pd.read_csv(args.science_audit / "false_accept_exceedance_cells.csv")

    fig, a = canvas(
        [
            "Service exceedance",
            "Substitute energy penalty",
        ]
    )

    exceed = next(c for c in severity if c in ["maximum_positive_excess_c"])

    a[0].hist(severity[exceed], bins=20, color=COLORS[1], alpha=0.75)

    a[0].set_xlabel("Maximum service-limit exceedance (°C)")

    a[0].set_ylabel("False-accepted selections")

    cost_col = next(c for c in severity if c in ["full_alternative_energy_penalty_percent"])

    z = np.sort(severity[cost_col].dropna().to_numpy())

    a[1].plot(z, np.arange(1, len(z) + 1) / len(z), color=COLORS[0])

    a[1].set_xlabel("Substitute energy increase (%)")

    a[1].set_ylabel("Cumulative fraction")

    manifest.append(
        save(
            fig,
            out,
            12,
            "Figure S12 | Annual service exceedance and substitute energy cost. a, Maximum excess over the mean or tail service budget in the 117 false-accepted annual selections at archive spread. b, Cumulative distribution of the annual energy increase for the full-qualified substitute in sixty-three cases with an available substitute. Fifty-four cases without such a substitute are excluded from b.",
            ["science_audit/false_accept_exceedance_cells.csv"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S12"
    final_stem = "Figure_S22"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (
        'Supplementary Figure S22 | Archive-scenario diagnostic of annual service exceedance '
        'and substitute energy cost. This diagnostic uses the ASHRAE-derived regional and pooled archive '
        'demand scenarios, separately from the six fixed demand compositions of the main '
        'control comparison. a, Maximum excess over the mean or fixed-tail service limit '
        'among 117 annual choices meeting the mean-demand criterion but failing the full-demand '
        'criteria at archive-derived spread. b, Cumulative distribution of annual energy '
        'increase for a full-demand-qualified substitute in 63 cases with an available '
        'substitute. The remaining 54 cases have no qualified substitute in the tested '
        'candidate library. Temperature limits use the same reference and full-demand '
        'evaluation within each scenario.'
    )
    (RESULTS / "captions/Figure_S22.txt").write_text(caption, encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s22()


In [ ]:
def draw_s23():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S23"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    dynamic = read("expected/dynamic_decisions_reference.csv.gz")

    fig, a = canvas(CONTEXTS)

    for ax, city in zip(a, CONTEXTS):
        for j, month in enumerate([2, 7]):
            d = dynamic[
                (dynamic.city == city) & (dynamic.event_month == month) & (dynamic["lambda"] == 1)
            ]
            z = d.groupby("delta_c").full_selected_event_average_reduction_kw.count()
            ax.plot(
                z.index,
                z,
                marker="o",
                color=[SEASON_COLORS["cold"], SEASON_COLORS["warm"]][j],
                label=["February", "July"][j],
            )
        ax.set_xlabel("Centre shift (°C)")
        ax.set_ylabel("Positive full selections / 36")
        ax.set_ylim(-1, 37)

    a[0].legend(frameon=False)

    manifest.append(
        save(
            fig,
            out,
            7,
            "Figure S7 | Seasonal response availability by demand centre. a–d, Positive event selections satisfying population-mean, fixed-tail and recovery service limits at archive-derived spread in Miami, New Delhi, Port Angeles and Seattle. Blue/terracotta denote February/July. Each centre–context–month point contains nine composition–seed scenarios and four nested budgets, for thirty-six design cells. Events last two hours with forty-eight consecutive hours of recovery.",
            ["expected/dynamic_decisions_reference.csv.gz"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S07"
    final_stem = "Figure_S23"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S23.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s23()

In [ ]:
def draw_s24():
    # Editable display settings. Data filters and panel commands are below.
    EXPORT_DPI = 300
    SIZE_SCALE = (1.0, 1.0)  # Width and height multipliers; 1 preserves the paper.

    import hashlib
    import json
    import re
    import shutil
    from pathlib import Path
    from types import SimpleNamespace
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
    from render_supplementary import (
        canvas, box, configure, save as style_and_save,
        COLORS, CONTEXTS, CONTEXT_COLORS, G_COLORS,
        POPULATION_COLORS, SEASON_COLORS,
    )
    from thermal_heterogeneity.figures import (
        CONTROL_COLORS, COUNT_CMAP, DEMAND_CMAP, GRIFFITHS_COLORS,
        PALETTE, REPRESENTATION_COLORS, VARIANCE_COLORS,
    )

    plt.rcdefaults()
    configure()
    b = RESULTS / "legacy_analysis"
    p = RESULTS / "panel_data"
    out = INTERMEDIATE / "S24"
    out.mkdir(parents=True, exist_ok=True)
    args = SimpleNamespace(
        science_audit=RESULTS / "support/science_audit",
        spatial_audit=RESULTS / "support/spatial_audit",
        land=ROOT / "src/thermal_heterogeneity/assets/ne_110m_land.geojson",
        energy_service_input_root=RESULTS / "energy_service",
    )
    manifest = []

    def normalized(frame):
        names = {"NewDelhi": "New Delhi", "PortAngeles_small2019": "Port Angeles",
                 "Seattle_small2019": "Seattle"}
        for key in ["city", "context"]:
            if key in frame:
                frame[key] = frame[key].replace(names)
        return frame

    def read(path):
        return normalized(pd.read_csv(b / path))

    def panel(name):
        return normalized(pd.read_csv(p / name))

    def save(fig, folder, layout_number, caption, sources):
        # The original number selects the frozen layout, not the final label.
        fig.set_size_inches(fig.get_size_inches() * np.asarray(SIZE_SCALE))
        result = style_and_save(fig, folder, layout_number, caption, sources)
        if EXPORT_DPI != 300:
            fig.savefig(folder / f"Figure_S{layout_number:02d}.png", dpi=EXPORT_DPI)
        return result

    econ = read("building/regional_carbon_cost.csv.gz")

    matched = econ[(econ.strategy != "uniform") & econ.annual_energy_cost_delta_currency.notna()]

    fig, a = canvas(CONTEXTS)

    for ax, city in zip(a, CONTEXTS):
        col = CONTEXT_COLORS[city]
        if city == "New Delhi":
            gas = np.linspace(0, 15, 100)
            delhi = econ[
                (econ.city == city)
                & (econ.strategy != "uniform")
                & (econ.comparison_status == "matched")
                & (econ.service_fraction == 0.75)
            ]
            for method, line_color, label in [
                ("personal_center", CONTROL_COLORS["personal_center"], "Personal centre"),
                ("ltpref_interval", CONTROL_COLORS["ltpref_interval"], "Personal range"),
            ]:
                q = delhi[delhi.strategy == method].dropna(
                    subset=[
                        "delhi_affine_headroom_intercept_at_pf095_INR_year",
                        "delhi_affine_headroom_slope_per_INR_kwh_gas",
                    ]
                )
                values = (
                    q.delhi_affine_headroom_intercept_at_pf095_INR_year.to_numpy()[:, None]
                    + q.delhi_affine_headroom_slope_per_INR_kwh_gas.to_numpy()[:, None]
                    * gas[None, :]
                )
                ax.fill_between(
                    gas, values.min(axis=0), values.max(axis=0), color=line_color, alpha=0.25
                )
                ax.plot(gas, values.mean(axis=0), color=line_color, label=f"{label} (n={len(q)})")
            ax.axhline(0, color="gray", lw=0.7)
            ax.set_xlabel("Assumed gas price (INR/kWh HHV)")
            from matplotlib.ticker import FuncFormatter
            ax.yaxis.set_major_formatter(FuncFormatter(lambda value, position: f"{value / 1000:g}"))
            ax.set_ylabel("Bill headroom (10³ INR/year)")
            ax.legend(frameon=False, fontsize=11)
            continue
        d = matched[matched.city == city]
        if len(d):
            ax.scatter(
                d.delta_operational_co2_kg / 1000,
                d.signed_management_headroom_currency_year,
                s=12,
                alpha=0.75,
                color=col,
            )
        ax.axhline(0, color="gray", lw=0.7)
        ax.axvline(0, color="gray", lw=0.7)
        ax.set_xlabel("Operational CO₂ change (t/year)")
        currency = d.currency.iloc[0] if len(d) else ("INR" if city == "New Delhi" else "USD")
        ax.set_ylabel(f"Energy-bill headroom ({currency}/year)")

    manifest.append(
        save(
            fig,
            out,
            10,
            "Figure S10 | Energy-bill headroom and carbon accounting. a,c,d, Annual operational-carbon change versus energy-bill headroom for matched non-uniform controls in Miami, Port Angeles and Seattle, using historical state-average commercial energy prices. b, New Delhi headroom at service-budget fraction 0.75 and assumed power factor 0.95 as the assumed commercial gas price varies from 0 to 15 INR/kWh HHV. Gold/green denote personal-centre/personal-range control. Lines are scenario means and bands span matched constructed scenarios. Positive/negative headroom denotes energy-bill saving/increase before implementation costs. The gas-price axis is an assumed stress range. USD and INR denote United States dollars and Indian rupees. HHV denotes higher heating value.",
            ["building/regional_carbon_cost.csv.gz", "contracts/carbon_boundary_contract.json"],
        )
    )

    destination = PLOT / "supplementary"
    destination.mkdir(parents=True, exist_ok=True)
    old_stem = "Figure_S10"
    final_stem = "Figure_S24"
    for extension in (".png", ".pdf", ".svg"):
        shutil.copyfile(out / (old_stem + extension), destination / (final_stem + extension))
    # Final captions carry the current SI numbering, definitions and cross-references.
    caption = (RESULTS / "captions/Figure_S24.txt").read_text(encoding="utf-8")
    (destination / (final_stem + "_caption.txt")).write_text(caption, encoding="utf-8")
    print(f"Saved {destination / final_stem} (.png, .pdf, .svg, caption)")
    try:
        from IPython.display import display, Image
    except ImportError:
        pass  # Headless reproduction saves the same files without a notebook viewer.
    else:
        display(Image(filename=str(destination / (final_stem + ".png"))))

draw_s24()

In [ ]:
# Figure_S25: edit this cell and rerun to update this figure.
def draw_figure_s25():
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from thermal_heterogeneity.figures import PALETTE, apply_figure_style

    apply_figure_style()
    plt.rcParams.update({"font.family": "Times New Roman", "font.size": 11,
                         "axes.labelsize": 11, "xtick.labelsize": 11,
                         "ytick.labelsize": 11, "legend.fontsize": 11,
                         "text.color": "black", "axes.labelcolor": "black"})
    root = RESULTS / "Joint_main"
    strata = pd.read_csv(root / "service/DIRECT_COMMON_STRATA.csv")
    near = pd.read_csv(root / "service/NEAR_SERVICE_CONDITION_SUMMARY.csv")
    spatial = pd.read_csv(root / "joint_spatial_diagnostics/SPATIAL_PAIRS.csv")
    interaction = pd.read_csv(root / "joint_spatial_diagnostics/SPATIAL_INTERACTIONS.csv")
    stages = pd.read_csv(root / "joint_events/JOINT_EVENT_STAGE_COUNTS.csv")
    decisions = pd.read_csv(root / "joint_events/JOINT_EVENT_STAGE_DECISIONS.csv")
    sensitivity = pd.read_csv(RESULTS / "Targeted_extension/ranges/DEMAND_SENSITIVITY_SUMMARY.csv")
    controls = ["uniform_average", "personal_center", "ltpref_interval"]
    colors = dict(zip(controls, [PALETTE["dark_grey"], PALETTE["orange"], PALETTE["green"]]))
    labels = {"uniform_average": "Uniform\naverage", "personal_center": "Personal\ncentre",
              "ltpref_interval": "Personal\nrange"}
    fig, axes = plt.subplots(4, 2, figsize=(180 / 25.4, 225 / 25.4))
    fig.subplots_adjust(left=0.15, right=0.975, bottom=0.08, top=0.96,
                        wspace=0.68, hspace=0.70)
    titles = ["Energy at common service", "Near-service comparisons", "Grouping demands",
              "Grouping and permissions", "Response qualification", "Electricity recovery",
              "Coefficient sensitivity", "Range-width sensitivity"]
    for letter, title, axis in zip("abcdefgh", titles, axes.flat):
        axis.set_title(f"({letter}) {title}", loc="left", fontsize=12, fontweight="bold", pad=9)
        for side in ["top", "right"]:
            axis.spines[side].set_visible(False)
        for side in ["left", "bottom"]:
            axis.spines[side].set_color("#8E8E8E")
            axis.spines[side].set_linewidth(0.65)
        axis.tick_params(colors="black", length=3, pad=3)
        axis.grid(False)

    # a: report the complete paired subset and the nonidentical-run subset.
    axis = axes[0, 0]
    for y, key in [(1, "all"), (0, "nonidentical_physical")]:
        row = strata[strata.stratum.eq(key)].iloc[0]
        direct = row.median_absolute_direct_delta_site_pct
        common = row.median_absolute_common_delta_site_pct
        axis.plot([common, direct], [y - .09, y + .09], color=PALETTE["grey"], lw=1.2)
        axis.scatter(direct, y + .09, s=30, color=PALETTE["dark_grey"], zorder=3)
        axis.scatter(common, y - .09, s=32, marker="D", facecolors="white",
                     edgecolors=PALETTE["dark_grey"], zorder=3)
        axis.annotate(f"{direct:.2f}", (direct, y + .09), xytext=(0, 5),
                      textcoords="offset points", ha="center", va="bottom")
        axis.annotate(f"{common:.2f}", (common, y - .09), xytext=(7, 0) if y == 0 else (4, -6),
                      textcoords="offset points", ha="left", va="center" if y == 0 else "top")
    pair_counts = [int(strata.loc[strata.stratum.eq(key), "pairs"].iloc[0]) for key in ("all", "nonidentical_physical")]
    axis.set_yticks([1, 0], [f"All pairs\n(n = {pair_counts[0]})", f"Different runs\n(n = {pair_counts[1]})"])
    axis.set_xlim(-.08, 1.7)
    axis.set_ylim(-.47, 2.1)
    axis.set_xlabel("Median absolute energy\ndifference (%)", labelpad=6)
    axis.legend(handles=[Line2D([], [], marker="o", color="none", markerfacecolor=PALETTE["dark_grey"],
                                      markeredgecolor=PALETTE["dark_grey"], label="Direct"),
                         Line2D([], [], marker="D", color="none", markerfacecolor="white",
                                      markeredgecolor=PALETTE["dark_grey"], label="Common limits")],
                loc="upper left", bbox_to_anchor=(-.02, 1.04), ncol=2, frameon=False,
                handletextpad=.3, handlelength=.6, columnspacing=.6, labelspacing=.2)

    # b: denominators count control-condition comparisons, not independent runs.
    axis = axes[0, 1]
    tolerances = [.02, .05, .10]
    for y, tolerance in enumerate(tolerances):
        group = near[np.isclose(near.tolerance_c, tolerance)]
        distinct = int(group.status.eq("distinct_run_matches").sum())
        shared = int(group.status.eq("shared_run_only").sum())
        axis.barh(y, distinct, color=PALETTE["dark_grey"], height=.52)
        axis.barh(y, shared, left=distinct, color=PALETTE["light_grey"], height=.52)
        axis.text((distinct + shared) / 2, y, f"{distinct} / {shared}", ha="center", va="center")
    axis.set_yticks(range(3), ["0.02", "0.05", "0.10"])
    axis.set_ylabel("Matching tolerance (°C)", labelpad=6)
    axis.set_xlabel("Controller–condition comparisons", labelpad=6)
    axis.set_xticks([0, 80, 160, 240])
    axis.set_xlim(0, 245)
    axis.set_ylim(-.6, 3.0)
    axis.text(.02, .98, "Different / shared runs only", transform=axis.transAxes,
              ha="left", va="top")

    # c,d: retain full paired distributions and count incomplete comparisons.
    common = spatial[spatial.weight_view.eq("profile_equal") & spatial.comparison_mode.eq("equal_service")
                     & spatial.service_fraction.eq(1) & spatial.axis.eq("layout")
                     & spatial.alternative.eq("demand_ordered") & spatial.permission.eq("native_zone")]
    four = interaction[interaction.weight_view.eq("profile_equal") & interaction.comparison_mode.eq("equal_service")
                       & interaction.service_fraction.eq(1) & interaction.layout.eq("demand_ordered")
                       & interaction.permission.eq("native_zone")]
    for axis, frame, metric, ylabel in [
        (axes[1, 0], common, "delta_coverage_pp", "Coverage change\n(percentage points)"),
        (axes[1, 1], four, "interaction_coverage_pp", "Coverage interaction\n(percentage points)")]:
        axis.axhline(0, color=PALETTE["grey"], lw=.7, zorder=0)
        for x, control in enumerate(controls[1:]):
            group = frame[frame.control.eq(control)]
            matched = group[group.status.eq("matched")].sort_values(["context", "composition_id"])
            values = matched[metric].to_numpy()
            jitter = np.linspace(-.12, .12, len(values)) if len(values) > 1 else np.zeros(len(values))
            axis.scatter(x + jitter, values, s=24, color=colors[control], alpha=.85,
                         edgecolors="white", linewidths=.35, zorder=3)
            if len(values):
                axis.plot([x - .22, x + .22], [np.median(values)] * 2,
                          color=colors[control], lw=2.2, zorder=4)
            axis.text(x, .98, f"{len(matched)}/{len(group)}", transform=axis.get_xaxis_transform(),
                      ha="center", va="top")
        axis.set_xticks([0, 1], [labels[name] for name in controls[1:]])
        axis.set_xlim(-.5, 1.5)
        low, high = axis.get_ylim()
        axis.set_ylim(low - .05 * (high - low), high + .23 * (high - low))
        axis.set_ylabel(ylabel, labelpad=5)

    # e,f: de-duplicated original July scenarios at the reference service budget.
    gate = stages[stages.event_month.eq(7)
                  & stages.service_fraction.eq(1)].set_index("stage")
    stage_keys = ["raw_positive", "baseline_qualified", "event_qualified", "recovery_qualified"]
    axis = axes[2, 0]
    counts = gate.loc[stage_keys, "decisions_with_positive_selection"].to_numpy()
    axis.bar(range(4), counts, width=.62, color=PALETTE["grey"])
    for x, count in enumerate(counts):
        axis.text(x, count + 9, str(int(count)), ha="center", va="bottom")
    axis.set_xticks(range(4), ["Positive", "Baseline", "Event", "Recovery"], rotation=30, ha="right")
    axis.set_ylim(0, 430)
    axis.set_ylabel("Controller decisions", labelpad=6)
    axis.set_yticks([0, 120, 240, 360])
    retained = decisions[decisions.source_scope.eq("main_peak_dates") & decisions.event_month.eq(7)
                         & decisions.service_fraction.eq(1) & decisions.stage.eq("recovery_qualified")
                         & decisions.selected.eq(True)]
    axis = axes[2, 1]
    for x, control in enumerate(controls):
        values = 100 * retained.loc[retained.control.eq(control), "recovery_fraction_of_event_saving"].to_numpy()
        jitter = np.linspace(-.16, .16, len(values))
        axis.scatter(x + jitter, values, s=16, color=colors[control], alpha=.55,
                     edgecolors="none", zorder=2)
        axis.plot([x - .23, x + .23], [np.median(values)] * 2, color=colors[control], lw=2, zorder=3)
        axis.text(x, 65, f"n = {len(values)}", ha="center", va="top")
    axis.set_xticks(range(3), [labels[name] for name in controls])
    axis.set_xlim(-.5, 2.5)
    axis.set_ylim(0, 70)
    axis.set_yticks([0, 20, 40, 60])
    axis.set_ylabel("Recovery electricity /\nevent saving (%)", labelpad=6)

    # g,h: original and symmetric four-variant supports are separate diagnostics.
    for axis, parameter, fixed, values in [
        (axes[3, 0], "griffiths_coefficient", "interval_width_multiplier", [.33, .67]),
        (axes[3, 1], "interval_width_multiplier", "griffiths_coefficient", [.5, 1.5])]:
        fixed_value = 1 if fixed == "interval_width_multiplier" else .5
        rows = sensitivity[np.isclose(sensitivity[fixed], fixed_value)
                           & sensitivity.candidate_support.isin(["primary", "equal_four_variants"])]
        for index, value in enumerate(values):
            for support in ["primary", "equal_four_variants"]:
                group = rows[np.isclose(rows[parameter], value) & rows.candidate_support.eq(support)]
                for row in group.itertuples(index=False):
                    x = 100 * row.primary_match_lost / row.primary_matched
                    y = 100 * row.strict_sign_reversals / row.both_matched if row.both_matched else np.nan
                    axis.scatter(x, y, s=52, marker=["o", "s"][index],
                                 facecolors=colors[row.control] if support == "primary" else "white",
                                 edgecolors=colors[row.control], linewidths=1.1, zorder=3)
        axis.set_xlim(-2, 32)
        axis.set_ylim(-1, 24)
        axis.set_xticks([0, 10, 20, 30])
        axis.set_yticks([0, 5, 10, 15, 20])
        axis.set_xlabel("Original matches lost (%)", labelpad=6)
        axis.set_ylabel("Energy-direction\nreversals (%)", labelpad=6)
        axis.text(.98, .98, f"○ {values[0]:g}   □ {values[1]:g}", transform=axis.transAxes,
                  ha="right", va="top")
    caption = 'Supplementary Figure S25 | Service, space, recovery and demand-definition checks. Panels a–f use all six compositions and both main layouts; equivalent single-floor permissions count once. a, Median absolute energy differences for 154 paired comparisons and the 38 with distinct common-limit trajectories. Filled/open symbols show direct/common-limit operation. b, Controller-condition counts with distinct-run or shared-run-only matches within each mean-and-tail tolerance, out of 240. c, Ordered-minus-random range coverage under zone commands, equal-profile weights and common random-layout limits. d, Placement-by-permission coverage interaction. Points are cases, lines are medians and labels give matches/24 conditions per personal control. e, Positive July choices through sequential service checks. f, Extra recovery electricity divided by event savings for 64 qualified July choices, with median lines and case counts. g,h, Original ASHRAE three-composition, three-layout coefficient and range-width sensitivities. Coordinates show original matches lost and energy-direction reversals, divided by original and jointly matched cases, respectively. Filled/open symbols denote original/symmetric candidate supports; circles/squares denote lower/higher parameter values. Gold/green denote Personal centre/Personal range. Personal range uses model-generated demand intervals; coverage is occupied time within them.'
    (RESULTS / "captions/Figure_S25.txt").write_text(caption, encoding="utf-8")
    save_plot(fig, "Figure_S25", size_mm=(180, 225))


draw_figure_s25()
